# PIV-Q27-EVO12 — evolution over verified programs, 12 random evaluation tasks

Same model (Qwen3.8-27B-FP8, 4×L4) and the same tested infrastructure as V3.1; the single long REPL conversation is replaced by an evolution loop (Imbue-style): per task 4 independent candidates (rule in plain words -> `solution(grid)`), fresh verification on every training pair, then up to two 2-slot follow-up generations subject to the 32,768-token hard cap: the 2 best distinct candidates repaired for 2 generations with an exact error map, ≤32k generated tokens per task, pass@2 from train-exact programs. Tasks: first 12 of the seed-20261004 sample (9 NVARC misses, 3 NVARC full solves).

Pre-registered verdict (`evo_verdict`): STRONG GO if ≥3 solved and ≥2 of them NVARC misses; PROMISING if ≥1 NVARC-miss solve; otherwise WEAK / REDESIGN. A deadline stop gives INCOMPLETE / NO DECISION; a verifier infrastructure failure gives INVALID. Scoring is canonical per-query pass@2 (two distinct outputs per test query from train-exact programs).

---

<details><summary>V3.1 notes (infrastructure provenance)</summary>

# PIV-Q27-V3.1 — EVAL120, ALL FIXES IN ONE RUN

All 120 public evaluation tasks (answers known), exact scoring, compared against the frozen baseline
`CANONICAL_BASELINE_EVAL120_2026-10-03.md` (Train-Verified 3/120, Strict Solved 2/120). Every problem in that
diagnosis is addressed at once; each mechanism has its own counter in `v3_mechanism_summary.json` so the run
shows which fixes fired.

| problem | v3 fix |
|---|---|
| 49/120 first turns spent 6,144 tokens transcribing grids | `ACT-LOW-FORCED`: low effort, forced `</think>` after 2,048 thinking tokens; prompt forbids transcription; `puzzle.keys()` instruction removed |
| forced arm's last action cut at 768 tokens | full 2,048-token action reserved |
| 3/173 tool calls ever tested a complete `solution()` | automatic fresh verification after every tool call, report shown to the model |
| verified code rewritten by the finalizer | train-exact candidate submitted verbatim |
| 38/120 finalizers cut at 2,048 tokens | 4,096-token finalizer reserve |
| ~65 % of failed finalizer text was comment reasoning | code-only contract, temperature 0.1 |
| handoff = last 1,024 reasoning tokens | best candidate + verifier report |
| hand-written BFS/geometry bugs | preloaded, tested helper library |
| repair: 52/79 parse failures, budget from leftovers | **off in this run**; returns later as a separate, fixed V3R ablation |
| 29 % of wave time with <=2 active tasks | dynamic backfill |

Budget: 12,288 tokens/task (8,192 exploration + 4,096 finalizer), concurrency 12.

**v3.1 (pre-run review of v3):** verbatim commit also requires the program to run on every test input; candidates ranked near-miss-aware (exact pairs, right-shaped pairs, test runs, wrong-cell fraction), used for best-candidate, exploration-final and finalizer fallback; dependency-aware candidate builder with regression self-tests; exploration-only vs final-safe helpers separated in the prompt; repair off; runtime accounting counts terminal tasks, not requested ones. NVARC comparison is post-hoc only.

---

<details><summary><b>HISTORICAL NOTES (v2.x) — SUPERSEDED.</b> Not this run's configuration (this run: 12,288 tokens = 8,192 exploration + 4,096 finalizer, ACT-LOW-FORCED, repair OFF).</summary>

All 120 public evaluation tasks, **answers known**, exact scoring. Derived from the audited solver
`PIV-Q27-C1-v2_5B` with only the run profile changed: 8,192 generated tokens per task, finalizer
reserve 2,048, single arm `ACT-LOW`, concurrency 12 - identical to what `PIV-Q27-SUB v1.3` does on the
hidden test file, so the accuracy measured here describes the submission's actual recipe.

The submission run can never be scored locally (the hidden answers are never published); this run is
the only place where real ARC-AGI-2 difficulty meets known answers.

---

> **Donor / baseline:** canonical v2.4G real run (`source SHA 50b9acb8…`, executed SHA `44934b20…`, evidence bundle SHA `387d4bb7…`). v2.4G established 16/16 standalone final programs + 16/16 fresh verifier executions with 0/16 all-train-exact. The terminal/finalizer path is therefore frozen; v2.5 targets semantic program correctness.
>
> **Single causal mechanism:** after candidate #1 fails fresh train verification, build compact *training-side* verifier feedback and make exactly **one** no-tool, non-thinking structured repair call. Fresh-verify candidate #2. Test execution remains embargoed until all training pairs are exact.
>
> **Feedback contract (v2.5):** per failed train pair report output/expected dimensions; for same-shape grids report wrong-cell count and a small mismatch sample; report the final exception line for execution failures; explicitly flag identity outputs (`output == input`). Feedback is tokenizer-bounded to <=600 tokens. Transport is verified with a canonical `strip()` hash + exact presence sentinel in the rendered repair prompt.
>
> **Important non-change:** ACT-BUDGET3072 keeps `ACTION_RESERVE_TOKENS=768`. Raw v2.4G evidence shows five late forced-close actions hitting exactly 768 tokens and ending `length/malformed_tool`, but changing 768→2048 would alter exploration allocation and become a second solver-policy intervention. v2.5 logs that truncation diagnostic and defers the reserve experiment to a separate follow-up.
>
> **Run discipline:** same model/runtime/C1 engine, frozen SMOKE8/order, LOW + BUDGET3072, same REPL/parser/verifier, same 12,288 generated-token ceiling. No reviewer, multi-sample, DSL, evolution, prompt search, seed diversification, PTQ, NVARC, prefix cache, or custom all-reduce.

---

# PIV-Q27-C1 v2.4G — ONE-SESSION ADAPTIVE TERMINAL-COMMIT SMOKE

> **v2.4G — per-task engine-cap isolation patch over v2.4F.** Real vLLM is expected to respect each request's `max_tokens`, but v2.4F treated a single over-cap RequestOutput as a whole-wave engine exception and terminalized every task in the batch. v2.4G keeps output-cardinality mismatch as a wave-fatal mapping error, while isolating a token-cap violation to the offending task only. The task is recorded infrastructure-invalid with the exact observed token count and requested cap; unaffected requests in the same wave continue. No model, prompt, budget, sampling, terminal-policy, verifier, or arm parameter is changed.


> **v2.4F — PREFLIGHT/runtime-validation patch over v2.4E.** The default ADAPTIVE_SMOKE solver policy is unchanged. This revision fixes the CPU PREFLIGHT terminal-policy assertion, adds explicit coverage for all terminal-trigger predicates, guards against engine outputs exceeding their requested token caps, and corrects TERMINAL_PILOT decision labels. ACT-BUDGET3072 intentionally remains 3,072 thinking tokens: under the 8,192 exploration cap this usually means two full forced-close turns plus at most a partial third; this is a known property of the terminal-policy package, not silently retuned here.


> **v2.4E — runtime-blocker patch over v2.4D.** Two independently reproduced pre-run failures are fixed:
> (1) the preload malformed-tool self-test now tests a *committed* action after `</think>`, while an
> independent regression asserts that unclosed-thinking tool-like text remains scratch `text`; and
> (2) `promotion_eligible_arms` is initialized in the PREFLIGHT branch so recommendation generation
> cannot raise `NameError`. No solver policy, model sampling, budgets, tasks, engine settings, or
> verifier semantics are changed.



> **v2.4D — pre-run correctness patch over v2.4C.** It preserves the one-session solver policy while fixing
> four confirmed scheduler failure paths: incomplete reasoning is handed to terminal commit through a bounded
> scratch note; exhausted exploration retries spend the protected terminal slot; REPL startup/runtime failures
> are propagated as infrastructure evidence; and Proxy48 authorization now requires the real promotion/compute gate.
> Terminal JSON framing is handled without mutating control-token-looking text inside `solution_code` strings.
>
> **v2.4C — final blocker patch over v2.4B.** It preserves the one-session solver policy while fixing two
> remaining evidence-validity issues found in the v2.4B re-audit: verifier infrastructure failures are
> propagated into task/arm infrastructure accounting, and tool feedback is token-truncated against the
> *actual rendered terminal prompt* so the protected 4,096-token finalizer capacity is a checked invariant.
> Partial PILOT2 evidence can no longer populate the matched recommended arm.

> **v2.4B — correctness revision of v2.4A.** Two defects in v2.4A made a real GPU run unusable and
> were invisible to the existing self-tests:
>
> 1. **Terminal artifact never parsed.** Sampling uses `skip_special_tokens=False`, so the decoded
>    finalizer text ends with `<|im_end|>`; `json.loads()` therefore failed on every *correct*
>    answer. The Cell 7 positive control hit it first, so engine bootstrap failed and no real
>    generation ran at all. Fixed in the shared parser now defined in Cell 6.
> 2. **The scoring gate crashed.** `compute_smoke_gate()` did `int(w["phase"])`, but the terminal wave
>    row carries `phase="terminal_finalizer"` → `ValueError`, killing all of Cell 10.
>
> The adaptive interface gate itself was also wrong in both directions (it passed a 100%-broken
> terminal parser and failed a run where the model solved during exploration). See
> `V24B_FIXES` in Cell 14 for the full list.


## Fast competition workflow: embedded safety checks → real positive control → 2-task diagnostic gate → automatic SMOKE8 continuation

**This revision supersedes the serial v2.4 `PREFLIGHT -> upload/review -> TERMINAL_PILOT -> upload/review -> SMOKE8` workflow.**

One Kaggle 4×L4 session now does all of the following without waiting for an external approval in the middle:

1. compact embedded CPU regression checks for the new terminal-policy invariants;
2. one score-excluded real-Qwen positive control for `enable_thinking=False` + no-tools + structured JSON;
3. both approved arms on diagnostic tasks `0934a4d8` and `cb2d8a2c`;
4. an automatic **interface gate** based only on execution integrity and terminal-stage reachability, not puzzle correctness;
5. if that gate passes, the same model session immediately continues the same two arms on the remaining frozen SMOKE8 tasks.

The diagnostic tasks do **not** have to solve. They only test whether the structural terminal path is alive.
If the interface gate fails, the notebook stops before spending the remaining SMOKE8 generation budget and still bundles all evidence.

Proxy-48 is not run inside this notebook. A completed, infrastructure-valid SMOKE8 with at least one fresh-verifier
all-train-exact program and at least one usable test prediction can authorize the next Proxy-48 experiment.

---
# PIV-Q27-C1 v2.4 — TERMINAL-COMMIT PILOT — Protected Finalization + Fresh Exact Verifier
## ARC-AGI-2 | Kaggle 4× NVIDIA L4 | Offline | Qwen3.8-27B-FP8 TP=4

**Purpose of v2.4:** test one solver-policy package only: protect a terminal code-generation budget so exploration cannot consume the entire 12,288-token task budget, then switch structurally to a no-tools, non-thinking terminal commit that emits one standalone `solution(grid)` candidate for the existing fresh verifier.

### v2.4 causal scope
- Total generated-token ceiling remains **12,288**.
- Exploration is hard-capped at **8,192** generated tokens.
- **4,096** tokens are protected for exactly one terminal finalizer request.
- Finalizer uses Qwen `enable_thinking=False`, advertises **no Python tool**, and requests a JSON object containing `solution_code`.
- Exploration prompts, C1 engine settings, REPL, verifier, reasoning arms, and exploration seed policy are otherwise frozen from v2.3.
- This is a **terminal-policy package** experiment; it does not claim to isolate reserve size alone.
- Real pilot tasks are exactly `0934a4d8` and `cb2d8a2c`, for both `ACT-LOW` and `ACT-BUDGET3072`.
- Pilot results can justify a frozen SMOKE8 follow-up, but **cannot authorize Proxy-48 directly**.

### Historical v2.4 serial sequence
Superseded by the v2.4A one-session adaptive workflow above.

---
# PIV-Q27-C1 v2.3 — REAL SOLVER QUALITY SMOKE — Qwen3.8-27B Stateful Python REPL + Fresh Exact Train Verifier
## ARC-AGI-2 | Kaggle 4× NVIDIA L4 | Offline | Qwen3.8-27B-FP8 TP=4

This notebook is the **safe-recovery-hardened preflight for the next decisive solver-quality experiment** after the validated v1.9d reasoning-activation probe and the v1.2.4 throughput probe.

### Frozen causal question

> With the v1.9d PIV solver mechanics frozen, does changing only the serving engine from eager execution to CUDA-graph/hybrid execution (`enforce_eager=True -> False`) allow the two activation-approved reasoning arms (`ACT-LOW`, `ACT-BUDGET3072`) to produce executable, fresh-verifier `all_train_exact` ARC programs and valid test outputs on the same frozen SMOKE8 tasks?

### Canonical donor / evidence

- Donor notebook SHA256: `007c8d68559c39a948dfef9d571fbd2e5da7335492a6f3330645db892648b525`
- Donor lineage: `PIV-Q27-ACT-v1.9d`
- Activation probe run: `20260915T124541Z`
- Approved quality-smoke arms: `ACT-LOW`, `ACT-BUDGET3072`
- Throughput evidence ZIP SHA256: `c34fd3736dd9d2565ef681c6fd1936bb7eb1f99b3a95d222eda419049cd3e2aa`
- Serving intervention: **only** `enforce_eager=False`; custom all-reduce OFF; prefix cache OFF.
- The old greedy-prefix numerics threshold is **not** a solver-quality gate. Real fresh all-train verification and exact ARC outputs are the correctness authority.

### Decision hierarchy

Integrity → executable final program → fresh all-train-exact → valid test execution → competition output-level exact score (strict all-query task exact kept as a diagnostic) → compute efficiency/stability.

Tool engagement is diagnostic only. `ACT-BUDGET3072` is **not gated on ACT-LOW succeeding**.

### Run discipline

Historical v2.3 note: the old quality-smoke sequence has been superseded by the v2.4 `PREFLIGHT -> TERMINAL_PILOT` sequence above.


### Source basis

Primary source snapshot: `gutfeeling/arc-agi-2-submission` at commit
`b3d4494892c6d85fa2a35d548271fb02ea3e191b`.

The P0 mechanism is source-faithful at the **semantic/invariant level**:

1. one reasoning-model lineage per ARC task;
2. one persistent Python REPL namespace across tool turns;
3. the puzzle is preloaded into the REPL and protected from overwrite/whole-puzzle printing;
4. Python stdout/errors/timeouts become model feedback;
5. the model must finish with one self-contained `solution(grid)` program;
6. the final program is executed again in a **fresh process** and must match **every training pair exactly** before any test output is admitted;
7. test grids are produced by execution of the verified program, never transcribed from prose.

### Explicit LOCAL adaptations for Kaggle 4×L4

- Direct `vllm.LLM.generate(...)` + Qwen3.8 native tool-capable chat template instead of an OpenAI/Responses server.
- Lightweight persistent local Python subprocess instead of Docker/IPyBox.
- Hard process timeout + restricted dangerous filesystem/process APIs.
- Bounded P0 budget: one lineage, 6 executed Python calls, at most 12 model turns, up to the remaining 12,288-token task budget per turn (context-bounded), and 12,288 generated tokens/task total.
- Tool output sent back to Qwen is compact; full bounded raw tool events are checkpointed.
- No reviewer, no second lineage, no deterministic PTQ repair loop, no NVARC seeding, no judge and no evolution. Those are later experiments.

v2.4 sequence: `PREFLIGHT -> TERMINAL_PILOT -> frozen SMOKE8 if the interface is clean`. Proxy-48 remains blocked.


### v2.3 safe-recovery and infrastructure-accounting hardening (mechanism unchanged)

Independent review of v2.2 found two related P1 evidence-integrity bugs in the phase-2 exception path. v2.3 fixes them without changing prompts, solver budgets, task selection, reasoning policies, verifier semantics, or the C1 serving intervention:

1. a phase-2 engine exception may salvage phase-1 code **only through the normal length-salvage contract**: the phase-1 text must have a closed `</think>` action region containing a closed, AST-valid `solution()` block. Draft code inside unclosed reasoning is never promoted to a final candidate;
2. engine failure is tracked independently from candidate quality. A recovered train-perfect/test-usable candidate remains useful quality evidence, but the arm is infrastructure-invalid and cannot pass the compute/promotion gate or produce a matched winner;
3. phase-2 output tokens are reported as **unknown/unobserved**, not zero, when `generate()` raises before returning `RequestOutput`; phase-1 observed tokens remain preserved as a lower bound;
4. `provisional_best_valid_arm` is suppressed whenever matched comparison is infrastructure-invalid.

PREFLIGHT includes regressions for unclosed-think draft rejection, legitimate closed-action salvage under infrastructure failure, infrastructure-gate invalidation, and unknown phase-2 token accounting.

### v2.2 failure-path hardening (mechanism unchanged)

Independent review of v2.1 found four remaining evidence-integrity gaps. v2.2 fixes them without changing the solver mechanism or the C1 serving intervention:

1. model import/allocation/warmup failures are captured as bootstrap evidence so downstream reporting/bundling can still finish;
2. a forced-arm phase-2 `generate()` exception preserves phase-1 token/finish evidence, does not destroy already-complete phase-1 final programs in the same batch, and terminalizes only genuinely affected continuations;
3. verifier `started` is true only after a subprocess is actually created; launch failures are distinguishable from started-process failures/timeouts;
4. recommendation logic refuses to call an infrastructure-invalid or incomplete arm a no-quality result.

Verifier timeout semantics are explicit: `VERIFIER_TIMEOUT_SEC` applies **per fresh subprocess stage** (one train stage plus one stage per test input), not to the candidate as a single aggregate 20-second wall-time cap. Any matched C0/C1 comparison must keep this verifier contract identical.

### v1.1 source/donor re-audit hardening

After re-comparing P0 against the frozen Q1 donor and the audited PiVOTools/Qwen sources, v1.1 fixes four issues without adding a new solver mechanism:
1. use the full source agentic prompt family rather than a shortened hybrid prompt;
2. reject length-truncated generations before parsing them as a tool/final answer;
3. use per-request generation caps inside a batch so one near-context-limit task cannot starve the whole wave;
4. harden the lightweight local worker against host/Kaggle filesystem and process/network access.

It also validates the actual mounted Qwen multi-turn tool-history rendering during PREFLIGHT.


### v1.2 additional verifier/accounting correction

A second source-level pass found that PiVOTools verifies training examples first and executes test inputs in a separate step. v1.1's single verifier exception boundary could erase a genuinely train-perfect result if test execution later failed. v1.2 separates those states and counts train-perfect rate over the entire frozen active task set, including missing/failed tasks as non-passes.


### v1.3 external-review verification

An independent review of the earlier v1.0 was checked against this v1.2 notebook and the audited PiVOTools source.
Some findings had already been fixed in v1.2 (per-request SamplingParams, multi-tool handling, finish-reason gate,
full source prompt, Qwen history probe, train-vs-test verifier state). The remaining real issues are fixed here:
kernel-level `copy` import, source-style bounded retries, a less starvation-prone per-turn ceiling, source-tolerant grid
normalization, per-test partial output retention, budget-aware finalization, think-aware action parsing, dead-worker restart,
puzzle reset hardening, narrower dunder blocking, `py` fences, and a scripted CPU preflight agent-loop test.

Two review suggestions are intentionally NOT adopted:
- no raw-text continuation after truncation (the PiVOTools source retries the agentic call; it does not splice partial text);
- no switch from XHIGH to MEDIUM (PIV-Q27-P0 is the planned XHIGH causal experiment).


### v1.4 second external-review re-audit

This revision corrects the remaining mismatch between the local vLLM loop and PiVOTools' two-level retry semantics. Incomplete/length-capped steps are retried first with the same conversation and persistent REPL state; only after step retries are exhausted may the outer source-style attempt restart.

The local per-turn ceiling now equals the remaining total P0 generation budget (subject to context headroom), the closest bounded analogue to the source configuration's lack of an explicit per-step output-token cap.

P0 still uses one Qwen XHIGH lineage and the same 12,288 total generated-token budget. `SMOKE_GPU` now uses the same generation/tool budget as `PROXY48`, so it is a real viability gate: if first-turn length truncation exceeds 50%, do not launch Proxy-48 without revisiting budget/effort/context policy in a separate revision.


### v1.5 third external-review re-audit

The v1.4 first-turn-length SMOKE gate was too weak. v1.5 gates on terminal budget death and empirical wall-clock projection from real generation waves.


### v1.6 deep-audit fixes (mechanism unchanged)

Third-review fixes, all LOCAL harness hygiene — the solver mechanism (one Qwen XHIGH lineage, persistent REPL,
6 executed Python calls, 12 turns, 12,288 total generated tokens, fresh all-train exact verifier, frozen Proxy-48,
Q1-frozen engine) is unchanged:

1. **Safety reserve 30 → 60 min + per-wave projected-time pre-check.** A single wave can run for the remaining
   budget (up to 12,288 tokens ≈ 33 min at the Q1-measured step latency), so a wave starting just before the old
   30-min deadline could cross the 12 h Kaggle limit and lose `/kaggle/working`. The loop now projects each wave
   from measured wave timings (`wave_metrics.jsonl`) and refuses to start one that would cross the deadline
   (`global_deadline_projected`, resumable).
2. **Stratified SMOKE8.** The previous selector took all four Proxy-48 tier-3 tasks (50% vs 8% in Proxy-48), which
   biased the feasibility gate. SMOKE8 now mirrors Proxy-48: 1 tier-3, 2 tier-2, 5 tier-1 (incl. easy_static,
   hard_tier1, multi_query, dynamic), deterministic in Proxy-48 order.
3. **Structured malformed-tool retry counter** resets only after a VALID structured tool turn is accepted (the
   early reset let the max-2 inner retry be bypassed).
4. **Guard-rejected snippets are not charged** to the 6-call tool budget (they never reached Python); fenced
   ```` ```python ```` parameter payloads are unwrapped before the guard.
5. **Tool output is sanitized** before it re-enters the prompt (`<|im_end|>`, `<tool_response>` … cannot break
   the chat structure).
6. **Length-capped final turns with a CLOSED, parseable `solution()` block are accepted** (LOCAL; the block is
   demonstrably complete, retrying would only burn budget).
7. `malformed_tool` requires an actual `<tool_call>`; prose merely mentioning `<function=` is no longer a retry.
8. SMOKE gate additionally requires complete terminal coverage and reports measured step latency + projected
   max wave minutes; `global_deadline_projected` counts as an infrastructure failure.
9. Stale `per-turn 4096` provenance string corrected.
10. PREFLIGHT now drives the **real** `run_real_piv()` scheduler with a scripted fake LLM (happy path, budget
    death, malformed-tool retry accounting, guard-rejection accounting, length salvage, deadline pre-check).

Optional throughput switch (default OFF to keep the Q1 engine fingerprint): `ENABLE_PREFIX_CACHING`.

### v1.7 cross-audit hardening

Independent re-audit confirmed the v1.6 deep-audit changes, but found one scheduler accounting bug:
inside a multi-call assistant turn, an early guard-rejected call did not increment `tool_calls`, yet later calls
were denied by their original positional index. Thus a rejected call could indirectly consume a valid remaining
execution slot. v1.7 makes the budget test dynamic per call.

Two defensive changes do not alter the PIV mechanism:
- SMOKE terminal coverage also requires unique task IDs;
- projected-deadline timing never becomes less conservative than the Q1-derived 0.20 s/output-token prior.


### v1.8/v1.9 — Reasoning-Activation Ablation (ACT)

The v1.7 SMOKE (frozen reference **ACT-XHIGH**) showed 8/8 tasks spending the entire 12,288-token budget inside the
first `<think>` without ever reaching a tool call (tool engagement 0.00, budget death 1.00). Qwen3.8's template places
tool calls **after** `</think>`, so under XHIGH the model must finish a full solve before it can act. This revision
keeps the PIV mechanism (persistent REPL, 6 executed calls, 12 turns, 12,288 total generated tokens, fresh all-train
exact verifier, frozen stratified SMOKE8, Q1-frozen engine) and ablates **only the reasoning-to-action policy**:

| arm | single change vs ACT-XHIGH |
|---|---|
| `ACT-LOW` | `reasoning_effort="low"` (template injects "keep your thinking brief…") |
| `ACT-MEDIUM` | `reasoning_effort="medium"` (template injects no reasoning instruction) |
| `ACT-BUDGET3072` | `xhigh` + Qwen's documented two-stage thinking budget: after 3,072 thinking tokens without `</think>`, the stop-thinking sentence + `</think>` is injected (token-exact continuation of the same request) and the model gets ≤ `ACTION_MAX_TOKENS` to act |

All arms run in one session (one model load) on the same 8 tasks, same prompts, same REPL/verifier, same seeds policy;
each arm has its own lineage, records, traces and work dir. `ACT-XHIGH` is **not rerun** — its v1.7 numbers are the frozen
reference row in `arm_comparison.csv`.

**Pre-registered measurement contract** (`MEASUREMENT_CONTRACT`, written to the bundle before any result is read) and
**pre-registered verdict tiers**: Activation (tool execution ≥ 50% of tasks) → Productivity (final program ≥ 25%, no
infrastructure failure) → Quality (all-train-exact > 0), plus the compute gate (budget death ≤ 50%, projected Proxy-48 with
20% margin ≤ 10 h, complete unique terminal coverage). `recommendation.json` applies these rules mechanically.

The report separates **PROVEN BY PREFLIGHT** (harness correctness, CPU, fake model) from **MEASURED ONLY ON GPU**
(model behaviour). Nothing in PREFLIGHT predicts what the model will do.

### v1.9 — cross-audit fixes (mechanism unchanged)
1. Byte-exact Qwen-documented early-stopping text (`"\n\nConsidering … now.\n</think>\n\n"`).
2. Qwen documented three-way flow: phase 1 `stop` → complete; `length` without `</think>` → inject stop text; `length` **with**
   `</think>` (action cut at the think cap) → token-exact continuation **without** injection (`continuation_mode`).
3. Token-exact continuation is mandatory (`prompt_token_ids`); no text re-render fallback. Warmup validates the token-prompt API on real vLLM.
4. Think/post tokens split exactly on the `</think>` token id; injected stop text never counted as thinking.
5. Phase-2 request uses a deterministic phase-specific seed (`seed + 1,000,000`).
6. Arm-independent wall-clock projection (`ACT_FIXED_SETUP_ELAPSED_SEC` measured once).
7. Ranked winner selection (`recommended_arm` from the ranked list).
8. PROXY48 is locked: `PROXY48_APPROVAL` must carry the SMOKE run id + `recommendation_sha256` from `recommendation.json`.
9. `repeated_tool_calls`, AST stateful-reuse metric, `action_cap_hit_rate`, phase-1/phase-2/combined throughput reported.
10. Template-level assert that `reasoning_effort="high"` is rejected. Frozen ACT-XHIGH reference bound to bundle SHA + run id.
11. Fake-LLM scheduler test: real tokenizer ids in fake outputs; two new lineages (natural-close-cut continuation, repeated tool code).
12. `ACTION_RESERVE_TOKENS=768`: in the forced arm, thinking never consumes the last 768 tokens of the task budget, so the final turn
    always keeps room for an action (SMOKE emulation showed 8/8 budget deaths without it: 3×(3,072+action) left 0 action tokens).
13. `stop_without_think_close` diagnostic (phase 1 ended with EOS inside thinking).
14. **`PROBE_GPU` profile (v1.9c)** — Kaggle-only cheap tier before SMOKE: same 8 tasks / prompts / REPL / verifier, budget
    3,072 per turn · 5,120 total · 2 calls · 2 turns. Prints a per-arm quick readout the moment each arm ends and writes
    `probe_verdict.json` (`smoke_go_arms`). SMOKE_GPU reads `PROBE_APPROVAL` and runs only the arms that showed activation.
    Worst case ≈ 50 min GPU incl. model load; typical far less. Re-runnable in an interactive session (`PROBE_ARMS`).
15. v1.9d (after reading the Kaggle PREFLIGHT bundle 20260915T113233Z): `stop_without_think_close` now reaches the records;
    fake lineage G exercises and asserts the stateful-reuse metric; injected-stop-token assertion uses `STOP_THINKING_TEXT`;
    fake-LLM scheduler traces are bundled under `fake_llm_selftest/<arm>/`.


### v2.1 correctness hardening before GPU quality smoke

This revision does **not** add a solver mechanism. It fixes measurement and failure-handling issues found by independent CPU counterexamples:

1. Proxy promotion now requires verified all-train-exact **and at least one usable test prediction**; tool engagement is diagnostic only.
2. Competition-equivalent quality is the exact rate over **task test outputs**, matching Kaggle scoring. Strict all-query task success is retained only as a diagnostic.
3. vLLM `generate()` exceptions are converted into terminal failed records and an engine-error artifact instead of aborting evidence generation.
4. `verifier_executed` means a verifier subprocess really started; guard rejection is tracked separately.
5. Train verification and test execution are separate fresh-process stages, so a test timeout cannot erase already established train-exact evidence.
6. Phase-2 request seeds are logged explicitly; empty candidate registries retain their schema; the donor prompt runtime text is restored byte-faithfully.

A synchronous CUDA/vLLM call that hard-hangs inside native code still cannot be reliably interrupted from this in-process notebook. That residual risk is recorded explicitly and is not disguised as solved.


</details>


</details>


In [ ]:
# =====================================================================================
# CELL 1 — PIV-Q27-P0 CONFIG + OFFLINE ENVIRONMENT + FROZEN SOURCE CONTRACT
# Must run before importing torch/vLLM.
# =====================================================================================
import os
import sys
import json
import math
import time
import random
import hashlib
import zipfile
import shutil
import traceback
import copy
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from collections import deque

SEED = 42
random.seed(SEED)
EXPERIMENT_ID = "PIV-Q27-EVO12"
GENERATION_CONTRACT_VERSION = "piv-q27-v3_1-eval120-allfix"
PARSER_CONTRACT_VERSION = "piv-q27-toolxml+terminal-json+repair-json-v2.5"
PROMPT_CONTRACT_VERSION = "piv-q27-source-agentic-prompt-v1.9"

# C1 v2.3 provenance: donor and upstream evidence are frozen inputs to this derivative.
DONOR_NOTEBOOK_SHA256 = "007c8d68559c39a948dfef9d571fbd2e5da7335492a6f3330645db892648b525"
ACTIVATION_PROBE_RUN_ID = "20260915T124541Z"
ACTIVATION_PROBE_BUNDLE_SHA256 = "d400bbb1cbcf3bc05af58b56b5c85040f8362b13e099809c1f49a25d43b2d26a"
THROUGHPUT_SMOKE_RUN_ID = "20260916T091344Z"
THROUGHPUT_EVIDENCE_SHA256 = "c34fd3736dd9d2565ef681c6fd1936bb7eb1f99b3a95d222eda419049cd3e2aa"
C1_EXPECTED_LONG8_TPS = 129.173172
C0_FROZEN_LONG8_TPS = 48.300656

# -------------------------------------------------------------------------------------
# FROZEN SOURCE AUDIT
# -------------------------------------------------------------------------------------
SOURCE_FILE_SHA256 = {
    "qwen_exp1_base": "07d7c7a7ca4567a6118ae8cef00b10277aa37d5ada8698afb66afeb43c88aeb1",
    "qwen_runtime_benchmark": "7d91965196ec3309bf4e3f16f90efcaf2067ee8575d245c8eab9eb100f204b78",
    "arc2_advanced_eda": "1a9384d4b2e01fc15cddad95dee1d6eb5a633b6ae2d041d3d9ae97ffd1879f64",
    "nvarc_31_experiment_diagnostics": "69a64354b6b1c696af1f1c7a68984275d4db9b7eef5f2a382f879dc862d7c978",
    "portability_notes_v10": "c2dd4b45e3697d298d8f814b4087cee742ef4f8632f0eac318ba16cb8f8abda8",
    "experiment_master_plan": "8ca1c34c2c3d864c4b8e04fadf64465cbf468a09b3647c690f7306a07daf0c42"
}

PROXY48_TASK_IDS = [
    "0934a4d8",
    "20a9e565",
    "4c7dc4dd",
    "89565ca0",
    "136b0064",
    "291dc1e1",
    "16b78196",
    "271d71e2",
    "28a6681f",
    "2b83f449",
    "2c181942",
    "31f7f899",
    "35ab12c3",
    "3dc255db",
    "409aa875",
    "4c416de3",
    "64efde09",
    "6e453dd6",
    "7491f3cf",
    "7b0280bc",
    "7b80bb43",
    "7c66cb00",
    "1818057f",
    "332f06d7",
    "53fb4810",
    "7666fa5d",
    "da515329",
    "8f3a5a89",
    "db695cfb",
    "97d7923e",
    "de809cff",
    "c7f57c3e",
    "cb2d8a2c",
    "58490d8a",
    "dbff022c",
    "5961cc34",
    "80a900e0",
    "269e22fb",
    "8b9c3697",
    "e376de54",
    "b99e7126",
    "c4d067a0",
    "135a2760",
    "4a21e3da",
    "71e489b6",
    "800d221b",
    "dd6b8c4b",
    "36a08778"
]
PROXY48_ORDERED_SHA256 = hashlib.sha256("\n".join(PROXY48_TASK_IDS).encode("utf-8")).hexdigest()
PROXY48_EXPECTED_SHA256 = "8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134"
PROXY48_SOURCE_COUNTS = {
    "total": 48,
    "tier1": 32,
    "tier2": 12,
    "tier3": 4,
    "easy_static": 16,
    "rare": 30,
    "multi_query": 8,
    "dynamic": 8,
    "hard_tier1": 2,
    "high_ratio": 5,
    "extreme_ratio": 4,
}

if len(PROXY48_TASK_IDS) != 48 or len(set(PROXY48_TASK_IDS)) != 48:
    raise RuntimeError("Frozen Proxy-48 must contain exactly 48 unique task IDs.")
if PROXY48_ORDERED_SHA256 != PROXY48_EXPECTED_SHA256:
    raise RuntimeError("Frozen Proxy-48 fingerprint mismatch. Refusing to run a changed suite.")

# NVARC notebook diagnostic anchors. These are SOURCE diagnostics only, never Q1 results.
NVARC_DIAGNOSTIC_ANCHORS = {
    "outputs_analyzed": 172,
    "gt_present_in_pool": 55,
    "gt_pool_coverage": 55/172,
    "gt_selected_by_current_dual": 45,
    "current_dual_score_reported": 33.50,
    "oracle_score_reported": 41.17,
    "avg_raw_duplicate_ratio": 0.378,
    "scorer_rank_correlation_on_gt_present": 0.961,
    "purpose": "Motivate complementary candidate coverage and provenance-aware dedup; not used to tune Q1 predictions.",
}


# =====================================================================================
# ONE-LINE RUN SWITCH
# =====================================================================================
RUN_PROFILE = "EVAL120"   # default: all public evaluation tasks, exact scoring

PIV_SOURCE_COMMIT = "b3d4494892c6d85fa2a35d548271fb02ea3e191b"
PIV_SOURCE_REPO = "https://github.com/gutfeeling/arc-agi-2-submission"
PIV_EXTERNAL_SOURCES = {
    "technical_article": "https://pivotools.github.io/pivotools-quarto-blog/posts/agentic_coding_arc_agi/",
    "audited_repo_snapshot": f"https://github.com/gutfeeling/arc-agi-2-submission/tree/{PIV_SOURCE_COMMIT}",
    "public_experiment_traces": "https://huggingface.co/datasets/arcagi2/arcagi2-agentic-coding-publication",
    "qwen_model_card": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8",
    "qwen_chat_template": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8/blob/main/chat_template.jinja",
    "vllm_019_llm_api": "https://docs.vllm.ai/en/v0.19.0/api/vllm/entrypoints/llm/",
}
PIV_SOURCE_FILES = {
    "clients.py": {"sha": "34a85ed34001923a5efae715f509e5aa7e8e235f", "role": "persistent model/tool loop + reasoning/tool history"},
    "solver.py": {"sha": "cf2ef554cc389322a5689988b300dc2bc05bf36a", "role": "one code turn -> fresh all-train verification -> test execution"},
    "turn.py": {"sha": "fe4dfc69c5963a56dfba84d0bd49821ae0d2e0bc", "role": "last code block + required solution() artifact"},
    "tools/repl_tool.py": {"sha": "691839335b1e65ba0bd2e4b8b84b70c50d559683", "role": "python tool, errors/timeouts as feedback, puzzle protection"},
    "utils/puzzle_utils.py": {"sha": "7e6ec3034e0dc2f94fba3aa9f1d1d4a0896120b4", "role": "all-train/all-test-input serialization; test outputs hidden"},
    "utils/solving_utils.py": {"sha": "197be842b3d6fa08e4e98066e801dc8cf2bc8327", "role": "fresh sandbox train check and test execution"},
    "utils/code_templates/check_solution_on_training_examples.py": {"sha": "c13ea5b89360ea997a30a1eabe2ccb4a67489b44", "role": "deep-copy input + exact train equality"},
    "solver/prompts/agentic_coding_solver.txt": {"sha": "dab87e08bbe3f071920f13eb98f4a4d6f03ada10", "role": "ARC priors, preloaded puzzle, self-contained solution()"},
    "solver/prompts/system_prompt.txt": {"sha": "265131a50fb42deebe0a8da353f466b11d6d7d6b", "role": "stateful Python tool contract"},
    "solver/config/gpt_oss_120b_high.py": {"sha": "8fd563ddcb10962c7564fbfa8cbf3424a7a437d7", "role": "source agentic config example"},
    "sandbox/ipybox_sandbox.py": {"sha": "195e0e66ad7328a4f823974f04778b639a16412a", "role": "source Docker/IPyBox persistent REPL implementation; semantics only, not ported literally"},
    "sandbox/base.py": {"sha": "517f5794ff0cac64024e47c184d116b1f36577ea", "role": "source stateful REPL/Sandbox interface and fresh run_cells semantics"},
    "evaluation/evaluate.py": {"sha": "8b48fb4482dc7d9772ac010ba6dd81f6cc15e0a4", "role": "source multi-sample evaluation / majority aggregation; not copied into this single-sample smoke"},
    "utils/code_templates/produce_output_grid.py": {"sha": "b6ab435ca4f7d77414764e3a4400bb8c6dbead5e", "role": "execute verified solution on all test inputs"},
}

PROFILE_TABLE = {
    "PREFLIGHT": {
        "use_model": False, "task_set": "PROXY48", "active_tasks": 48,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_C1_V25_PREFLIGHT_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": False,
    },
    "TERMINAL_PILOT": {
        "use_model": True, "task_set": "PILOT2", "active_tasks": 2,
        "agent_batch_size": 2, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_C1_V25_TERMINAL_PILOT_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": True,
    },
    "EVAL120": {
        # Every public evaluation task. Budget and concurrency match the submission notebook on purpose.
        "use_model": True, "task_set": "EVAL120", "active_tasks": 120,
        "agent_batch_size": 12, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 8192, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_EVO12_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": True,
    },
    "ADAPTIVE_SMOKE": {
        "use_model": True, "task_set": "SMOKE8", "active_tasks": 8,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_C1_V25_ADAPTIVE_SMOKE_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": True,
    },
}
RUN_PROFILE = str(RUN_PROFILE).strip().upper()
if RUN_PROFILE not in PROFILE_TABLE:
    raise ValueError(f"RUN_PROFILE={RUN_PROFILE!r} invalid; choose {list(PROFILE_TABLE)}")
PROFILE = PROFILE_TABLE[RUN_PROFILE]
USE_MODEL = bool(PROFILE["use_model"])
TASK_SET = str(PROFILE["task_set"])
ACTIVE_TASK_LIMIT = int(PROFILE["active_tasks"])
# EVO12: first 12 of random.Random(20261004).sample(sorted(eval ids), 24). Empty list = all 120.
EVAL_SUBSET_TASK_IDS = ["16de56c4", "2d0172a1", "332f06d7", "4c3d4a41", "4e34c42c", "6e453dd6", "88bcf3b4", "a6f40cea", "b5ca7ac4", "c4d067a0", "c7f57c3e", "dbff022c"]
EVO_RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))   # scoring rerun after Submit: hidden test set
if EVO_RERUN:
    EVAL_SUBSET_TASK_IDS = []
print("EVO MODE:", "COMPETITION RERUN (hidden test -> submission.json)" if EVO_RERUN else "EVALUATION (12 frozen eval tasks, scored)")
if TASK_SET == "EVAL120" and EVAL_SUBSET_TASK_IDS:
    assert len(set(EVAL_SUBSET_TASK_IDS)) == len(EVAL_SUBSET_TASK_IDS), "duplicate subset task"
    ACTIVE_TASK_LIMIT = len(EVAL_SUBSET_TASK_IDS)
AGENT_BATCH_SIZE = int(PROFILE["agent_batch_size"])
MAX_TOOL_CALLS = int(PROFILE["max_tool_calls"])
MAX_MODEL_TURNS = int(PROFILE["max_model_turns"])
MAX_RETRIES = int(PROFILE.get("max_retries", 2))

# =====================================================================================
# v1.9d REASONING-ACTIVATION ARMS (solver mechanics frozen)
# =====================================================================================
# Official Qwen3.8 template values are xhigh / medium / low. 'high' is invalid for this model.
STOP_THINKING_SENTENCE = "Considering the limited time by the user, I have to give the solution based on the thinking directly now."
STOP_THINKING_TEXT = "\n\n" + STOP_THINKING_SENTENCE + "\n</think>\n\n"
assert STOP_THINKING_TEXT == "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
ARM_CONFIG = {
    "ACT-LOW":        {"reasoning_effort": "low",    "think_policy": "natural",      "think_budget": None},
    "ACT-MEDIUM":     {"reasoning_effort": "medium", "think_policy": "natural",      "think_budget": None},
    "ACT-BUDGET3072": {"reasoning_effort": "xhigh",  "think_policy": "forced_close", "think_budget": 3072},
    # v3: low effort keeps LOW's speed; forced </think> after 2,048 thinking tokens ends the first-turn runaway
    # (49/120 tasks spent all 6,144 exploration tokens transcribing grids and never acted).
    "ACT-LOW-FORCED": {"reasoning_effort": "low",    "think_policy": "forced_close", "think_budget": 2048},  # v2.4F: intentionally frozen; 8,192 exploration tokens typically allow ~2 full forced turns + partial third
}
ACTION_MAX_TOKENS = 2048
# v3: reserve a FULL action. With 768 the last forced turn's tool call was cut at exactly 768 tokens (v2.4G).
ACTION_RESERVE_TOKENS = ACTION_MAX_TOKENS

# v2.4 terminal-commit policy. Total task budget is unchanged; exploration may never consume this reserve.
EVAL120_MODE = (RUN_PROFILE == "EVAL120")
TASK_TOKEN_CAP = int(PROFILE["total_generated_token_cap"])
# v3: 38/120 finalizers were cut at exactly 2,048 tokens; successful ones peaked at 1,824.
FINAL_RESERVE_TOKENS = 4096
EXPLORATION_TOKEN_CAP = TASK_TOKEN_CAP - FINAL_RESERVE_TOKENS
FINALIZER_CONTEXT_MARGIN_TOKENS = 512
FINALIZER_CONTEXT_RESERVE_TOKENS = FINAL_RESERVE_TOKENS + FINALIZER_CONTEXT_MARGIN_TOKENS
FINALIZER_MAX_TOKENS = FINAL_RESERVE_TOKENS
FINALIZER_TURN_SLOTS = 1
TERMINAL_HANDOFF_MAX_TOKENS = 1024  # latest incomplete-reasoning tail, fit against full finalizer reserve

# v2.5 candidate-first repair policy. Total generated-token ceiling stays 12,288; repair uses only leftover tokens.
REPAIR_ROUNDS = 1
REPAIR_FEEDBACK_MAX_TOKENS = 600
REPAIR_MAX_TOKENS = FINALIZER_MAX_TOKENS  # actual cap = min(leftover total budget, context headroom, 4096)
REPAIR_SEED_OFFSET = 3_000_000
# v3.1 (review): repair is OFF for this run. In its current form it could exceed the 12,288-token task budget,
# ignored the global deadline, stored repaired programs without the helper library, and ranked candidates by exact
# pairs only. It returns later as a separate, fixed V3R ablation.
REPAIR_POLICY_ENABLED = False
# ---------------------------------------------------------------- v3 mechanisms (all on in this run)
AUTO_VERIFY_ENABLED = True                # fresh-verify solution() after every tool call; report goes to the model
CANDIDATE_COMMIT_VERBATIM = True          # a train-exact candidate is submitted as-is (no finalizer rewrite)
TERMINAL_STRUCTURED_HANDOFF = True        # finalizer sees best candidate + verifier report, not a reasoning tail
REPAIR_SELECTIVE = True                   # repair only near misses
REPAIR_NEAR_MISS_MAX_WRONG_FRACTION = 0.20
BACKFILL_ENABLED = True                   # refill a finished task's slot immediately
# v2.5A PRE-REGISTRATION (measured on the frozen v2.4G run 20260918T104631Z, not a bug):
# the repair round draws from the SAME TOTAL_GENERATED_TOKEN_CAP as exploration, so its cap is
# min(REPAIR_MAX_TOKENS, leftover budget, context headroom). Observed leftovers were 1,229-8,839 (ACT-LOW)
# and 2,175-4,315 (ACT-BUDGET3072). For ACT-LOW 271d71e2 / 2c181942 / 16b78196 the leftover (1,841 / 1,229 /
# 1,305) is SMALLER than candidate #1's own JSON size (2,254 / 2,866 / 2,790), so those repairs cannot even
# re-emit a program of the same length and are expected to end as repair_output_parse_failed (finish=length).
# Read any "repair helped k/N" result against this ceiling; a separate repair budget is a future decision.
REPAIR_BUDGET_PREREGISTRATION = {
    "repair_shares_total_generated_token_cap": True,
    "cap_formula": "min(REPAIR_MAX_TOKENS, TOTAL_GENERATED_TOKEN_CAP - generated_tokens, MAX_MODEL_LEN - prompt_tokens - CONTEXT_SAFETY_TOKENS)",
    "v24g_leftover_tokens": {"ACT-LOW": [1229, 1305, 1841, 2403, 2888, 3107, 3256, 8839],
                             "ACT-BUDGET3072": [2175, 2299, 2661, 2728, 2788, 2946, 3425, 4315]},
    "v24g_expected_structural_truncations": ["ACT-LOW/271d71e2", "ACT-LOW/2c181942", "ACT-LOW/16b78196"],
}
V24G_CANONICAL_SOURCE_SHA256 = "50b9acb8b62b3c1860914f562c5154dd58e8906b3342cd8ad6d8875fdc882769"
V24G_EXECUTED_SHA256 = "44934b207807f2b4a870304205b579f78daea3f23b64720aa6fec73455ffb66f"
V24G_EVIDENCE_BUNDLE_SHA256 = "387d4bb7383c9e7917c6deff5d2999e0ab83ff9b105d1487aa7dcd7a8cb56837"
# Deliberately frozen for the repair-only experiment. v2.4G observed five BUDGET late phase-2 768-token length hits;
# changing this here would create a second solver-policy intervention.
assert ACTION_RESERVE_TOKENS == ACTION_MAX_TOKENS
TERMINAL_POLICY_ENABLED = (RUN_PROFILE in {"TERMINAL_PILOT","ADAPTIVE_SMOKE","EVAL120"})
TERMINAL_PILOT_TASK_IDS = ["0934a4d8", "cb2d8a2c"]
FINALIZER_JSON_SCHEMA = {
    "type": "object",
    "properties": {"solution_code": {"type": "string"}},
    "required": ["solution_code"],
    "additionalProperties": False,
}
FINALIZER_SAMPLING = {
    "temperature": 0.1,   # v3: faithful commit, not exploration
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 0.0,
    "repetition_penalty": 1.0,
}
TERMINAL_FINALIZER_INSTRUCTION = """[TERMINAL COMMIT]
Exploration is over. You have no Python tool and must not continue analysis or ask for more tool use.
Return exactly one JSON object with the single key "solution_code".
The value must be a complete Python program defining `solution(grid)` that does not depend on variables or state
that existed only inside the earlier REPL. The preloaded helper functions (components, bbox, crop, rot90, flip_h,
flip_v, transpose, color_counts, background_color, grid_shape, diff_cells) are available; do not redefine them.
STRICT: no comments, no docstrings, no explanations, no alternative hypotheses. Do not reconsider the rule here:
commit the best rule found during exploration. If a best candidate is shown below, start from it and change only
what its verifier report shows to be wrong. Do not return Markdown or prose."""
assert EXPLORATION_TOKEN_CAP + FINAL_RESERVE_TOKENS == TASK_TOKEN_CAP

# Frozen real-model activation result from v1.9d PROBE_GPU.
PROBE_APPROVAL = {
    "probe_run_id": ACTIVATION_PROBE_RUN_ID,
    "arms": ["ACT-LOW", "ACT-BUDGET3072"],
}
QUALITY_SMOKE_ARMS = ["ACT-LOW", "ACT-BUDGET3072"]
assert PROBE_APPROVAL["probe_run_id"] == "20260915T124541Z"
assert QUALITY_SMOKE_ARMS == PROBE_APPROVAL["arms"]
assert "ACT-MEDIUM" not in QUALITY_SMOKE_ARMS

# PREFLIGHT remains available for debugging, but ADAPTIVE_SMOKE is the default competition workflow.
# One arm only: the whole budget goes into covering all 120 tasks with the recipe the submission uses.
EVAL120_ARM = "ACT-LOW-FORCED"
ARMS = ([EVAL120_ARM] if EVAL120_MODE else
        (list(ARM_CONFIG) if RUN_PROFILE == "PREFLIGHT" else list(QUALITY_SMOKE_ARMS)))
IS_PROBE = False
for _a in ARMS:
    if ARM_CONFIG[_a]["reasoning_effort"] not in ("xhigh", "medium", "low"):
        raise ValueError(f"{_a}: reasoning_effort must be one of xhigh/medium/low for the Qwen3.8 template")

# Frozen reference row = v1.7 SMOKE_GPU (run 20260914T122911Z). Not rerun; used only for deltas/comparison.
FROZEN_XHIGH_REFERENCE = {
    "bundle_sha256": "7435b795691d1542100f7f5cd0764ebcdacc61be6128f236c845f10eb22b143f",   # PIV_Q27_P0_SMOKE_GPU_BUNDLE.zip (v1.7)
    "run_id": "20260914T122911Z", "notebook": "experiment-1-arc-agi-2-with-qwen-3-8 (11).ipynb",
    "task_ids": ["0934a4d8","16b78196","2c181942","271d71e2","136b0064","cb2d8a2c","291dc1e1","28a6681f"],
    "reasoning_effort": "xhigh", "think_policy": "natural", "think_budget": None, "tasks": 8,
    "tool_engagement_rate": 0.0, "final_program_rate": 0.0, "all_train_exact_rate": 0.0, "output_weighted_exact": 0.0,
    "budget_death_rate": 1.0, "infrastructure_failure_rate": 0.0, "first_turn_length_rate": 1.0,
    "natural_close_rate_turns": 0.0, "forced_close_rate_turns": 0.0, "think_tokens_per_turn_median": 12288,
    "executed_calls_per_task_mean": 0.0, "generated_tokens_per_task_mean": 12288.0,
    "sec_per_output_token": 0.16014132123575847, "projected_proxy48_hours_20pct": 4.056694742527999,
    "compute_gate_pass": True, "activation_pass": False, "productivity_pass": False, "quality_signal": False,
    "promotion_candidate": False, "tier_reached": "none",
}
VERDICT_THRESHOLDS = {"activation_tool_engagement_rate": 0.50, "productivity_final_program_rate": 0.25,
                      "compute_budget_death_rate_max": 0.50, "compute_projected_hours_20pct_max": 10.0}

# Pre-registered measurement contract: every effect below is computed after the run from records/traces/waves;
# nothing is added post hoc. 'post_hoc' items need trace inspection and are reported by the external analysis.
MEASUREMENT_CONTRACT = {
 "L1_thinking_behavior":  {"metrics": ["natural_think_close_rate_turns","forced_think_close_rate_turns","think_tokens_per_turn","think_tokens_per_task","first_turn_finish_reasons","tasks_with_any_natural_close"], "computed_by": "notebook"},
 "L2_tool_activation":    {"metrics": ["tool_engagement_rate_tasks","tool_turn_rate_turns","executed_calls_total","executed_calls_per_task","first_tool_turn","first_tool_generated_tokens","tasks_never_calling_tool"], "computed_by": "notebook"},
 "L3_tool_usefulness":    {"metrics": ["tool_ok_total","tool_exception_total","guard_rejections_total","tool_timeouts_total","budget_denials_total","puzzle_keys_only_calls","duplicate_tool_code_rate"], "computed_by": "notebook"},
 "L4_stateful_benefit":   {"metrics": ["worker_restarts_total","tasks_with_2plus_calls","stateful_reuse_events_total","tasks_with_stateful_reuse","repeated_tool_calls_total"], "computed_by": "notebook"},
 "L5_reason_action_transitions": {"metrics": ["transition_counts","turns_per_task","turn_kind_totals"], "computed_by": "notebook"},
 "L6_program_production": {"metrics": ["final_program_rate","final_programs","syntax_valid_program_rate","salvaged_final_on_length_total","final_code_chars"], "computed_by": "notebook"},
 "L7_program_correctness":{"metrics": ["verifier_execution_rate","all_train_exact_rate","all_train_exact_tasks","train_exact_pairs","verifier_status_counts"], "computed_by": "notebook"},
 "L8_test_execution":     {"metrics": ["test_execution_ok_tasks","any_test_output_ok_tasks","valid_grid_rate_outputs"], "computed_by": "notebook"},
 "L9_arc_quality":        {"metrics": ["competition_output_exact_rate","all_queries_exact_task_rate","task_normalized_exact","solved_task_ids_strict"], "computed_by": "notebook"},
 "L10_budget_behavior":   {"metrics": ["generated_tokens_per_task","generated_tokens_per_turn","injected_stop_tokens_total","budget_death_rate","tokens_at_budget_death","context_headroom_deaths","stop_reason_counts"], "computed_by": "notebook"},
 "L11_wall_clock":        {"metrics": ["wave_runtime","projected_proxy48_total_hours","projected_proxy48_total_hours_20pct_margin"], "computed_by": "notebook"},
 "L12_batch_scheduler":   {"metrics": ["waves","active_count_per_wave","idle_tail_fraction_per_wave"], "computed_by": "notebook"},
 "L13_forced_close_effects": {"metrics": ["forced_turns","post_close_action_counts","natural_turn_action_counts","phase2_skipped_reasons","phase2_tokens"], "computed_by": "notebook"},
 "L14_retry_effects":     {"metrics": ["step_retries_total","source_retries_total","tokens_spent_in_retried_turns","retry_reason_counts"], "computed_by": "notebook"},
 "L15_parser_interface":  {"metrics": ["malformed_tool_turns","text_turns_no_action","salvaged_final_on_length"], "computed_by": "notebook"},
 "L16_sandbox_safety":    {"metrics": ["guard_rejections","tool_timeouts","worker_restarts"], "computed_by": "notebook"},
 "L17_diversity":         {"metrics": ["unique_final_code_hashes","unique_predicted_grids","unique_tool_code_hashes_per_task"], "computed_by": "notebook"},
 "L18_eda_slices":        {"metrics": ["category_scores per tier/dynamic/multi_query/high_ratio/extreme_ratio/easy_static/rare"], "computed_by": "notebook"},
 "L19_baseline_deltas":   {"metrics": ["delta vs frozen ACT-XHIGH for engagement/final-program/train-exact/exact/budget-death/first-turn-length/tokens"], "computed_by": "notebook"},
 "L20_provenance":        {"metrics": ["arm","reasoning_effort","think_policy","seed","prompt_sha","template_sha","engine/model/runtime fingerprints","verifier_contract"], "computed_by": "notebook"},
 "L21_matched_arm_quality":{"metrics": ["per-task LOW vs BUDGET3072 executable/train-exact/test-exact/runtime comparison"], "computed_by": "notebook"},
 "L22_terminal_commit":{"metrics": ["finalizer_started","finalizer_trigger_reason","finalizer_prompt_tokens","finalizer_output_tokens","finalizer_finish_reason","terminal_parse_status","feedback_present_in_rendered_prompt","terminal_handoff_source_tokens","terminal_handoff_kept_tokens","terminal_handoff_truncated"], "computed_by": "notebook"},
 "verdict_tiers": {"activation_diagnostic": "tool_engagement_rate >= 0.50 (never a promotion prerequisite)", "productivity_diagnostic": "final_program_rate >= 0.25 and infrastructure_failure_rate == 0",
                   "quality": "all_train_exact_rate > 0", "usable_test": "at least one valid test prediction from a train-exact program",
                   "promotion": "quality and usable_test and compute_gate and no infrastructure failure",
                   "compute_gate": "budget_death_rate <= 0.50 and projected_20pct <= 10h and complete unique terminal coverage"},
}
PER_TURN_MAX_TOKENS = int(PROFILE["per_turn_max_tokens"])
TOTAL_GENERATED_TOKEN_CAP = int(PROFILE["total_generated_token_cap"])
if TOTAL_GENERATED_TOKEN_CAP != 12288:
    raise RuntimeError(f"unexpected total task budget {TOTAL_GENERATED_TOKEN_CAP} for profile {RUN_PROFILE}")
if EXPLORATION_TOKEN_CAP != TOTAL_GENERATED_TOKEN_CAP - FINAL_RESERVE_TOKENS:
    raise RuntimeError("terminal reserve arithmetic drift")
TOOL_TIMEOUT_SEC = float(PROFILE["tool_timeout_sec"])
VERIFIER_TIMEOUT_SEC = float(PROFILE["verifier_timeout_sec"])
VERIFIER_CONTRACT_VERSION = "fresh-train-plus-per-test-stage-v2.2"
VERIFIER_TIMEOUT_CONTRACT = {
    "version": VERIFIER_CONTRACT_VERSION,
    "train_stage_timeout_sec": VERIFIER_TIMEOUT_SEC,
    "per_test_stage_timeout_sec": VERIFIER_TIMEOUT_SEC,
    "aggregate_candidate_timeout_sec": None,
    "semantics": "fresh process for train, then one fresh process per test input; timeout applies independently to each stage",
    "comparison_rule": "Any matched C0/C1 comparison must use this identical verifier contract.",
}
BUNDLE_NAME = str(PROFILE["bundle_name"])
ACCURACY_IS_REAL = bool(PROFILE["accuracy_is_real"])

MODEL_PATH = Path("/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1")
WHEELHOUSE = Path("/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3")
ARC_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
TRAIN_CHALLENGES_PATH = ARC_ROOT / "arc-agi_training_challenges.json"
TRAIN_SOLUTIONS_PATH  = ARC_ROOT / "arc-agi_training_solutions.json"
EVAL_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS_PATH   = ARC_ROOT / "arc-agi_evaluation_solutions.json"
TEST_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_test_challenges.json"
SAMPLE_SUBMISSION_PATH = ARC_ROOT / "sample_submission.json"

BASE_WORK_ROOT = Path("/kaggle/working/piv_q27_eval120")
WORK_ROOT = BASE_WORK_ROOT / RUN_PROFILE.lower()
# v2.4C: reset_outputs wipes WORK_ROOT on every Run All, so the checkpoint/resume machinery only
# works WITHIN one session (adaptive stage A -> stage B). Set RESUME_PREVIOUS_SESSION=True to keep a
# previous session's records. NOTE: wave metrics are filtered by RUN_ID, so a resumed run reports
# wall-clock only for the tasks it actually executed in the new session.
RESUME_PREVIOUS_SESSION = False
if PROFILE["reset_outputs"] and WORK_ROOT.exists() and not RESUME_PREVIOUS_SESSION:
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NOTEBOOK_WALL_START = time.perf_counter()
TOTAL_RUNTIME_SEC = 12 * 60 * 60
# v1.6: one wave may run for the remaining P0 budget (up to 12,288 tokens ~= 33 min at the Q1-measured
# ~0.16 s/step). The reserve must exceed the longest possible wave plus bundle time, and each wave is
# additionally pre-checked against a projection built from measured wave timings.
SAFETY_RESERVE_SEC = (45 * 60) if EVAL120_MODE else (60 * 60)
DEFAULT_SEC_PER_OUTPUT_TOKEN = 0.20   # projection prior before the first measured wave
WAVE_TIME_SAFETY_FACTOR = 1.25
WAVE_OVERHEAD_SEC = 90.0              # prefill / scheduling / tool + verifier allowance per wave

# Frozen from the successful Q1 / v1.9d lineage. C1 changes only enforce_eager in Cell 7.
MAX_MODEL_LEN = 29696
MAX_NUM_BATCHED_TOKENS = 32768
MAX_NUM_SEQS = 16
GPU_MEMORY_UTILIZATION = 0.90
# C1 terminal pilot keeps prefix caching explicitly OFF. Do not change it in this experiment.
ENABLE_PREFIX_CACHING = False
assert ENABLE_PREFIX_CACHING is False

# EXP-0 diagnostics preserved unchanged.
WEIGHT_CLIP_LOW = 0.20
WEIGHT_CLIP_HIGH = 5.00
N_PROPENSITY_FOLDS = 5
HIGH_RATIO_THRESHOLD = 10.0
LARGE_TEST_INPUT_CELLS = 300
MANY_COLORS_THRESHOLD = 7
LOW_BG_THRESHOLD = 0.30

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("=" * 116)
print(EXPERIMENT_ID, "| Stateful Python REPL + Fresh Exact Train Verifier")
print("=" * 116)
print("RUN_ID:", RUN_ID, "| RUN_PROFILE:", RUN_PROFILE, "| TASK_SET:", TASK_SET)
print("PIV source commit:", PIV_SOURCE_COMMIT)
print("Agent budget: calls/turns/per-turn/total =", MAX_TOOL_CALLS, MAX_MODEL_TURNS, PER_TURN_MAX_TOKENS, TOTAL_GENERATED_TOKEN_CAP)
print("ARMS:", ARMS, "| ACTION_MAX_TOKENS:", ACTION_MAX_TOKENS)
print("Terminal policy:", TERMINAL_POLICY_ENABLED, "| explore/final reserve:", EXPLORATION_TOKEN_CAP, FINAL_RESERVE_TOKENS)
print("Proxy-48 SHA256:", PROXY48_ORDERED_SHA256)
print("WORK_ROOT:", WORK_ROOT, "| wheelhouse attached:", WHEELHOUSE.exists())

required = [MODEL_PATH, ARC_ROOT, TRAIN_CHALLENGES_PATH, TRAIN_SOLUTIONS_PATH,
            EVAL_CHALLENGES_PATH, EVAL_SOLUTIONS_PATH, TEST_CHALLENGES_PATH, SAMPLE_SUBMISSION_PATH]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Required Kaggle input is missing: {p}")
config_path = MODEL_PATH / "config.json"
index_path = MODEL_PATH / "model.safetensors.index.json"
with open(config_path, "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
qcfg = model_cfg.get("quantization_config", {})
if model_cfg.get("model_type") != "qwen3_5":
    raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
    raise RuntimeError(f"Unexpected quantization_config={qcfg}")

def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

def stable_json_sha256(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def small_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_RUNTIME_PACKAGES = {"vllm":"0.19.0", "transformers":"4.57.6", "tokenizers":"0.22.2", "huggingface-hub":"0.36.2"}
_loaded_module_names = {"vllm":"vllm", "transformers":"transformers", "tokenizers":"tokenizers", "huggingface-hub":"huggingface_hub"}
before_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for _pkg, _module in _loaded_module_names.items():
    if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _module in sys.modules:
        raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart Kaggle session before Run All.")

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(
            f"These runtime packages must be installed offline: {requirements}, but the wheelhouse is not attached: "
            f"{WHEELHOUSE}. Attach that dataset, or run on an image that already ships the pinned versions.")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])
if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
    offline_pip_install(["vllm==0.19.0"])
else:
    repair=[]
    for package in ["transformers","tokenizers","huggingface-hub"]:
        if before_stack[package] != EXPECTED_RUNTIME_PACKAGES[package]:
            module_name = package.replace("-", "_")
            if module_name in sys.modules or (package=="huggingface-hub" and "huggingface_hub" in sys.modules):
                raise RuntimeError(f"{package} already imported with wrong version; restart session.")
            repair.append(f"{package}=={EXPECTED_RUNTIME_PACKAGES[package]}")
    offline_pip_install(repair)
after_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for package, wanted in EXPECTED_RUNTIME_PACKAGES.items():
    if after_stack.get(package) != wanted:
        raise RuntimeError(f"Runtime parity failure: {package}={after_stack.get(package)!r}, expected {wanted!r}")
print("Runtime dependency parity PASS:", after_stack)


In [ ]:

# =====================================================================================
# CELL 2 — LOAD ARC DATA
# =====================================================================================
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(SEED)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_challenges = load_json(TRAIN_CHALLENGES_PATH)
train_solutions  = load_json(TRAIN_SOLUTIONS_PATH)
eval_challenges  = load_json(EVAL_CHALLENGES_PATH)
eval_solutions   = load_json(EVAL_SOLUTIONS_PATH)
test_challenges  = load_json(TEST_CHALLENGES_PATH)
sample_submission = load_json(SAMPLE_SUBMISSION_PATH)

assert set(eval_challenges) == set(eval_solutions)
assert set(test_challenges) == set(sample_submission)
missing_proxy_ids = [k for k in PROXY48_TASK_IDS if k not in eval_challenges or k not in eval_solutions]
if missing_proxy_ids:
    raise RuntimeError(f"Frozen Proxy-48 IDs missing from public Evaluation data: {missing_proxy_ids}")

print("Training tasks       :", len(train_challenges))
print("Evaluation tasks     :", len(eval_challenges))
print("Evaluation outputs   :", sum(len(v) for v in eval_solutions.values()))
print("Test tasks           :", len(test_challenges))
print("Visible test queries :", sum(len(v["test"]) for v in test_challenges.values()))


In [ ]:

# =====================================================================================
# CELL 3 — EXP-0 STRUCTURAL FEATURES + TEST-RELEVANCE WEIGHTS
# Recomputed here so EXP-1 is self-contained.
# =====================================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, log_loss

def arr(grid):
    return np.asarray(grid, dtype=np.int16)

def is_rectangular_grid(grid, max_hw=30):
    if not isinstance(grid, list) or len(grid) < 1 or len(grid) > max_hw:
        return False
    if not all(isinstance(row, list) for row in grid):
        return False
    widths = [len(row) for row in grid]
    if min(widths, default=0) < 1 or max(widths, default=0) > max_hw or len(set(widths)) != 1:
        return False
    for row in grid:
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, np.integer)):
                return False
            if int(x) < 0 or int(x) > 9:
                return False
    return True

def grid_basic_stats(grid):
    a = arr(grid)
    vals, counts = np.unique(a, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())
    return {
        "h": int(a.shape[0]),
        "w": int(a.shape[1]),
        "cells": int(a.size),
        "n_colors": int(len(vals)),
        "bg0_density": float(np.mean(a == 0)),
        "color_entropy": entropy,
    }

def colored_components_count(grid):
    # Diagnostic only: 4-connected same-color non-zero components.
    a = arr(grid)
    h, w = a.shape
    seen = np.zeros((h, w), dtype=bool)
    n = 0
    sizes = []
    for r in range(h):
        for c in range(w):
            if seen[r, c] or a[r, c] == 0:
                continue
            color = a[r, c]
            q = deque([(r, c)])
            seen[r, c] = True
            size = 0
            while q:
                rr, cc = q.popleft()
                size += 1
                for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                    nr, nc = rr+dr, cc+dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr, nc] and a[nr, nc] == color:
                        seen[nr, nc] = True
                        q.append((nr, nc))
            n += 1
            sizes.append(size)
    return n, (float(np.mean(sizes)) if sizes else 0.0)

def summarize(values, prefix):
    x = np.asarray(values, dtype=float)
    if len(x) == 0:
        return {
            f"{prefix}_mean": 0.0, f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0, f"{prefix}_max": 0.0,
        }
    return {
        f"{prefix}_mean": float(x.mean()),
        f"{prefix}_std": float(x.std()),
        f"{prefix}_min": float(x.min()),
        f"{prefix}_max": float(x.max()),
    }

def task_features(task_id, task, split):
    train = task["train"]
    tests = task["test"]

    tr_in_stats, tr_out_stats, te_in_stats = [], [], []
    tr_in_components, tr_out_components, te_components = [], [], []
    tr_in_component_sizes, tr_out_component_sizes, te_component_sizes = [], [], []
    ratios, h_ratios, w_ratios, same_shape = [], [], [], []
    all_visible_colors = set()

    for ex in train:
        si = grid_basic_stats(ex["input"])
        so = grid_basic_stats(ex["output"])
        tr_in_stats.append(si); tr_out_stats.append(so)
        ci, csi = colored_components_count(ex["input"])
        co, cso = colored_components_count(ex["output"])
        tr_in_components.append(ci); tr_out_components.append(co)
        tr_in_component_sizes.append(csi); tr_out_component_sizes.append(cso)
        ratios.append(so["cells"] / max(si["cells"], 1))
        h_ratios.append(so["h"] / max(si["h"], 1))
        w_ratios.append(so["w"] / max(si["w"], 1))
        same_shape.append(int((si["h"], si["w"]) == (so["h"], so["w"])))
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())
        all_visible_colors.update(np.unique(arr(ex["output"])).tolist())

    for ex in tests:
        st = grid_basic_stats(ex["input"])
        te_in_stats.append(st)
        ct, cst = colored_components_count(ex["input"])
        te_components.append(ct); te_component_sizes.append(cst)
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())

    feat = {
        "task_id": task_id,
        "split": split,
        "n_train": len(train),
        "n_test_queries": len(tests),
        "visible_n_colors": len(all_visible_colors),
        "same_shape_frac": float(np.mean(same_shape)),
        "dynamic_any": int(any(x == 0 for x in same_shape)),
        "multi_query": int(len(tests) > 1),
    }

    for key in ["h", "w", "cells", "n_colors", "bg0_density", "color_entropy"]:
        feat.update(summarize([x[key] for x in tr_in_stats], f"tr_in_{key}"))
        feat.update(summarize([x[key] for x in tr_out_stats], f"tr_out_{key}"))
        feat.update(summarize([x[key] for x in te_in_stats], f"te_in_{key}"))

    feat.update(summarize(ratios, "area_ratio"))
    feat.update(summarize(h_ratios, "h_ratio"))
    feat.update(summarize(w_ratios, "w_ratio"))
    feat.update(summarize(tr_in_components, "tr_in_components"))
    feat.update(summarize(tr_out_components, "tr_out_components"))
    feat.update(summarize(te_components, "te_in_components"))
    feat.update(summarize(tr_in_component_sizes, "tr_in_component_size"))
    feat.update(summarize(tr_out_component_sizes, "tr_out_component_size"))
    feat.update(summarize(te_component_sizes, "te_in_component_size"))

    feat["n_train_eq_2"] = int(len(train) == 2)
    feat["n_train_ge_3"] = int(len(train) >= 3)
    feat["high_ratio"] = int(max(ratios) >= HIGH_RATIO_THRESHOLD)
    feat["large_test_input"] = int(max(x["cells"] for x in te_in_stats) >= LARGE_TEST_INPUT_CELLS)
    feat["many_colors"] = int(len(all_visible_colors) >= MANY_COLORS_THRESHOLD)
    feat["low_bg_train_input"] = int(
        np.mean([x["bg0_density"] for x in tr_in_stats]) < LOW_BG_THRESHOLD
    )
    return feat

def build_feature_df(data, split):
    return pd.DataFrame(
        [task_features(k, v, split) for k, v in data.items()]
    ).sort_values("task_id").reset_index(drop=True)

train_feat = build_feature_df(train_challenges, "train")
eval_feat  = build_feature_df(eval_challenges, "eval")
test_feat  = build_feature_df(test_challenges, "test")

for name, df in [("train", train_feat), ("eval", eval_feat), ("test", test_feat)]:
    df.to_csv(WORK_ROOT / f"task_features_{name}.csv", index=False)

numeric_cols = [
    c for c in eval_feat.columns
    if c not in {"task_id", "split"} and pd.api.types.is_numeric_dtype(eval_feat[c])
]
EXCLUDE_PROPENSITY = {
    "task_id", "split",
    "n_train_eq_2", "n_train_ge_3", "high_ratio", "large_test_input",
    "many_colors", "low_bg_train_input", "multi_query",
}
prop_cols = [c for c in numeric_cols if c not in EXCLUDE_PROPENSITY]

combined = pd.concat(
    [eval_feat.assign(_is_test=0), test_feat.assign(_is_test=1)],
    ignore_index=True
)
X = combined[prop_cols].astype(float)
y = combined["_is_test"].astype(int).to_numpy()

prop_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        C=0.5, penalty="l2", solver="liblinear",
        max_iter=5000, random_state=SEED
    )),
])
cv = StratifiedKFold(n_splits=N_PROPENSITY_FOLDS, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(prop_model, X, y, cv=cv, method="predict_proba")[:, 1]

prop_auc = roc_auc_score(y, p_oof)
prop_logloss = log_loss(y, p_oof)

n_eval = len(eval_feat)
n_test = len(test_feat)
prior_eval = n_eval / (n_eval + n_test)
prior_test = n_test / (n_eval + n_test)

p_eval_as_test = np.clip(p_oof[:n_eval], 1e-4, 1 - 1e-4)
density_raw = (p_eval_as_test / (1 - p_eval_as_test)) * (prior_eval / prior_test)
density_clip = np.clip(density_raw, WEIGHT_CLIP_LOW, WEIGHT_CLIP_HIGH)
density_norm = density_clip / np.mean(density_clip)

eval_weights = eval_feat[["task_id"]].copy()
eval_weights["p_test_oof"] = p_eval_as_test
eval_weights["density_ratio_raw"] = density_raw
eval_weights["weight"] = density_norm
eval_weights["test_relevance_rank_pct"] = pd.Series(density_norm).rank(pct=True).to_numpy()
eval_weights["test_relevance_quartile"] = pd.qcut(
    pd.Series(density_norm).rank(method="first"),
    q=4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"]
).astype(str).to_numpy()

weight_ess = float((density_norm.sum() ** 2) / np.sum(density_norm ** 2))
eval_weights.to_csv(WORK_ROOT / "eval_test_propensity_weights.csv", index=False)

print("EXP-0 reproduction:")
print("  Eval-vs-Test CV AUC :", round(prop_auc, 6))
print("  Log loss            :", round(prop_logloss, 6))
print("  Weight ESS          :", round(weight_ess, 3), "/", len(eval_weights))
print("  Weight min/median/max:",
      round(float(density_norm.min()), 4),
      round(float(np.median(density_norm)), 4),
      round(float(density_norm.max()), 4))


# =====================================================================================
# SOURCE-ALIGNED PROXY-48 FEATURES
# Exact feature/tag rules copied from the source EDA notebook for audit comparability.
# These DO NOT replace EXP-0 features or official scoring.
# =====================================================================================
def source_eda_n_cells(grid):
    return int(np.prod(np.shape(grid)))

def source_eda_proxy_features(task_id, data):
    puzzle = data[task_id]
    train = puzzle["train"]
    input_sizes = [source_eda_n_cells(ex["input"]) for ex in train]
    output_sizes = [source_eda_n_cells(ex["output"]) for ex in train]
    ratios = [max(i, o) / max(min(i, o), 1) for i, o in zip(input_sizes, output_sizes)]
    colors = set()
    pixels = []
    for ex in train:
        for field in ["input", "output"]:
            a = np.asarray(ex[field])
            colors.update(np.unique(a).tolist())
            pixels.extend(a.reshape(-1).tolist())

    avg_input_size = float(np.mean(input_sizes))
    avg_ratio = float(np.mean(ratios))
    max_ratio = float(max(ratios))
    n_colors = len(colors)
    n_train = len(train)
    n_queries = len(puzzle["test"])
    is_dynamic = any(np.shape(ex["input"]) != np.shape(ex["output"]) for ex in train)
    is_rare = bool(6 in colors or 9 in colors)
    bg_density = pixels.count(0) / max(len(pixels), 1)

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )
    if n_train >= 6 and tier < 3:
        fine *= 0.7

    tags = [f"tier{tier}"]
    if tier == 1 and (not is_dynamic) and max_ratio == 1 and n_colors <= 5:
        tags.append("easy_static")
    if tier == 1 and fine <= 0.22:
        tags.append("easy")
    if tier == 1 and fine > 0.45:
        tags.append("hard_tier1")
    if is_dynamic:
        tags.append("dynamic")
    if is_rare:
        tags.append("rare")
    if n_queries > 1:
        tags.append("multi_query")
    if max_ratio > 5:
        tags.append("high_ratio")
    if max_ratio > 30:
        tags.append("extreme_ratio")
    if n_colors >= 8:
        tags.append("many_colors")
    if avg_input_size >= 500:
        tags.append("large")
    if bg_density >= 0.70:
        tags.append("sparse_bg")
    if bg_density <= 0.30:
        tags.append("dense_non_bg")
    if n_train <= 2:
        tags.append("low_train")

    return {
        "task_id": task_id,
        "proxy_tier": int(tier),
        "proxy_fine": float(fine),
        "proxy_avg_input_size": avg_input_size,
        "proxy_max_ratio": max_ratio,
        "proxy_avg_ratio": avg_ratio,
        "proxy_n_colors": int(n_colors),
        "proxy_dynamic": int(is_dynamic),
        "proxy_rare": int(is_rare),
        "proxy_n_train": int(n_train),
        "proxy_n_queries": int(n_queries),
        "proxy_bg_density": float(bg_density),
        "proxy_easy_static": int("easy_static" in tags),
        "proxy_hard_tier1": int("hard_tier1" in tags),
        "proxy_high_ratio": int("high_ratio" in tags),
        "proxy_extreme_ratio": int("extreme_ratio" in tags),
        "proxy_tags": "|".join(tags),
    }

proxy_source_feat = pd.DataFrame([
    source_eda_proxy_features(k, eval_challenges) for k in eval_challenges
]).sort_values("task_id").reset_index(drop=True)
proxy_source_feat.to_csv(WORK_ROOT / "source_eda_features_eval.csv", index=False)

proxy48_source = proxy_source_feat[proxy_source_feat["task_id"].isin(PROXY48_TASK_IDS)].copy()
proxy48_order = {k: i for i, k in enumerate(PROXY48_TASK_IDS)}
proxy48_source["_order"] = proxy48_source["task_id"].map(proxy48_order)
proxy48_source = proxy48_source.sort_values("_order").drop(columns="_order").reset_index(drop=True)
proxy48_source.to_csv(WORK_ROOT / "proxy48_source_eda.csv", index=False)

proxy_source_counts = {
    "total": int(len(proxy48_source)),
    "tier1": int((proxy48_source["proxy_tier"] == 1).sum()),
    "tier2": int((proxy48_source["proxy_tier"] == 2).sum()),
    "tier3": int((proxy48_source["proxy_tier"] == 3).sum()),
    "easy_static": int(proxy48_source["proxy_easy_static"].sum()),
    "rare": int(proxy48_source["proxy_rare"].sum()),
    "multi_query": int((proxy48_source["proxy_n_queries"] > 1).sum()),
    "dynamic": int(proxy48_source["proxy_dynamic"].sum()),
    "hard_tier1": int(proxy48_source["proxy_hard_tier1"].sum()),
    "high_ratio": int(proxy48_source["proxy_high_ratio"].sum()),
    "extreme_ratio": int(proxy48_source["proxy_extreme_ratio"].sum()),
}
if proxy_source_counts != PROXY48_SOURCE_COUNTS:
    raise RuntimeError(
        "Proxy-48 source-EDA profile drift detected. "
        f"expected={PROXY48_SOURCE_COUNTS}, actual={proxy_source_counts}"
    )
with open(WORK_ROOT / "proxy48_source_distribution.json", "w", encoding="utf-8") as f:
    json.dump(proxy_source_counts, f, indent=2)
print("Proxy-48 source-EDA contract: PASS", proxy_source_counts)


In [ ]:
# =====================================================================================
# CELL 4 — TOKENIZER + SOURCE-ALIGNED AGENTIC PROMPT + QWEN TOOL TEMPLATE + ACTIVE TASKS
# =====================================================================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True, trust_remote_code=False)
print("Tokenizer:", tokenizer.__class__.__name__, "| vocab:", len(tokenizer))

# OpenAI-style schema; Qwen3.8's official chat template serializes it into its XML tool-call contract.
PYTHON_TOOL = {
    "type": "function",
    "function": {
        "name": "python",
        "description": "Execute Python in a persistent per-puzzle REPL and return stdout or a concise error.",
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "Python code to execute"}},
            "required": ["code"],
            "additionalProperties": False,
        },
    },
}
TOOL_SCHEMA_SHA256 = stable_json_sha256(PYTHON_TOOL)

SOURCE_SYSTEM_PROMPT = '# Tools\n\n## python\n\nUse this tool to execute Python code in your chain of thought. \n\nWhen you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. The environment supports common third party libraries like numpy, scipy, shapely, networkx, scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver. python will respond with the output of the execution or time out after 120.0 seconds. \n\n### Usage guideline\n\n1. Use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions.\n2. Prefer to write reusable functions. These will persist in the stateful Jupyter notebook and can be referenced in later code snippets without having to define them again.'
SOURCE_AGENTIC_PROMPT_TEMPLATE = 'You are an expert ARC AGI puzzle solver. Please solve the puzzle given below.\n\n# Background information about ARC AGI puzzles\n\nEach task consists of a small number of demonstration examples (3.3 on average), and a small number of test examples (1 - 3).\n\nEach example consists of an “input grid” and an “output grid”. Each “grid” is a literal grid of colored squares There are 10 unique colors. A grid can be any height or width between 1x1 and 30x30, inclusive.\n\nThe color code is as follows:\n\n0 - black\n1 - navy blue\n2 - red\n3 - green\n4 - yellow\n5 - gray\n6 - pink\n7 - orange\n8 - sky blue\n9 - maroon\n\nWhen solving an evaluation task, a test-taker has access to the training examples for the task (both the input and output grids), as well as the input grid of the test examples for the task. The test-taker must construct on its own the output grid corresponding to the input grid of each test example. “Constructing the output grid” is done entirely from scratch, meaning that the test-taker must decide what the height and width of the output grid should be, what colors they should place on the grid, and where. The task is successfully solved if the test-taker can produce the exact correct answer on all test examples for the task (binary measure of success).\n\nThe mapping from integers (0-9) to color names is arbitrary and should be treated as categorical labels; the transformation rule never relies on the specific integer\u2013color pairing.\n\nARC AGI puzzle solutions feel "natural" to humans. Natural rules tend to build on visual priors and small-number counting. They are visually immediate or prominent for humans. That is why humans from all educational backgrounds are able to solve these puzzles. Rules that depend on non-visual considerations (e.g. mathematics that\'s more advanced than small number counting) or inelegant mechanics (e.g. tie-breakers, complex conditional rules, special exceptions) are considered "unnatural."  \n\nThis doesn\'t mean puzzles have to be simple or easy. Common patterns that puzzles use to increase difficulty are:  \n\n- **Rule composition:** multiple simple, natural rules are applied in sequence, with one transformation setting up the next.\n- **Contextual application:** a simple if/else based on a visually clear feature.\n- **In-grid symbol definition:** certain shapes or patterns act as symbols whose meaning is established using the training examples and which control some aspect of the transformation. The meaning can even vary across training examples.\n- **Iterated steps:** a simple action is applied repeatedly (as in assembling pieces of a jigsaw), where each step constrains the next. Repetition is restricted to a small number.\n- **Approximate features**: Features that are easily detected by humans, but hard to detect using naive algorithms. Examples are slightly broken symmetry, slightly broken continuity, off-center symmetry axes, objects defined via 8-connectivity etc.\n- **Non-locality:** the solution requires applying changes to one region, with the changes depending on features in a different, distant region.\n\n# Puzzle to solve\n\n{{puzzle}}\n\n# Task aids\n\nYou may execute code in the Jupyter notebook environment.\n\nVariables available in the Jupyter notebook:\n\n- `puzzle`: A Python `dict` containing the training and test grids. The key "train" contains the training examples in a list. Similarly, the key "test" contains the test examples as a list. Each training/test example is a dict containing keys "input" and "output". The values are the grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.\n\nYour very first code execution should be `puzzle.keys()` to check data access. \n\nDon\'t print the `puzzle` dict, because it prints out the grids that are already present in this user message and consumes valuable token budget.\n\n# Your task\n\nPlease solve the puzzle. Solving the puzzle means constructing a function `solution(grid: list[list[int]])` that implements the transformation from input grid to output grid shown in the training examples. The solution must work on all training examples.\n\n```\nfor idx, example in enumerate(puzzle["train"]):\n    assert solution(example["input"]) == example["output"], f"Train example {idx}: mismatch"\n```\n\nThe solution must generalize to the test input(s). Solutions that cheat by hardcoding the training example outputs are not acceptable.\n\n# Output format (STRICT)\n\nYour response must contain a single Python code block fenced by ```python ... ```. Must include a `solution` function. Must be self-contained i.e. include all necessary imports and helper functions.'

# LOCAL adaptation: source advertises a broad dependency list and 120s IPython timeout.
# The portability audit recommends a smaller P0 stack. Probe the actual current Python
# environment and advertise only approved libraries that really import.
import subprocess as _piv_subprocess
P0_LIBRARY_PROBES = [
    ("numpy","numpy"), ("scipy","scipy"), ("scikit-image","skimage"),
    ("networkx","networkx"), ("Pillow","PIL"),
]
AVAILABLE_P0_LIBRARIES=[]
P0_LIBRARY_PROBE_RESULTS={}
for _label,_module in P0_LIBRARY_PROBES:
    try:
        _cp=_piv_subprocess.run(
            [sys.executable,"-c",f"import {_module}"],
            text=True,capture_output=True,timeout=8,
        )
        _ok=(_cp.returncode==0)
        P0_LIBRARY_PROBE_RESULTS[_label]={"ok":_ok,"stderr":(_cp.stderr or "")[-500:]}
        if _ok: AVAILABLE_P0_LIBRARIES.append(_label)
    except Exception as _e:
        P0_LIBRARY_PROBE_RESULTS[_label]={"ok":False,"stderr":f"{type(_e).__name__}: {_e}"}

_source_lib_sentence=(
    "The environment supports common third party libraries like numpy, scipy, shapely, networkx, "
    "scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver."
)
_local_lib_sentence=(
    "For this LOCAL P0 port, approved third party libraries verified in the current environment are: "
    + (", ".join(AVAILABLE_P0_LIBRARIES) if AVAILABLE_P0_LIBRARIES else "none beyond the Python standard library")
    + ". Do not assume optional source libraries such as OR-Tools, z3-solver, python-constraint, or shapely are available."
)
SYSTEM_PROMPT = SOURCE_SYSTEM_PROMPT.replace(_source_lib_sentence,_local_lib_sentence).replace(
    "time out after 120.0 seconds",
    f"time out after {TOOL_TIMEOUT_SEC:.1f} seconds",
)
SYSTEM_PROMPT += (
    f"\n\n### Local P0 compute budget\n"
    f"You may execute Python at most {MAX_TOOL_CALLS} times in this task. "
    "Use the calls deliberately. When the final Python call has been used, do not request another tool call; "
    "return the self-contained final `solution(grid)` code block on the next assistant turn."
)

def grid_lines(grid):
    return "\n".join(" ".join(str(int(x)) for x in row) for row in grid)

def puzzle_to_source_text(task):
    parts = ["## Training examples"]
    for i, ex in enumerate(task["train"]):
        parts += [f"### Training example {i}", "Input:", grid_lines(ex["input"]), "Output:", grid_lines(ex["output"])]
    parts.append("## Tests" if len(task["test"]) > 1 else "## Test")
    for i, ex in enumerate(task["test"]):
        if len(task["test"]) > 1:
            parts.append(f"### Test {i}")
        parts += ["Input:", grid_lines(ex["input"])]
    return "\n\n".join(parts)

V3_HELPERS_BULLET = (
    "- Exploration-only helpers (use them in the notebook; NEVER call them inside `solution`, they read `puzzle`): "
    "`arc_overview()` prints per-pair shapes, colors, changed-cell counts and component counts without printing grids; "
    "`verify_solution(fn=None)` runs `fn` (default `solution`) on every training pair and prints PASS / DIMS / wrong cells.\n"
    "- Grid helpers that are also available inside your final `solution` (do not redefine them): "
    "`components(grid, color=None, diag=False, background=None)` returns connected same-colour components as dicts "
    "with keys color, cells, bbox=(r0, c0, r1, c1), size; `bbox(grid, ignore=None)`; `crop(grid, r0, c0, r1, c1)`; "
    "`rot90(grid)` (clockwise), `flip_h(grid)`, `flip_v(grid)`, `transpose(grid)`; `color_counts(grid)`; "
    "`background_color(grid)` (the most common colour: a heuristic, pass `background=` explicitly when unsure); "
    "`grid_shape(grid)`; `diff_cells(a, b)` -> list of (r, c, a_value, b_value)."
)
V3_WORKFLOW = (
    "# Workflow (STRICT)\n\n"
    "- Never transcribe, restate or enumerate grid rows or cell values in your reasoning. The grids are already in "
    "`puzzle`; inspect them with Python instead.\n"
    "- Keep your thinking short. Your first Python execution should call `arc_overview()`.\n"
    "- By your second Python execution at the latest, define a complete `solution(grid)` for your current best "
    "hypothesis. Every time a Python execution defines or changes `solution` (or a helper it uses), the environment "
    "automatically runs it on all training examples in a fresh process and appends the result to the tool output.\n"
    "- Use that feedback to revise `solution` until every training example passes and it also runs cleanly on the "
    "test input(s). When both hold, the task is submitted automatically.\n"
    "- Define `solution` with `def`. Everything it depends on must be a top-level import, a function or class "
    "definition, or a constant; it must never depend on `puzzle` or on variables computed from `puzzle`.\n"
    "- Do not put reasoning into code comments."
)

def build_agent_user_prompt(task):
    # Keep the source prompt intact except for two necessary local substitutions:
    # 1) serialize the current puzzle; 2) clarify that test outputs are absent from the preloaded variable.
    prompt = SOURCE_AGENTIC_PROMPT_TEMPLATE.replace("{{puzzle}}", puzzle_to_source_text(task))
    prompt = prompt.replace(
        'Each training/test example is a dict containing keys "input" and "output". '
        'The values are the grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.',
        'Each training example is a dict containing keys "input" and "output"; each test example contains only "input". '
        'The values are grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.'
    )
    # ---- v3 prompt contract ----
    prompt = prompt.replace("Your very first code execution should be `puzzle.keys()` to check data access. \n\n", "")
    prompt = prompt.replace("\n\nDon't print the `puzzle` dict", "\n" + V3_HELPERS_BULLET + "\n\nDon't print the `puzzle` dict", 1)
    prompt = prompt.replace("# Your task\n\n", V3_WORKFLOW + "\n\n# Your task\n\n", 1)
    prompt = prompt.replace(
        "Must be self-contained i.e. include all necessary imports and helper functions.",
        "Must be self-contained i.e. include all necessary imports and helper functions; the preloaded helper "
        "functions listed above are always available and must not be redefined.")
    return prompt

# v3 prompt contract self-check: every substitution must have taken effect.
_v3_probe = build_agent_user_prompt({"train": [{"input": [[0]], "output": [[1]]}], "test": [{"input": [[0]]}]})
assert "puzzle.keys()" not in _v3_probe, "v3: puzzle.keys() instruction still present"
assert "# Workflow (STRICT)" in _v3_probe and "arc_overview()" in _v3_probe and "verify_solution" in _v3_probe
assert "must not be redefined" in _v3_probe
assert "NEVER call them inside `solution`" in _v3_probe and "Define `solution` with `def`" in _v3_probe
print("v3 prompt contract PASS")
SOURCE_AGENTIC_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_AGENTIC_PROMPT_TEMPLATE.encode("utf-8")).hexdigest()
SOURCE_SYSTEM_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_SYSTEM_PROMPT.encode("utf-8")).hexdigest()
with open(WORK_ROOT/"p0_library_probe.json","w") as _f:
    json.dump({"available":AVAILABLE_P0_LIBRARIES,"results":P0_LIBRARY_PROBE_RESULTS},_f,indent=2)
print("P0 approved libraries available:",AVAILABLE_P0_LIBRARIES)

CURRENT_ARM = ARMS[0]

def render_agent_chat(messages, reasoning_effort=None):
    # v1.8: the reasoning effort is the arm's variable; everything else in the render is frozen.
    effort = reasoning_effort or ARM_CONFIG[CURRENT_ARM]["reasoning_effort"]
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
        tools=[PYTHON_TOOL],
        enable_thinking=True,
        reasoning_effort=effort,
        preserve_thinking=True,
    )
    try:
        return tokenizer.apply_chat_template(messages, **kwargs), f"qwen_tools+{effort}+preserve_thinking"
    except TypeError as exc:
        # Preserve the causal mechanism if this repacked tokenizer predates the explicit preserve kwarg.
        kwargs.pop("preserve_thinking", None)
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        return rendered, f"qwen_tools+{effort}(template-preserve-default); fallback={type(exc).__name__}"

def render_terminal_chat(messages):
    """v2.4 structural terminal renderer: no tools are advertised and thinking is hard-disabled."""
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
        preserve_thinking=True,
    )
    try:
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        method = "qwen_terminal_no_tools+nonthinking+preserve_thinking"
    except TypeError as exc:
        kwargs.pop("preserve_thinking", None)
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        method = f"qwen_terminal_no_tools+nonthinking(template-preserve-default); fallback={type(exc).__name__}"
    return rendered, method

_terminal_probe_messages=[
    {"role":"system","content":"terminal contract probe"},
    {"role":"user","content":"return final answer only"},
]
_terminal_probe_rendered,_terminal_probe_method=render_terminal_chat(_terminal_probe_messages)
if "<function=python>" in _terminal_probe_rendered or "<tools>" in _terminal_probe_rendered:
    raise RuntimeError("Terminal render unexpectedly advertises the Python tool.")
if _terminal_probe_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Terminal render unexpectedly reopens a thinking block.")
QWEN_TERMINAL_RENDER_SHA256=hashlib.sha256(_terminal_probe_rendered.encode("utf-8")).hexdigest()

# Frozen task selection: same Proxy-48; SMOKE is a deterministic subset of it.
def choose_smoke8_from_proxy():
    """v1.6: stratified to mirror Proxy-48 (tier1 32/48, tier2 12/48, tier3 4/48 -> 5/2/1), deterministic in
    Proxy-48 order. The previous first-match selector took all four tier-3 tasks and biased the feasibility gate."""
    f = proxy_source_feat.set_index("task_id").loc[PROXY48_TASK_IDS].copy()
    chosen=[]
    def add(x):
        if x is not None and x not in chosen and len(chosen)<8: chosen.append(x)
    def first(mask):
        for t in f.index[mask.to_numpy()]:
            if t not in chosen: return t
        return None
    t1=(f["proxy_tier"]==1); t2=(f["proxy_tier"]==2); t3=(f["proxy_tier"]==3)
    add(first(t3))
    add(first(t2)); add(first(t2))
    add(first(t1 & (f["proxy_easy_static"]==1)))
    add(first(t1 & (f["proxy_hard_tier1"]==1)))
    add(first(t1 & (f["proxy_n_queries"]>1)))
    add(first(t1 & (f["proxy_dynamic"]==1)))
    while len(chosen)<8 and first(t1) is not None: add(first(t1))
    for x in PROXY48_TASK_IDS:
        if len(chosen)>=8: break
        add(x)
    return chosen[:8]

# EVO dual mode: in the scoring rerun every downstream cell addresses the HIDDEN test tasks through eval_challenges
# (no answers). The public evaluation data stays available as EVAL_CHALLENGES_PUBLIC.
EVAL_CHALLENGES_PUBLIC = eval_challenges
if EVO_RERUN:
    eval_challenges = test_challenges
    eval_solutions = {}
    ACTIVE_TASK_LIMIT = len(test_challenges)
if TASK_SET == "PROXY48": ACTIVE_TASK_IDS = list(PROXY48_TASK_IDS)
elif TASK_SET == "SMOKE8": ACTIVE_TASK_IDS = choose_smoke8_from_proxy()
elif TASK_SET == "PILOT2": ACTIVE_TASK_IDS = list(TERMINAL_PILOT_TASK_IDS)
elif TASK_SET == "EVAL120": ACTIVE_TASK_IDS = (list(EVAL_SUBSET_TASK_IDS) if EVAL_SUBSET_TASK_IDS else sorted(eval_challenges))
else: raise RuntimeError(f"Unknown TASK_SET={TASK_SET}")
ACTIVE_TASK_SET=set(ACTIVE_TASK_IDS)
PLANNED_TASK_IDS = tuple(ACTIVE_TASK_IDS)   # v3.1: immutable; reports always use this denominator
if len(ACTIVE_TASK_IDS)!=ACTIVE_TASK_LIMIT or len(ACTIVE_TASK_SET)!=ACTIVE_TASK_LIMIT:
    raise RuntimeError("Active task count/uniqueness mismatch")
if TASK_SET != "EVAL120" and not ACTIVE_TASK_SET.issubset(set(PROXY48_TASK_IDS)):
    raise RuntimeError("Active set escaped frozen Proxy-48")
if EVO_RERUN:
    print("RERUN:", len(ACTIVE_TASK_IDS), "hidden test tasks |", sum(len(eval_challenges[t]["test"]) for t in ACTIVE_TASK_IDS), "queries")
elif TASK_SET == "EVAL120":
    if not ACTIVE_TASK_SET.issubset(set(eval_solutions)):
        raise RuntimeError("EVAL120 requires a known answer for every active task")
    print("EVAL120:", len(ACTIVE_TASK_IDS), "tasks |",
          sum(len(eval_challenges[t]["test"]) for t in ACTIVE_TASK_IDS), "queries | answers known")
# v2.4B: fail BEFORE the 27B model is allocated if the adaptive pilot is not inside frozen SMOKE8.
# The original check lived in Cell 8, i.e. after model load, so a mismatch wasted a whole GPU session.
if RUN_PROFILE=="ADAPTIVE_SMOKE":
    _smoke8_preview=choose_smoke8_from_proxy()
    if not set(TERMINAL_PILOT_TASK_IDS).issubset(set(_smoke8_preview)):
        raise RuntimeError(f"Adaptive pilot {TERMINAL_PILOT_TASK_IDS} is not a subset of frozen SMOKE8 {_smoke8_preview}")
    print("Adaptive pilot-in-SMOKE8 precheck PASS:",TERMINAL_PILOT_TASK_IDS,"in",_smoke8_preview)

def build_prompt_records(arm):
    """Arm-specific initial prompts: identical user content; only the template-injected reasoning instruction differs."""
    recs=[]
    for task_id, task in eval_challenges.items():
        # v2.4B: only ACTIVE tasks are ever used. Rendering all 120 evaluation prompts on every
        # activate_arm() call (stage A, gate, stage B, score_arm => ~8x per run) was pure overhead.
        if task_id not in ACTIVE_TASK_SET:
            continue
        user_prompt=build_agent_user_prompt(task)
        messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_prompt}]
        rendered, method=render_agent_chat(messages, reasoning_effort=ARM_CONFIG[arm]["reasoning_effort"])
        recs.append({
            "arm":arm,"task_id":task_id, "user_prompt":user_prompt, "initial_messages":messages,
            "prompt":rendered, "prompt_sha256":hashlib.sha256(rendered.encode()).hexdigest(),
            "prompt_tokens":len(tokenizer.encode(rendered, add_special_tokens=False)),
            "n_train":len(task["train"]), "n_test":len(task["test"]), "render_method":method,
        })
    df=pd.DataFrame([{k:v for k,v in r.items() if k not in {"prompt","initial_messages","user_prompt"}} for r in recs])
    return recs, df
prompt_records, prompt_df = build_prompt_records(CURRENT_ARM)
prompt_df.to_csv(WORK_ROOT/"prompt_stats.csv", index=False)
PROMPT_RECORD_MAP={r["task_id"]:r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET}
# arm render sanity: every arm must render, and LOW/MEDIUM/XHIGH prompts must differ only in the injected instruction
_arm_probe={e:render_agent_chat([{"role":"system","content":"S"},{"role":"user","content":"U"}],reasoning_effort=e)[0] for e in sorted({ARM_CONFIG[a]["reasoning_effort"] for a in ARM_CONFIG})}
assert len({_arm_probe[a] for a in _arm_probe})==len(_arm_probe), "arm renders are not distinct"
print("Arm render probe OK:",{a:len(_arm_probe[a]) for a in _arm_probe})
# v1.9: template-level reasoning-effort contract: 'high' must be REJECTED by the mounted template.
def assert_qwen_reasoning_effort_contract():
    probe=[{"role":"system","content":"effort contract probe"},{"role":"user","content":"reply"}]
    r={e:render_agent_chat(probe,reasoning_effort=e)[0] for e in ("xhigh","medium","low")}
    assert "Reasoning effort is set to xhigh" in r["xhigh"] and "Reasoning effort is set to low" in r["low"]
    assert "Reasoning effort is set to" not in r["medium"]
    try:
        render_agent_chat(probe,reasoning_effort="high"); rejected=False
    except Exception: rejected=True
    if not rejected: raise RuntimeError("Mounted Qwen template unexpectedly accepts reasoning_effort='high'; template contract changed.")
    return {"supported":["xhigh","medium","low"],"high_rejected":True,"medium_injects_no_instruction":True}
QWEN_REASONING_EFFORT_CONTRACT=assert_qwen_reasoning_effort_contract()
print("Qwen reasoning-effort contract:",QWEN_REASONING_EFFORT_CONTRACT)

chat_template=getattr(tokenizer,"chat_template","") or ""
template_sha=hashlib.sha256(chat_template.encode()).hexdigest()
if "<tool_call>" not in chat_template or "tool_response" not in chat_template:
    raise RuntimeError("Mounted Qwen chat template does not expose the required tool-call/tool-response contract.")

# Validate the *actual mounted tokenizer* on the multi-turn structure PIV will use.
_transport_probe_messages = [
    {"role":"system","content":SYSTEM_PROMPT},
    {"role":"user","content":"transport probe"},
    {"role":"assistant","content":"","reasoning_content":"PIV_REASONING_SENTINEL",
     "tool_calls":[{"type":"function","function":{"name":"python","arguments":{"code":"x=1\nx+1"}}}]},
    {"role":"tool","content":"2"},
]
_transport_rendered, _transport_method = render_agent_chat(_transport_probe_messages)
for _required in ("PIV_REASONING_SENTINEL", "<tool_call>", "<function=python>", "<parameter=code>", "<tool_response>", "2"):
    if _required not in _transport_rendered:
        raise RuntimeError(f"Qwen multi-turn tool-history rendering contract failed: missing {_required!r}")
if not _transport_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Qwen tool-history render did not reopen thinking for the next assistant turn.")
QWEN_TOOL_HISTORY_TRANSPORT_SHA256 = hashlib.sha256(_transport_rendered.encode("utf-8")).hexdigest()

# v2.4 terminal-history probe: historical tool calls/results must remain renderable even though no tool is advertised
# for the new terminal assistant turn.
_terminal_history_messages=copy.deepcopy(_transport_probe_messages)+[{"role":"user","content":TERMINAL_FINALIZER_INSTRUCTION}]
_terminal_history_rendered,_terminal_history_method=render_terminal_chat(_terminal_history_messages)
if "PIV_REASONING_SENTINEL" not in _terminal_history_rendered or "<tool_response>" not in _terminal_history_rendered or "2" not in _terminal_history_rendered:
    raise RuntimeError("Terminal renderer lost historical reasoning/tool feedback.")
if _terminal_history_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Terminal renderer reopened thinking after historical tool use.")
QWEN_TERMINAL_HISTORY_RENDER_SHA256=hashlib.sha256(_terminal_history_rendered.encode("utf-8")).hexdigest()
# v2.4B diagnostic: with no tools advertised, does the terminal render still carry the historical
# ASSISTANT tool call, or only the tool response? If only the response survives, the finalizer sees
# results without the code that produced them. Recorded, not asserted, so the template stays the authority.
QWEN_TERMINAL_HISTORY_KEEPS_TOOL_CALLS=bool("<function=python>" in _terminal_history_rendered)
if not QWEN_TERMINAL_HISTORY_KEEPS_TOOL_CALLS:
    print("WARNING: terminal render drops historical assistant tool calls; the finalizer sees tool OUTPUT only.")
with open(WORK_ROOT/"chat_template_fingerprint.txt","w") as f:
    f.write(f"sha256={template_sha}\ntool_schema_sha256={TOOL_SCHEMA_SHA256}\n")

profile_task_table=eval_feat[eval_feat["task_id"].isin(ACTIVE_TASK_IDS)].copy()
profile_task_table=profile_task_table.merge(proxy_source_feat,on="task_id",how="left")
profile_task_table["prompt_tokens_piv_initial"]=profile_task_table["task_id"].map(prompt_df.set_index("task_id")["prompt_tokens"])
profile_task_table["proxy_order"]=profile_task_table["task_id"].map(proxy48_order)
profile_task_table=profile_task_table.sort_values("proxy_order").reset_index(drop=True)
profile_task_table.to_csv(WORK_ROOT/"active_task_selection.csv",index=False)

SOURCE_ALIGNMENT={
    "experiment":EXPERIMENT_ID,
    "piv_source_repo":PIV_SOURCE_REPO,
    "piv_source_commit":PIV_SOURCE_COMMIT,
    "piv_source_files":PIV_SOURCE_FILES,
    "external_sources":PIV_EXTERNAL_SOURCES,
    "proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,
    "source_file_sha256":SOURCE_FILE_SHA256,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
    "chat_template_sha256":template_sha,
    "qwen_tool_history_transport_sha256":QWEN_TOOL_HISTORY_TRANSPORT_SHA256,
    "qwen_terminal_render_sha256":QWEN_TERMINAL_RENDER_SHA256,
    "qwen_terminal_history_render_sha256":QWEN_TERMINAL_HISTORY_RENDER_SHA256,
    "qwen_terminal_history_keeps_tool_calls":QWEN_TERMINAL_HISTORY_KEEPS_TOOL_CALLS,
    "source_agentic_prompt_local_sha256":SOURCE_AGENTIC_PROMPT_LOCAL_SHA256,
    "source_system_prompt_local_sha256":SOURCE_SYSTEM_PROMPT_LOCAL_SHA256,
    "source_invariants":[
        "one persistent Python REPL per lineage",
        "puzzle preloaded without test outputs",
        "tool errors returned to model",
        "self-contained solution(grid) final artifact",
        "fresh independent all-train exact verification",
        "test grids produced by verified program execution",
    ],
    "local_adaptations":[
        "direct vLLM + Qwen native tool chat template; no OpenAI Responses server",
        "lightweight subprocess worker instead of Docker/IPyBox",
        "stricter grid/domain/time/file-system safety",
        "bounded 6-execution P0 tool budget and context guard",
        "source-style max_retries=2 reliability retries share the same total token/tool/turn budgets",
        "per-turn ceiling = remaining total P0 budget (12288) bounded by context headroom; total generated-token cap 12288",
        "v2.4 terminal pilot protects 4096 tokens + one turn slot for a no-tools, non-thinking structured terminal commit",
    ],
}
with open(WORK_ROOT/"SOURCE_ALIGNMENT.json","w") as f: json.dump(SOURCE_ALIGNMENT,f,indent=2)
print("Chat template/tool contract PASS | SHA:",template_sha)
print("ACTIVE TASKS:",len(ACTIVE_TASK_IDS),TASK_SET)
display(prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].describe().to_frame("initial_prompt_tokens").round(1))


In [ ]:
# =====================================================================================
# CELL 5 — PIV CONTEXT / COMPUTE SAFETY (preserve history; do not add P3 compaction in P0)
# =====================================================================================
active_prompt_tokens=prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].to_numpy()
max_initial_prompt=int(active_prompt_tokens.max())
CONTEXT_SAFETY_TOKENS=512
MAX_RENDERED_INPUT_TOKENS=MAX_MODEL_LEN-PER_TURN_MAX_TOKENS-CONTEXT_SAFETY_TOKENS
EVAL120_SKIPPED_PROMPT_TOO_LONG = {}
if max_initial_prompt > MAX_RENDERED_INPUT_TOKENS:
    if not EVAL120_MODE:
        raise RuntimeError(f"Initial PIV prompt exceeds context: {max_initial_prompt} > {MAX_RENDERED_INPUT_TOKENS}")
    # Evaluation grids are about twice the size of the training/placeholder ones; a task that cannot fit is
    # reported and skipped instead of killing a two-hour measurement run.
    _pt={r["task_id"]:int(r["prompt_tokens"]) for r in prompt_records}
    _over=[t for t in ACTIVE_TASK_IDS if _pt.get(t,0) > MAX_RENDERED_INPUT_TOKENS]
    for _t in _over:
        EVAL120_SKIPPED_PROMPT_TOO_LONG[_t]={"prompt_tokens":_pt.get(_t),"limit":int(MAX_RENDERED_INPUT_TOKENS)}
    ACTIVE_TASK_IDS=[t for t in ACTIVE_TASK_IDS if t not in set(_over)]
    ACTIVE_TASK_SET=set(ACTIVE_TASK_IDS)
    PROMPT_RECORD_MAP={k:v for k,v in PROMPT_RECORD_MAP.items() if k in ACTIVE_TASK_SET}
    active_prompt_tokens=prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].to_numpy()
    max_initial_prompt=int(active_prompt_tokens.max())
    print("EVAL120: dropped", len(_over), "task(s) whose prompt exceeds", MAX_RENDERED_INPUT_TOKENS, "tokens:", _over[:10])
MAX_TOOL_OUTPUT_CHARS=4000
MAX_TOOL_RAW_ARTIFACT_CHARS=100000
MAX_CODE_CHARS=30000
print("PIV CONTEXT CONFIG")
print("  max_model_len           :",MAX_MODEL_LEN)
print("  max initial prompt      :",max_initial_prompt)
print("  per-turn max generation:",PER_TURN_MAX_TOKENS)
print("  total generated cap     :",TOTAL_GENERATED_TOKEN_CAP)
print("  max rendered input      :",MAX_RENDERED_INPUT_TOKENS)
print("  max tool output to model:",MAX_TOOL_OUTPUT_CHARS,"chars")
print("  history policy          : PRESERVE (P0); no P3 compact-state intervention")
print("  arms                    :",ARMS)
for _a in ARMS: print(f"    {_a}: {ARM_CONFIG[_a]}")
print("  action cap after forced close:",ACTION_MAX_TOKENS)


In [ ]:
# =====================================================================================
# CELL 6 — QWEN TOOL XML PARSER + PERSISTENT LOCAL REPL + FRESH EXACT VERIFIER + SELF-TESTS
# =====================================================================================
import ast as py_ast
import re, subprocess, selectors, tempfile, uuid, textwrap

DANGEROUS_IMPORT_ROOTS={"os","subprocess","pathlib","shutil","socket","requests","urllib","http","ftplib","paramiko","builtins","importlib","inspect"}
DANGEROUS_CALL_NAMES={"open","exec","eval","compile","__import__","input","breakpoint","globals","locals","vars","getattr","setattr","delattr","exit","quit"}
DANGEROUS_DUNDER_ATTRS={"__subclasses__","__globals__","__builtins__","__code__","__closure__","__getattribute__"}
ALLOWED_SYS_ATTRS={"setrecursionlimit","getrecursionlimit","maxsize"}

def canonical_grid(grid):
    if isinstance(grid,np.ndarray): grid=grid.tolist()
    if not is_rectangular_grid(grid): return None
    return tuple(tuple(int(x) for x in row) for row in grid)
def grids_equal(a,b):
    ca,cb=canonical_grid(a),canonical_grid(b)
    return ca is not None and cb is not None and ca==cb

def code_defines_solution(code):
    try: tree=py_ast.parse(code)
    except Exception: return False
    return any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in tree.body)

def extract_last_solution_code(text):
    blocks=re.findall(r"```(?:python|py)?\s*\n?(.*?)```", text or "", flags=re.I|re.S)
    for code in reversed(blocks):
        code=code.strip()
        if code_defines_solution(code): return code
    return None

def split_qwen_reasoning(text, action_pos=None):
    text=text or ""
    prefix=text if action_pos is None else text[:action_pos]
    if "</think>" in prefix:
        a,b=prefix.rsplit("</think>",1)
        return a.replace("<think>","",1).strip(), b.strip(), "think_close"
    # Generation prompt already opens <think>; without a close, pre-action text is reasoning.
    return prefix.replace("<think>","",1).strip(), "", "implicit_open_think"

def _action_region(text):
    """Return only the committed action region after the final </think>.

    Qwen's generation prompt opens <think>. Without a close, the entire response remains scratch reasoning,
    even if it contains a fenced solution() draft or tool-like XML.
    """
    text=text or ""
    if "</think>" in text:
        pos=text.rfind("</think>")+len("</think>")
        return text[pos:],pos
    return "",len(text)

def parse_tool_calls(text):
    text=text or ""
    region,offset=_action_region(text)
    calls=[]
    pattern=r"<tool_call>\s*<function=([^>\s]+)>\s*(.*?)</function>\s*</tool_call>"
    for m in re.finditer(pattern,region,re.S):
        name=m.group(1).strip(); body=m.group(2); args={}
        for p in re.finditer(r"<parameter=([^>\s]+)>[ \t]*\r?\n?(.*?)[ \t]*\r?\n?[ \t]*</parameter>",body,re.S):
            val=p.group(2)
            # v1.6: unwrap a fenced ```python payload; the fence itself is not Python.
            _fm=re.fullmatch(r"```(?:python|py)?[ \t]*\n?(.*?)\n?```",val.strip(),flags=re.S|re.I)
            if _fm: val=_fm.group(1)
            # v2.4B: the old `\s*(.*?)\s*` stripped the FIRST line's indentation only, so a uniformly
            # indented XML payload became "x=1\n    print(x)" -> IndentationError inside the REPL.
            # Strip only the framing newline, then remove any uniform leading indent.
            val=textwrap.dedent(val)
            args[p.group(1).strip()]=val
        calls.append({"name":name,"arguments":args,"start":offset+m.start(),"end":offset+m.end(),"raw":m.group(0)})
    return calls

def parse_agent_turn(text):
    text=text or ""
    tcs=parse_tool_calls(text)
    if tcs:
        reasoning,content,rmode=split_qwen_reasoning(text,tcs[0]["start"])
        suffix=text[tcs[-1]["end"]:].strip()
        suffix=re.sub(r"(?:<\|im_end\|>|<\|endoftext\|>)\s*$","",suffix).strip()
        return {"kind":"tool","tool_calls":tcs,"tool_name":tcs[0]["name"],"tool_args":tcs[0]["arguments"],
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":suffix,"final_code":None}
    action_text,_=_action_region(text)
    if "<tool_call>" in action_text:   # v1.6: prose merely mentioning <function= is not a malformed call
        reasoning,content,rmode=split_qwen_reasoning(text,None)
        return {"kind":"malformed_tool","tool_calls":[],"tool_name":None,"tool_args":{},
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":"","final_code":None}
    code=extract_last_solution_code(action_text)
    reasoning,content,rmode=split_qwen_reasoning(text,None)
    return {"kind":"final" if code else "text","tool_calls":[],"tool_name":None,"tool_args":{},
            "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
            "suffix_after_tool":"","final_code":code}

def generation_finish_reason_ok(finish_reason):
    # Mirrors the source client's rule that a truncated response is not a valid turn result.
    return str(finish_reason).lower() == "stop"

# v1.6: tool output is inserted verbatim into the next prompt. Neutralize chat-control strings the model's
# own code could print so they cannot open/close turns or tool blocks.
_CHAT_CONTROL_PATTERNS=[("<|","< |"),("<tool_response>","< tool_response>"),("</tool_response>","< /tool_response>"),
                        ("<tool_call>","< tool_call>"),("</tool_call>","< /tool_call>"),
                        ("<think>","< think>"),("</think>","< /think>")]
def sanitize_tool_output(text):
    text=str(text or "")
    for a,b in _CHAT_CONTROL_PATTERNS: text=text.replace(a,b)
    return text

def salvage_final_code_on_length(text, parsed):
    """v1.6 LOCAL: a length-capped final turn whose action region (after </think>) already contains a CLOSED,
    ast-parseable ```python block defining solution() is demonstrably complete; retrying it would only burn
    budget. Drafts inside an unclosed <think> are never salvaged."""
    if parsed.get("kind")!="final" or parsed.get("reasoning_mode")!="think_close": return None
    return parsed.get("final_code")

def guard_python_code(code, *, final=False):
    if not isinstance(code,str) or not code.strip(): return False,"empty_code"
    if len(code)>MAX_CODE_CHARS: return False,"code_too_long"
    try: tree=py_ast.parse(code)
    except SyntaxError as e: return False,f"syntax_error:{e.msg}"

    # `sys.setrecursionlimit(...)` is common in DFS/flood-fill ARC solutions.
    # Permit only a narrow sys surface; block module-table/process introspection.
    sys_aliases=set()
    for _n in py_ast.walk(tree):
        if isinstance(_n,py_ast.Import):
            for _a in _n.names:
                if _a.name=="sys": sys_aliases.add(_a.asname or "sys")
        elif isinstance(_n,py_ast.ImportFrom) and (_n.module or "")=="sys":
            imported={_a.name for _a in _n.names}
            if not imported.issubset(ALLOWED_SYS_ATTRS):
                return False,f"blocked_sys_importfrom:{sorted(imported)}"

    for node in py_ast.walk(tree):
        if isinstance(node,py_ast.Name) and node.id=="__builtins__":
            return False,"blocked_builtins_escape"
        if isinstance(node,py_ast.Attribute) and node.attr in DANGEROUS_DUNDER_ATTRS:
            return False,f"blocked_dunder_attribute:{node.attr}"
        if isinstance(node,py_ast.Attribute) and isinstance(node.value,py_ast.Name) and node.value.id in sys_aliases:
            if node.attr not in ALLOWED_SYS_ATTRS:
                return False,f"blocked_sys_attribute:{node.attr}"
        if isinstance(node,py_ast.Constant) and isinstance(node.value,str) and "/kaggle" in node.value.lower():
            return False,"blocked_host_path_literal"
        if isinstance(node,(py_ast.Import,py_ast.ImportFrom)):
            names=[a.name.split('.')[0] for a in node.names] if isinstance(node,py_ast.Import) else [(node.module or '').split('.')[0]]
            if any(x in DANGEROUS_IMPORT_ROOTS for x in names): return False,f"blocked_import:{names}"
        if isinstance(node,py_ast.Call):
            if isinstance(node.func,py_ast.Name) and node.func.id in DANGEROUS_CALL_NAMES:
                return False,f"blocked_call:{node.func.id}"
            if isinstance(node.func,py_ast.Attribute) and node.func.attr in DANGEROUS_DUNDER_ATTRS:
                return False,f"blocked_dunder_call:{node.func.attr}"
        if isinstance(node,py_ast.Raise):
            exc=node.exc
            if isinstance(exc,py_ast.Name) and exc.id=="SystemExit":
                return False,"blocked_system_exit"
            if isinstance(exc,py_ast.Call) and isinstance(exc.func,py_ast.Name) and exc.func.id=="SystemExit":
                return False,"blocked_system_exit"
        if isinstance(node,(py_ast.Assign,py_ast.AnnAssign,py_ast.AugAssign,py_ast.NamedExpr)):
            targets=[]
            if isinstance(node,py_ast.Assign): targets=node.targets
            else: targets=[node.target]
            def rooted_at_puzzle(t):
                while isinstance(t,(py_ast.Subscript,py_ast.Attribute)):
                    t=t.value
                return isinstance(t,py_ast.Name) and t.id=="puzzle"
            for t in targets:
                if rooted_at_puzzle(t): return False,"protected_puzzle_assignment"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Attribute):
            base=node.func.value
            while isinstance(base,(py_ast.Subscript,py_ast.Attribute)): base=base.value
            if isinstance(base,py_ast.Name) and base.id=="puzzle" and node.func.attr in {"update","clear","pop","popitem","setdefault","append","extend","insert","remove"}:
                return False,"protected_puzzle_mutation"
        if isinstance(node,py_ast.Expr) and isinstance(node.value,py_ast.Name) and node.value.id=="puzzle":
            return False,"blocked_whole_puzzle_display"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Name) and node.func.id=="print":
            if any(isinstance(a,py_ast.Name) and a.id=="puzzle" for a in node.args):
                return False,"blocked_whole_puzzle_print"
    if final and not code_defines_solution(code): return False,"missing_solution_function"
    return True,"ok"

def compact_tool_output(text,limit=MAX_TOOL_OUTPUT_CHARS):
    text=str(text or "")
    if len(text)<=limit: return text
    keep=max(1,(limit-120)//2)
    return text[:keep]+f"\n...[tool output truncated; original chars={len(text)}]...\n"+text[-keep:]

# =====================================================================================
# v3 PRELOADED ARC HELPER LIBRARY (REPL preload + prepended to every verified program)
# =====================================================================================
PIV_LIBRARY_SRC = r"""# ===== PIV preloaded ARC helpers (available in every program; do not redefine) =====
from collections import Counter as _PIV_Counter, deque as _PIV_deque


def grid_shape(g):
    return (len(g), len(g[0]) if g else 0)


def color_counts(g):
    return dict(_PIV_Counter(v for row in g for v in row))


def background_color(g):
    c = _PIV_Counter(v for row in g for v in row)
    return c.most_common(1)[0][0] if c else 0


def components(g, color=None, diag=False, background=None):
    # Connected same-colour components. color=None: every non-background colour; diag=True: 8-connectivity.
    # Returns dicts {color, cells, bbox=(r0,c0,r1,c1), size}, in reading order of their first cell.
    H, W = len(g), len(g[0])
    bg = background_color(g) if background is None else background
    seen = [[False] * W for _ in range(H)]
    nbrs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if diag:
        nbrs += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    out = []
    for r in range(H):
        for c in range(W):
            if seen[r][c]:
                continue
            v = g[r][c]
            if (color is None and v == bg) or (color is not None and v != color):
                continue
            q = _PIV_deque([(r, c)])
            seen[r][c] = True
            cells = []
            while q:
                y, x = q.popleft()
                cells.append((y, x))
                for dy, dx in nbrs:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny][nx] and g[ny][nx] == v:
                        seen[ny][nx] = True
                        q.append((ny, nx))
            ys = [p[0] for p in cells]
            xs = [p[1] for p in cells]
            out.append({"color": v, "cells": cells, "bbox": (min(ys), min(xs), max(ys), max(xs)), "size": len(cells)})
    return out


def bbox(g, ignore=None):
    # Bounding box (r0, c0, r1, c1) of cells different from `ignore` (default: background). None if empty.
    bg = background_color(g) if ignore is None else ignore
    pts = [(r, c) for r, row in enumerate(g) for c, v in enumerate(row) if v != bg]
    if not pts:
        return None
    ys = [p[0] for p in pts]
    xs = [p[1] for p in pts]
    return (min(ys), min(xs), max(ys), max(xs))


def crop(g, r0, c0, r1, c1):
    return [list(row[c0:c1 + 1]) for row in g[r0:r1 + 1]]


def rot90(g):
    return [list(row) for row in zip(*g[::-1])]


def flip_h(g):
    return [list(row[::-1]) for row in g]


def flip_v(g):
    return [list(row) for row in g[::-1]]


def transpose(g):
    return [list(row) for row in zip(*g)]


def diff_cells(a, b):
    # Cells where same-shape grids differ: list of (r, c, a_value, b_value).
    return [(r, c, a[r][c], b[r][c]) for r in range(len(a)) for c in range(len(a[0])) if a[r][c] != b[r][c]]


def arc_overview():
    # Compact per-pair structure of the preloaded puzzle, without transcribing any grid.
    lines = []
    for i, ex in enumerate(puzzle["train"]):
        a, b = ex["input"], ex["output"]
        sa, sb = grid_shape(a), grid_shape(b)
        line = "train %d: in %dx%d -> out %dx%d" % (i, sa[0], sa[1], sb[0], sb[1])
        line += " | in colors %s | out colors %s" % (dict(sorted(color_counts(a).items())), dict(sorted(color_counts(b).items())))
        if sa == sb:
            d = diff_cells(a, b)
            line += " | changed cells %d/%d" % (len(d), sa[0] * sa[1])
            if d:
                line += " | top in->out changes %s" % (_PIV_Counter((x[2], x[3]) for x in d).most_common(4),)
        line += " | components in %d out %d" % (len(components(a)), len(components(b)))
        lines.append(line)
    for j, t in enumerate(puzzle["test"]):
        s = grid_shape(t["input"])
        lines.append("test %d: in %dx%d | colors %s" % (j, s[0], s[1], dict(sorted(color_counts(t["input"]).items()))))
    print("\n".join(lines))


def verify_solution(fn=None, show=4):
    # Run fn (default: solution) on every training pair; print PASS / dims / wrong cells. Returns #exact pairs.
    if fn is None:
        try:
            fn = solution
        except NameError:
            print("no solution() is defined yet")
            return 0
    ok = 0
    for i, ex in enumerate(puzzle["train"]):
        try:
            p = fn([list(row) for row in ex["input"]])
        except Exception as e:
            print("pair %d: EXCEPTION %s: %s" % (i, type(e).__name__, e))
            continue
        o = ex["output"]
        if p == o:
            ok += 1
            print("pair %d: PASS" % i)
            continue
        if not (isinstance(p, list) and p and isinstance(p[0], list)):
            print("pair %d: invalid output (%s)" % (i, type(p).__name__))
            continue
        if grid_shape(p) != grid_shape(o):
            print("pair %d: DIMS got %s expected %s" % (i, grid_shape(p), grid_shape(o)))
            continue
        d = diff_cells(p, o)
        print("pair %d: %d/%d cells wrong; first (r, c, got, expected): %s" % (i, len(d), len(o) * len(o[0]), d[:show]))
    print("=> %d/%d training pairs exact" % (ok, len(puzzle["train"])))
    return ok
# ===== end PIV helpers =====
"""
PIV_LIBRARY_MARKER = "# ===== PIV preloaded ARC helpers"
PIV_LIBRARY_END = "# ===== end PIV helpers ====="

def with_library(code):
    code = str(code or "")
    return code if PIV_LIBRARY_MARKER in code else PIV_LIBRARY_SRC + "\n\n" + code

def strip_library(code):
    code = str(code or "")
    if PIV_LIBRARY_MARKER in code and PIV_LIBRARY_END in code:
        return code.split(PIV_LIBRARY_END, 1)[1].lstrip("\n")
    return code

import builtins as _v3_builtins
import symtable as _v3_symtable
_V3_BUILTIN_NAMES = set(dir(_v3_builtins))
# Calls that only make sense during exploration (they print, read `puzzle`, or touch the session) never reach a program.
_V3_EXPLORATION_ONLY = {"print", "arc_overview", "verify_solution", "input", "open", "exec", "eval", "globals", "locals",
                        "vars", "breakpoint", "exit", "quit", "help", "display"}
_V3_MUTATING_METHODS = {"append", "extend", "insert", "pop", "remove", "clear", "update", "add", "discard",
                        "setdefault", "sort", "reverse", "popitem", "fill", "put", "__setitem__", "__delitem__"}

def _v3_library_names():
    out = set()
    for n in py_ast.parse(PIV_LIBRARY_SRC).body:
        if isinstance(n, (py_ast.FunctionDef, py_ast.ClassDef)):
            out.add(n.name)
        elif isinstance(n, (py_ast.Import, py_ast.ImportFrom)):
            out |= {(a.asname or a.name).split(".")[0] for a in n.names}
    return out
_V3_LIBRARY_NAMES = _v3_library_names()

def _v3_base_name(n):
    while isinstance(n, (py_ast.Attribute, py_ast.Subscript, py_ast.Starred)):
        n = n.value
    return n.id if isinstance(n, py_ast.Name) else None

def _v3_module_level_nodes(node):
    """AST nodes of one top-level statement that run when the statement runs (function and lambda bodies excluded;
    their decorators and default values included)."""
    def _head(f):
        a = f.args
        return list(getattr(f, "decorator_list", [])) + [d for d in list(a.defaults) + list(a.kw_defaults) if d is not None]
    stack = _head(node) if isinstance(node, (py_ast.FunctionDef, py_ast.AsyncFunctionDef)) else [node]
    while stack:
        n = stack.pop()
        yield n
        for ch in py_ast.iter_child_nodes(n):
            if isinstance(ch, (py_ast.FunctionDef, py_ast.AsyncFunctionDef, py_ast.Lambda)):
                stack.extend(_head(ch))
            else:
                stack.append(ch)

def _v3_scope(node):
    """Scope facts of ONE top-level statement from the compiler's own symbol table (symtable), plus what symtable does
    not report (AugAssign read, := inside a comprehension, in-place mutation of an existing object).
    Returns (reads_now, reads_later, writes, potential_writes):
      reads_now   names read when the statement runs (for a def: decorators/defaults/annotations only)
      reads_later names read when a function defined here is later called (function/lambda bodies)
      writes      module names the statement (re)binds, deletes or mutates when it runs
      potential   module names a function defined here may rebind when called (`global x; x = ...`)."""
    top = _v3_symtable.symtable(py_ast.unparse(node), "<piv-candidate>", "exec")
    now, later, writes, potential = set(), set(), set(), set()
    for sym in top.get_symbols():
        if sym.is_referenced():
            now.add(sym.get_name())
        if sym.is_assigned() or sym.is_imported() or sym.is_namespace():
            writes.add(sym.get_name())
    is_def = isinstance(node, (py_ast.FunctionDef, py_ast.AsyncFunctionDef))
    stack = list(top.get_children())
    while stack:
        t = stack.pop()
        stack.extend(t.get_children())
        for sym in t.get_symbols():
            if sym.is_global() and sym.is_referenced():
                (later if is_def else now).add(sym.get_name())   # non-def statements: be conservative, all reads now
            if sym.is_declared_global() and sym.is_assigned():
                potential.add(sym.get_name())
    for n in _v3_module_level_nodes(node):
        if isinstance(n, py_ast.AugAssign) and isinstance(n.target, py_ast.Name):
            now.add(n.target.id)
        elif isinstance(n, py_ast.NamedExpr) and isinstance(n.target, py_ast.Name):
            writes.add(n.target.id)
        elif isinstance(n, (py_ast.Subscript, py_ast.Attribute)) and isinstance(n.ctx, (py_ast.Store, py_ast.Del)):
            b = _v3_base_name(n.value)
            if b:
                writes.add(b)
        elif isinstance(n, py_ast.Call) and isinstance(n.func, py_ast.Attribute) and n.func.attr in _V3_MUTATING_METHODS:
            b = _v3_base_name(n.func.value)
            if b:
                writes.add(b)
    return now, later, writes, potential

def _v3_called_names(nodes):
    return {n.func.id for n in nodes if isinstance(n, py_ast.Call) and isinstance(n.func, py_ast.Name)}

def build_candidate_module(cells):
    return build_candidate_module_ex(cells)[0]

def build_candidate_module_ex(cells):
    """Standalone program from the REPL history, in execution order. Kept: imports; function/class definitions whose
    definition-time reads are available; module-level assignments and in-place method calls on kept objects whose
    every read is a builtin, a library helper or something kept, that read nothing derived from `puzzle` and call no
    exploration-only helper. Everything else is dropped. Any name whose current REPL value comes from a dropped
    statement (rebinding, :=, del, loop/with/except target, mutation, `global` write) is tainted; taint propagates
    through kept definitions and assignments to a fixpoint (scope facts from symtable, so nested functions,
    comprehensions and shadowing follow Python's own rules). A tainted `solution` yields no module, and the caller
    tells the model why. As a second line of defence the scheduler compares the module's outputs with the live REPL's
    solution() on every training input (`_v3_fidelity_check`). Returns (module_or_None, info)."""
    known = set(_V3_BUILTIN_NAMES) | set(_V3_LIBRARY_NAMES)
    BASE_TAINT = {"puzzle", "arc_overview", "verify_solution"}
    tainted = set(BASE_TAINT)    # names whose use needs `puzzle` (incrementally maintained for admission)
    dropped = set()              # names whose latest REPL value comes from a dropped statement
    keep, deps = [], []
    global_writers = {}          # kept function name -> module names it rebinds via `global` when called
    for code in list(cells or []):
        try:
            tree = py_ast.parse(code)
        except Exception:
            continue
        for node in tree.body:
            try:
                now, later, writes, potential = _v3_scope(node)
            except Exception:
                # unanalysable: drop it and taint every name it mentions
                _eff = {n.id for n in py_ast.walk(node) if isinstance(n, py_ast.Name)}
                dropped |= _eff; known -= _eff
                continue
            calls = _v3_called_names(list(_v3_module_level_nodes(node)))
            admissible = now <= known and not (now & tainted) and not (calls & _V3_EXPLORATION_ONLY)
            if isinstance(node, (py_ast.Import, py_ast.ImportFrom)):
                ok = True
            elif isinstance(node, (py_ast.FunctionDef, py_ast.AsyncFunctionDef, py_ast.ClassDef)):
                ok = admissible
            elif isinstance(node, (py_ast.Assign, py_ast.AugAssign)) or (isinstance(node, py_ast.AnnAssign) and node.value is not None):
                ok = admissible
            elif isinstance(node, py_ast.Expr) and isinstance(node.value, py_ast.Call) and isinstance(node.value.func, py_ast.Attribute):
                b = _v3_base_name(node.value.func.value)
                ok = admissible and b is not None and b in known and b not in _V3_BUILTIN_NAMES
            else:
                ok = False
            if ok:
                keep.append(node)
                known |= writes; dropped -= writes
                if (now | later) & tainted:
                    tainted |= writes
                else:
                    tainted -= writes
                deps.append((set(writes), now | later))
                for _w in writes:
                    global_writers.pop(_w, None)
                if potential and isinstance(node, (py_ast.FunctionDef, py_ast.AsyncFunctionDef)):
                    global_writers[node.name] = set(potential)
            else:
                _eff = writes | potential
                for _f in (now & set(global_writers)):
                    _eff |= global_writers[_f]   # a dropped call of a kept function that writes globals
                dropped |= _eff; known -= _eff
    T = set(BASE_TAINT) | dropped
    changed = True
    while changed:
        changed = False
        for _w, _r in deps:
            if (_r & T) and not (_w <= T):
                T |= _w; changed = True
    body = py_ast.unparse(py_ast.Module(body=keep, type_ignores=[])) if keep else ""
    info = {"defines_solution": bool(body) and code_defines_solution(body), "tainted_solution": False, "tainted_deps": []}
    if info["defines_solution"] and "solution" in T:
        sol_reads = set()
        for _w, _r in deps:
            if "solution" in _w:
                sol_reads |= _r
        info.update({"tainted_solution": True,
                     "tainted_deps": sorted(((T - {"solution"}) & sol_reads) - _V3_BUILTIN_NAMES)[:8]})
        return None, info
    if not info["defines_solution"]:
        return None, info
    return with_library(body), info

_V3_FIDELITY_PROBE = r"""
def _piv_fid_run():
    import json as _j, hashlib as _h, copy as _c, numbers as _n
    def _clean(g):
        try:
            import numpy as _np
            if isinstance(g, _np.ndarray): g = g.tolist()
        except Exception:
            pass
        rows = [list(r.tolist() if hasattr(r, "tolist") else r) for r in list(g)]
        if not (1 <= len(rows) <= 30): raise ValueError("rows")
        w = {len(r) for r in rows}
        if len(w) != 1 or min(w) < 1 or max(w) > 30: raise ValueError("shape")
        out = []
        for r in rows:
            rr = []
            for x in r:
                if isinstance(x, bool) or not isinstance(x, _n.Integral): raise ValueError("cell")
                x = int(x)
                if x < 0 or x > 9: raise ValueError("domain")
                rr.append(x)
            out.append(rr)
        return out
    sig = []
    for _ex in puzzle["train"]:
        try:
            sig.append(_h.sha256(_j.dumps(_clean(solution(_c.deepcopy(_ex["input"])))).encode()).hexdigest()[:16])
        except BaseException:
            sig.append("ERR")
    return sig
print("PIV_FIDELITY:" + __import__("json").dumps(_piv_fid_run()))
del _piv_fid_run
"""

def _v3_fresh_signature(verifier, task):
    sig = []
    by_index = {int(x.get("index", i)): x for i, x in enumerate(list((verifier or {}).get("train") or []))}
    for i, ex in enumerate(task.get("train", [])):
        e = by_index.get(i)
        if e is None:
            sig.append(None)
        elif e.get("exact"):
            sig.append(hashlib.sha256(json.dumps(ex["output"]).encode()).hexdigest()[:16])
        elif e.get("pred") is not None:
            sig.append(hashlib.sha256(json.dumps(e["pred"]).encode()).hexdigest()[:16])
        else:
            sig.append("ERR")
    return sig

def _v3_fidelity_check(worker, verifier, task):
    """Does the fresh module behave like solution() in the live REPL on every training input?
    'match' | 'mismatch' | 'unknown'. The probe is not recorded as a REPL cell; on timeout the worker restarts and
    replays its successful cells, as for any tool call."""
    try:
        r = worker._request({"op": "exec", "code": _V3_FIDELITY_PROBE}, worker.timeout_sec)
    except Exception:
        try:
            worker.restart_count += 1; worker.start()
        except Exception:
            pass
        return "unknown", None
    txt = str((r or {}).get("text") or "")
    if not r.get("ok") or "PIV_FIDELITY:" not in txt:
        return "unknown", None
    try:
        repl = json.loads(txt.rsplit("PIV_FIDELITY:", 1)[1].strip().splitlines()[0])
    except Exception:
        return "unknown", None
    fresh = _v3_fresh_signature(verifier, task)
    if any(x is None for x in fresh):
        # no per-pair results (module failed to load, or verifier timeout): same behaviour only if the REPL fails too
        if (verifier or {}).get("verifier_status") == "train_timeout":
            return "unknown", None
        return ("match" if all(x == "ERR" for x in repl) else "mismatch"), [i for i, x in enumerate(repl) if x != "ERR"]
    bad = [i for i, (a, b) in enumerate(zip(repl, fresh)) if a != b]
    return ("mismatch" if bad or len(repl) != len(fresh) else "match"), bad

def _v3_train_quality(verifier, task):
    """Order candidates by how close they are, not only by exact pairs (review v3, defect 4):
    (train-exact and test runs, train-exact, exact pairs, right-shaped pairs, test runs, pairs without exception,
     -fraction of wrong cells over all expected cells; wrong-shaped/exception pairs count every cell wrong)."""
    v = verifier or {}
    by_index = {int(x.get("index", i)): x for i, x in enumerate(list(v.get("train") or []))}
    exact = shaped = no_exc = wrong = total = 0
    for i, ex in enumerate(task.get("train", [])):
        d = _v25_pair_diag(by_index.get(i, {"index": i, "exact": False, "error": "missing verifier pair result"}), ex)
        dims = _v25_grid_dims(ex.get("output")) or (1, 1)
        cells = int(dims[0] * dims[1]); total += cells
        if d["kind"] == "exact":
            exact += 1; shaped += 1; no_exc += 1
        elif d["kind"] == "cells_wrong":
            shaped += 1; no_exc += 1; wrong += int(d.get("wrong_cells") or 0)
        else:
            wrong += cells
            if d["kind"] != "exception":
                no_exc += 1
    all_exact = bool(v.get("all_train_exact")); test_ok = bool(v.get("test_execution_ok"))
    return (all_exact and test_ok, all_exact, exact, shaped, test_ok, no_exc, -round(wrong / max(total, 1), 6))

def _v3_report_text(verifier, task):
    """Verifier report shown to the model; says explicitly when training passes but the test input fails."""
    try:
        txt = _v25_feedback_text(verifier, task)["text"]
    except Exception as _exc:
        txt = "verifier report unavailable: %s" % type(_exc).__name__
    v = verifier or {}
    if v.get("all_train_exact") and not v.get("test_execution_ok"):
        err = _v25_exception_last_line(v.get("test_error") or v.get("error") or "no valid grid returned")
        txt += ("\nTEST INPUT FAILURE: solution() passes every training example but did not return a valid grid for the "
                "test input(s): %s. It has NOT been submitted; make it handle the test input too." % err)
    return txt

WORKER_SCRIPT = r"""
import sys,json,traceback,io,contextlib,copy,ast,os
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV sandbox blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
original_puzzle=None
for line in sys.stdin:
    try:
        msg=json.loads(line); op=msg.get("op")
        if op=="init":
            original_puzzle=copy.deepcopy(msg["puzzle"])
            ns={"__name__":"__main__","puzzle":copy.deepcopy(original_puzzle)}
            if msg.get("library"):
                exec(compile(msg["library"],"<piv-lib>","exec"),ns,ns)
            out={"ok":True,"text":"REPL initialized."}
        elif op=="exec":
            # Restore the protected puzzle binding before every cell. Other REPL state persists.
            ns["puzzle"]=copy.deepcopy(original_puzzle)
            buf=io.StringIO()
            try:
                with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                    # Jupyter-like semantics: execute statements, then display repr(last expression).
                    tree=ast.parse(msg["code"],mode="exec")
                    if tree.body and isinstance(tree.body[-1],ast.Expr):
                        prefix=ast.Module(body=tree.body[:-1],type_ignores=[])
                        if prefix.body: exec(compile(prefix,"<piv-repl>","exec"),ns,ns)
                        value=eval(compile(ast.Expression(tree.body[-1].value),"<piv-repl>","eval"),ns,ns)
                        if value is not None: print(repr(value))
                    else:
                        exec(compile(tree,"<piv-repl>","exec"),ns,ns)
                out={"ok":True,"text":buf.getvalue() or "Code executed successfully. No output was returned."}
            except BaseException:
                out={"ok":False,"text":buf.getvalue()+traceback.format_exc(limit=8)}
        elif op=="close":
            print(json.dumps({"ok":True,"text":"closing"}),flush=True); break
        else: out={"ok":False,"text":"Unknown worker op"}
    except BaseException:
        out={"ok":False,"text":traceback.format_exc(limit=8)}
    print(json.dumps(out,ensure_ascii=False),flush=True)
"""

class PersistentPythonWorker:
    def __init__(self,puzzle,timeout_sec=10.0):
        self.puzzle=puzzle; self.timeout_sec=float(timeout_sec); self.proc=None; self.successful_cells=[]; self.restart_count=0
        self.start()
    def start(self):
        self.close(silent=True)
        self.proc=subprocess.Popen([sys.executable,"-u","-c",WORKER_SCRIPT],stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                                   stderr=subprocess.DEVNULL,text=True,bufsize=1,cwd=tempfile.mkdtemp(prefix="piv_repl_"))
        r=self._request({"op":"init","puzzle":self.puzzle,"library":PIV_LIBRARY_SRC},timeout=max(2.0,self.timeout_sec))
        if not r.get("ok"): raise RuntimeError(f"REPL init failed: {r}")
        # Replay only previously successful bounded cells after a timeout/crash.
        if self.successful_cells:
            cells=list(self.successful_cells); self.successful_cells=[]
            for c in cells:
                rr=self.execute(c,record=True,replay=True)
                if not rr.get("ok"): break
    def _request(self,msg,timeout):
        if self.proc is None or self.proc.poll() is not None: raise RuntimeError("REPL process is not alive")
        self.proc.stdin.write(json.dumps(msg,ensure_ascii=False)+"\n"); self.proc.stdin.flush()
        sel=selectors.DefaultSelector(); sel.register(self.proc.stdout,selectors.EVENT_READ)
        events=sel.select(timeout); sel.close()
        if not events: raise TimeoutError(f"REPL tool exceeded {timeout}s")
        line=self.proc.stdout.readline()
        if not line: raise RuntimeError("REPL worker closed stdout")
        return json.loads(line)
    def execute(self,code,record=True,replay=False):
        ok,reason=guard_python_code(code,final=False)
        if not ok: return {"ok":False,"text":f"Tool code rejected by local safety guard: {reason}","guard_rejected":True,"restarted":False}
        try:
            r=self._request({"op":"exec","code":code},self.timeout_sec); r["restarted"]=False
            if r.get("ok") and record: self.successful_cells.append(code)
            return r
        except TimeoutError:
            self.restart_count+=1
            old=list(self.successful_cells)
            self.successful_cells=old
            try: self.start()
            except Exception: pass
            return {"ok":False,"text":f"Code execution terminated after {self.timeout_sec}s. Persistent worker was restarted; prior successful cells were replayed when possible.","timeout":True,"restarted":True}
        except Exception as e:
            self.restart_count+=1
            restarted=False
            try:
                self.start(); restarted=True
            except Exception:
                restarted=False
            return {"ok":False,"text":f"REPL infrastructure error: {type(e).__name__}: {e}. "
                                      f"Worker restart attempted; prior successful cells replayed when possible.",
                    "infrastructure_error":True,"restarted":restarted}
    def close(self,silent=False):
        p=getattr(self,"proc",None)
        if p is not None and p.poll() is None:
            try: p.terminate(); p.wait(timeout=1)
            except Exception:
                try: p.kill()
                except Exception: pass
        self.proc=None

VERIFIER_STAGE_SCRIPT = r"""
import sys,json,copy,traceback,numbers,os
payload=json.loads(sys.stdin.read()); code=payload["code"]; mode=payload["mode"]
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV verifier blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
def clean(g):
    try:
        import numpy as np
        if isinstance(g,np.ndarray): g=g.tolist()
    except Exception:
        pass
    try: rows=list(g)
    except Exception: raise ValueError("invalid_grid_rows")
    if not (1<=len(rows)<=30): raise ValueError("invalid_grid_rows")
    out=[]
    for r in rows:
        try:
            if hasattr(r,"tolist"): r=r.tolist()
            rr0=list(r)
        except Exception: raise ValueError("invalid_grid_rows")
        out.append(rr0)
    widths=[len(r) for r in out]
    if not widths or min(widths)<1 or max(widths)>30 or len(set(widths))!=1: raise ValueError("invalid_grid_shape")
    clean_out=[]
    for r in out:
        rr=[]
        for x in r:
            if isinstance(x,bool) or not isinstance(x,numbers.Integral): raise ValueError("non_integer_cell")
            x=int(x)
            if x<0 or x>9: raise ValueError("cell_out_of_domain")
            rr.append(x)
        clean_out.append(rr)
    return clean_out
RESULT_SENTINEL="PIV_RESULT:"
def emit(obj):
    sys.__stdout__.write("\n"+RESULT_SENTINEL+json.dumps(obj)+"\n"); sys.__stdout__.flush()
try:
    exec(code,ns,ns); fn=ns.get("solution")
    if not callable(fn): raise ValueError("solution_not_callable")
except BaseException:
    emit({"stage":mode,"ok":False,"code_error":traceback.format_exc(limit=10)}); raise SystemExit(0)
if mode=="train":
    task=payload["task"]
    train=[]
    for i,ex in enumerate(task["train"]):
        try:
            pred=clean(fn(copy.deepcopy(ex["input"])))
            exact=(pred==ex["output"])
            train.append({"index":i,"exact":exact,"pred":pred if not exact else None,"error":None})
        except BaseException:
            train.append({"index":i,"exact":False,"pred":None,"error":traceback.format_exc(limit=8)})
    emit({"stage":"train","ok":True,"train":train,"all_train_exact":bool(train) and all(x["exact"] for x in train)})
elif mode=="test":
    try:
        pred=clean(fn(copy.deepcopy(payload["test_input"])))
        emit({"stage":"test","ok":True,"pred":pred,"error":None})
    except BaseException:
        emit({"stage":"test","ok":False,"pred":None,"error":traceback.format_exc(limit=10)})
else:
    emit({"stage":mode,"ok":False,"code_error":"unknown_verifier_mode"})
"""

def task_without_test_solutions(task):
    return {"train":copy.deepcopy(task["train"]),"test":[{"input":copy.deepcopy(x["input"])} for x in task["test"]]}

def _run_verifier_stage(payload, timeout_sec):
    """Run one fresh verifier subprocess and return (started, result).

    `started` becomes True only after Popen succeeds. This distinguishes launch failures
    (no verifier process ever existed) from timeout/process/protocol failures after launch.
    """
    workdir=tempfile.mkdtemp(prefix="piv_verify_")
    proc=None; started=False
    stdout=""; stderr=""
    try:
        proc=subprocess.Popen([sys.executable,"-u","-c",VERIFIER_STAGE_SCRIPT],
                              stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                              text=True,cwd=workdir)
        started=True
        stdout,stderr=proc.communicate(input=json.dumps(payload),timeout=float(timeout_sec))
        base={"verifier_started":True,"returncode":proc.returncode}
        if proc.returncode!=0:
            return True,{**base,"ok":False,"error":(stderr or "")[-4000:] or f"verifier_returncode={proc.returncode}","stage_status":"process_error"}
        sentinel="PIV_RESULT:"
        payload_lines=[ln[len(sentinel):] for ln in (stdout or "").splitlines() if ln.startswith(sentinel)]
        if not payload_lines:
            return True,{**base,"ok":False,"error":"missing_verifier_result_sentinel","stage_status":"protocol_error","stdout_tail":(stdout or "")[-2000:]}
        r=json.loads(payload_lines[-1]); r.update(base); r["stage_status"]="ok"
        return True,r
    except subprocess.TimeoutExpired:
        if proc is not None:
            try: proc.kill()
            except Exception: pass
            try:
                _out,_err=proc.communicate(timeout=2)
                stdout=(stdout or "")+(_out or ""); stderr=(stderr or "")+(_err or "")
            except Exception: pass
        return bool(started),{"verifier_started":bool(started),"ok":False,"error":"verifier_timeout","stage_status":"timeout",
                              "stdout_tail":(stdout or "")[-2000:]}
    except Exception as e:
        if proc is not None:
            try: proc.kill()
            except Exception: pass
            try: proc.communicate(timeout=2)
            except Exception: pass
        _status="error" if started else "launch_error"
        return bool(started),{"verifier_started":bool(started),"ok":False,"error":f"{type(e).__name__}: {e}","stage_status":_status}
    finally:
        shutil.rmtree(workdir,ignore_errors=True)

VERIFIER_INFRA_STAGE_STATUSES={"launch_error","process_error","protocol_error","error"}

def _verifier_stage_is_infrastructure_failure(stage_status):
    return str(stage_status or "").strip().lower() in VERIFIER_INFRA_STAGE_STATUSES

def fresh_verify_solution(code,task,timeout_sec=VERIFIER_TIMEOUT_SEC):
    """Fresh two-stage verifier. Train evidence is finalized before any test subprocess starts.

    v2.4C keeps candidate-code failures distinct from verifier transport/process failures. A launch/process/
    protocol/host-side verifier failure invalidates infrastructure evidence; ordinary code rejection and timeouts
    remain candidate outcomes unless another infrastructure signal exists.
    """
    ok,reason=guard_python_code(code,final=True)
    if not ok:
        return {"ok":False,"verifier_started":False,"train_verifier_started":False,"test_verifier_started_count":0,
                "all_train_exact":False,"train":[],"test_outputs":[],"test_errors":[],"test_execution_ok":False,
                "any_test_output_ok":False,"error":reason,"verifier_status":"guard_rejected",
                "infrastructure_failure":False,"infrastructure_errors":[]}
    safe_task=task_without_test_solutions(task)
    train_started,tr=_run_verifier_stage({"mode":"train","code":code,"task":{"train":safe_task["train"]}}, timeout_sec)
    base={"ok":False,"verifier_started":bool(train_started),"train_verifier_started":bool(train_started),"test_verifier_started_count":0,
          "train":tr.get("train",[]),"all_train_exact":bool(tr.get("all_train_exact")),"test_outputs":[],"test_errors":[],
          "test_execution_ok":False,"any_test_output_ok":False,"train_error":None,"test_error":None,"error":None,
          "infrastructure_failure":False,"infrastructure_errors":[]}
    if tr.get("code_error"):
        base["train_error"]=tr.get("code_error"); base["error"]=tr.get("code_error"); base["verifier_status"]="train_code_error"; return base
    if tr.get("stage_status")!="ok":
        _st=str(tr.get("stage_status"))
        base["train_error"]=tr.get("error"); base["error"]=tr.get("error"); base["verifier_status"]="train_"+_st
        if _verifier_stage_is_infrastructure_failure(_st):
            base["infrastructure_failure"]=True
            base["infrastructure_errors"].append(f"train_verifier_{_st}: {tr.get('error')}")
        return base
    if not base["all_train_exact"]:
        base["verifier_status"]="train_rejected"; return base
    tests=[]; errors=[]; started_count=0
    for _test_idx,ex in enumerate(safe_task["test"]):
        started,te=_run_verifier_stage({"mode":"test","code":code,"test_input":ex["input"]}, timeout_sec)
        started_count += int(bool(started))
        _stage=str(te.get("stage_status") or "")
        if _stage=="ok" and te.get("ok") and te.get("pred") is not None:
            tests.append(te.get("pred")); errors.append(None)
        else:
            _err=te.get("error") or te.get("code_error") or _stage or "test_execution_failed"
            tests.append(None); errors.append(_err)
            if _verifier_stage_is_infrastructure_failure(_stage):
                base["infrastructure_failure"]=True
                base["infrastructure_errors"].append(f"test[{_test_idx}]_verifier_{_stage}: {_err}")
    base["test_verifier_started_count"]=started_count
    base["test_outputs"]=tests; base["test_errors"]=errors
    base["test_execution_ok"]=bool(tests) and all(x is not None for x in tests)
    base["any_test_output_ok"]=any(x is not None for x in tests)
    base["ok"]=base["any_test_output_ok"] and not base["infrastructure_failure"]
    if not base["test_execution_ok"]: base["test_error"]="\n".join(str(e) for e in errors if e)
    if base["infrastructure_failure"]:
        base["verifier_status"]="train_exact_test_infrastructure_failed"
    elif base["test_execution_ok"]: base["verifier_status"]="train_exact_test_ok"
    elif base["any_test_output_ok"]: base["verifier_status"]="train_exact_test_partial"
    else: base["verifier_status"]="train_exact_test_failed"
    return base

def verifier_candidate_status(verifier):
    """Single candidate-status authority; never relabel verifier infrastructure failure as model train rejection."""
    v=verifier or {}
    if v.get("infrastructure_failure"): return "verifier_infrastructure_failed"
    if v.get("all_train_exact") and v.get("test_execution_ok"): return "verified"
    if v.get("all_train_exact") and v.get("any_test_output_ok"): return "verified_partial_test"
    if v.get("all_train_exact"): return "test_execution_failed"
    return "train_rejected"

# ---------------------------- v2.4B TERMINAL ARTIFACT PARSER ----------------------------
# Moved here from Cell 8: Cell 7's GPU positive control must use the identical parser, and Cell 7
# runs before Cell 8. Keeping one implementation means the control really tests the production path.
_MODEL_CONTROL_TOKEN_RE=re.compile(r"<\|[^|>]*\|>")
def strip_model_control_tokens(text):
    """vLLM is called with skip_special_tokens=False (needed for </think> accounting), so the decoded
    text keeps chat-control tokens such as <|im_end|>. They are never part of the structured artifact."""
    return _MODEL_CONTROL_TOKEN_RE.sub("",str(text or ""))

def parse_terminal_json_output(text):
    """Parse terminal JSON without mutating its payload.

    Control tokens may surround the JSON with skip_special_tokens=False. Find the first JSON object and
    raw-decode it; do not regex-strip the whole string because solution_code may contain a literal like
    <|im_end|> that must survive byte-for-byte.
    """
    raw=str(text or "").strip()
    out={"ok":False,"status":"empty_output","solution_code":None,"json_object":None}
    if not raw or not strip_model_control_tokens(raw).strip():
        return out
    start=raw.find("{")
    if start<0:
        out["status"]="json_object_not_found"
        return out
    try:
        # raw_decode tolerates trailing whitespace/junk after a complete JSON object.
        obj,_end=json.JSONDecoder().raw_decode(raw[start:])
    except Exception as exc:
        out["status"]=f"json_parse_error:{type(exc).__name__}"
        return out
    out["json_object"]=obj
    if not isinstance(obj,dict) or set(obj.keys())!={"solution_code"}:
        out["status"]="json_schema_shape_mismatch"
        return out
    code=obj.get("solution_code")
    if not isinstance(code,str) or not code.strip():
        out["status"]="solution_code_missing_or_empty"
        return out
    try:
        tree=py_ast.parse(code)
    except Exception as exc:
        out["status"]=f"python_syntax_error:{type(exc).__name__}"
        return out
    if not any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in tree.body):
        out["status"]="missing_top_level_solution_function"
        return out
    safe,why=guard_python_code(code,final=True)
    if not safe:
        out["status"]=f"final_guard_rejected:{why}"
        return out
    out.update({"ok":True,"status":"ok","solution_code":code})
    return out

# ---------------------------- PRELOAD SELF-TESTS ----------------------------
self_tests={}
t=parse_agent_turn("reasoning\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>")
assert t["kind"]=="tool" and len(t["tool_calls"])==1 and t["tool_name"]=="python" and "puzzle.keys" in t["tool_args"]["code"]
t_multi=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nx=1\n</parameter>\n</function>\n</tool_call>\n<tool_call>\n<function=python>\n<parameter=code>\nx+1\n</parameter>\n</function>\n</tool_call>")
assert t_multi["kind"]=="tool" and len(t_multi["tool_calls"])==2
# v2.4E: malformed-tool regression must be in the committed action region. Without </think>,
# tool-like text is deliberately scratch reasoning and must remain kind="text".
t_unclosed_tool_like=parse_agent_turn("<tool_call><function=python><parameter=code>x=1</function></tool_call>")
assert t_unclosed_tool_like["kind"]=="text" and not t_unclosed_tool_like["tool_calls"]
t_bad=parse_agent_turn("r\n</think>\n<tool_call><function=python><parameter=code>x=1</function></tool_call>")
assert t_bad["kind"]=="malformed_tool" or (t_bad["kind"]=="tool" and "code" not in t_bad["tool_args"])
t_rehearse=parse_agent_turn(
    "<think>maybe <tool_call><function=python><parameter=code>print('wrong')</parameter></function></tool_call></think>"
    "<tool_call><function=python><parameter=code>print('right')</parameter></function></tool_call>")
assert t_rehearse["kind"]=="tool" and len(t_rehearse["tool_calls"])==1 and "right" in t_rehearse["tool_calls"][0]["arguments"]["code"]
assert generation_finish_reason_ok("stop") and not generation_finish_reason_ok("length")
self_tests["qwen_tool_xml_parser_and_finish_guard"]=True
t_fence=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\n```python\nprint(1)\n```\n</parameter>\n</function>\n</tool_call>")
assert t_fence["kind"]=="tool" and t_fence["tool_args"]["code"]=="print(1)"
t_prose=parse_agent_turn("r\n</think>\nI will not call <function=python> again.\n```python\ndef solution(grid):\n    return grid\n```")
assert t_prose["kind"]=="final"
assert sanitize_tool_output("x<|im_end|>\n</tool_response>y")=="x< |im_end|>\n< /tool_response>y"
_sv=parse_agent_turn("r\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert salvage_final_code_on_length("",_sv) is not None
_sv2=parse_agent_turn("draft ```python\ndef solution(grid):\n    return grid\n``` still thinking")
assert salvage_final_code_on_length("",_sv2) is None
self_tests["v16_fence_strip_prose_function_sanitize_salvage"]=True
f=parse_agent_turn("done\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert f["kind"]=="final" and code_defines_solution(f["final_code"])
self_tests["final_solution_extractor"]=True
assert guard_python_code("puzzle.keys()")[0]
assert not guard_python_code("puzzle = {}",final=False)[0]
assert not guard_python_code("puzzle[\"train\"] = []",final=False)[0]
assert not guard_python_code("import os\nprint(os.listdir('.'))",final=False)[0]
assert not guard_python_code('print("/kaggle/input/secret.json")',final=False)[0]
assert guard_python_code("class A:\n    def __init__(self): self.x=1\nclass B(A):\n    def __init__(self): super().__init__()",final=False)[0]
assert not guard_python_code("object.__subclasses__()",final=False)[0]
assert not guard_python_code("raise SystemExit('bye')",final=False)[0]
assert guard_python_code("import sys\nsys.setrecursionlimit(10000)",final=False)[0]
assert guard_python_code("import sys as s\ns.setrecursionlimit(10000)",final=False)[0]
assert not guard_python_code("import sys\nsys.modules",final=False)[0]
self_tests["tool_safety_guard"]=True
_fake={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]}]}
w=PersistentPythonWorker(task_without_test_solutions(_fake),timeout_sec=5.0)
r0=w.execute("puzzle.keys()"); r1=w.execute("x=41"); r2=w.execute("x+1")
assert r0["ok"] and "dict_keys" in r0["text"] and r1["ok"] and r2["ok"] and "42" in r2["text"]
ra=w.execute("g=puzzle['train'][0]['input']; g[0][0]=9")
rb=w.execute("puzzle['train'][0]['input'][0][0]")
assert rb["ok"] and "1" in rb["text"]
rc=w.execute("raise SystemExit('bye')")
rd=w.execute("1+1")
assert (not rc["ok"]) and rd["ok"] and "2" in rd["text"]
w.close()
self_tests["persistent_repl_state_and_expression_display"]=True
v=fresh_verify_solution("def solution(grid):\n    return [row[:] for row in grid]",_fake,timeout_sec=5)
assert v["all_train_exact"] and v["test_outputs"]==[[[2]]]
v_noise=fresh_verify_solution(
    "print('noise', end='')\ndef solution(grid):\n    return [row[:] for row in grid]",
    _fake,timeout_sec=5)
assert v_noise["all_train_exact"] and v_noise["test_outputs"]==[[[2]]]
self_tests["verifier_result_sentinel"]=True
self_tests["fresh_exact_verifier"]=True
v2=fresh_verify_solution("def solution(grid):\n    return [[9]]",_fake,timeout_sec=5)
assert not v2["all_train_exact"] and v2["test_outputs"]==[]
self_tests["train_fail_blocks_test_execution"]=True
v3=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('test-only failure')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake,timeout_sec=5)
assert v3["all_train_exact"] and not v3["test_execution_ok"] and v3["test_outputs"]==[None]
self_tests["train_exact_survives_test_execution_failure"]=True
_fake_multi={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]},{"input":[[3]]}]}
v4=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('one test fails')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake_multi,timeout_sec=5)
assert v4["all_train_exact"] and v4["test_outputs"][0] is None and v4["test_outputs"][1]==[[3]]
self_tests["partial_multiquery_test_outputs_preserved"]=True
# v2.1 regression: a test-only timeout must preserve already established train-exact evidence.
v_timeout=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]:\n        while True: pass\n    return [row[:] for row in grid]",
    _fake,timeout_sec=2.0)
assert v_timeout["verifier_started"] and v_timeout["all_train_exact"] and not v_timeout["test_execution_ok"] and v_timeout["test_outputs"]==[None]
self_tests["train_exact_survives_test_timeout"]=True
# v2.1 regression: guard rejection must not claim a verifier subprocess actually started.
v_guard=fresh_verify_solution("import os\ndef solution(grid): return grid",_fake,timeout_sec=1)
assert v_guard["verifier_status"]=="guard_rejected" and not v_guard["verifier_started"]
self_tests["guard_reject_does_not_overstate_verifier_execution"]=True
# v2.2 regression: a process-launch failure must NOT claim verifier_started=True.
_orig_popen=subprocess.Popen
def _launch_boom(*a,**k):
    raise FileNotFoundError("synthetic verifier launch failure")
try:
    subprocess.Popen=_launch_boom
    _started,_launch_res=_run_verifier_stage({"mode":"train","code":"def solution(grid): return grid","task":{"train":[]}},1.0)
finally:
    subprocess.Popen=_orig_popen
assert (not _started) and (not _launch_res.get("verifier_started")) and _launch_res.get("stage_status")=="launch_error", _launch_res
self_tests["verifier_launch_failure_not_counted_as_started"]=True
# v2.4C regression: launch failure must propagate through fresh_verify_solution as infrastructure-invalid evidence.
_orig_popen_v24c=subprocess.Popen
def _launch_boom_v24c(*a,**k):
    raise FileNotFoundError("synthetic fresh-verifier launch failure")
try:
    subprocess.Popen=_launch_boom_v24c
    _v24c_vi=fresh_verify_solution("def solution(grid): return [row[:] for row in grid]",_fake,timeout_sec=1)
    assert _v24c_vi["verifier_status"]=="train_launch_error"
    assert _v24c_vi["infrastructure_failure"] and _v24c_vi["infrastructure_errors"]
    assert verifier_candidate_status(_v24c_vi)=="verifier_infrastructure_failed"
finally:
    subprocess.Popen=_orig_popen_v24c
self_tests["v24c_verifier_infrastructure_propagation"]=True

# ---- v2.4B regressions -------------------------------------------------------------------
# 1) Uniformly indented XML payloads must stay valid Python.
_t_indent=parse_agent_turn("r\n</think>\n<tool_call><function=python><parameter=code>\n    x=1\n    print(x)\n</parameter></function></tool_call>")
assert _t_indent["kind"]=="tool" and _t_indent["tool_args"]["code"]=="x=1\nprint(x)", _t_indent["tool_args"]
_t_inline=parse_agent_turn("r\n</think>\n<tool_call><function=python><parameter=code>puzzle.keys()</parameter></function></tool_call>")
assert _t_inline["tool_args"]["code"]=="puzzle.keys()", _t_inline["tool_args"]
# genuinely indented bodies (a function defined inside the payload) must NOT be flattened
_t_body=parse_agent_turn("r\n</think>\n<tool_call><function=python><parameter=code>\ndef f():\n    return 1\nprint(f())\n</parameter></function></tool_call>")
assert _t_body["tool_args"]["code"]=="def f():\n    return 1\nprint(f())", _t_body["tool_args"]
py_ast.parse(_t_indent["tool_args"]["code"]); py_ast.parse(_t_body["tool_args"]["code"])
self_tests["v24b_parameter_dedent"]=True
# 2) The terminal parser must survive skip_special_tokens=False decoding.
_good_json=json.dumps({"solution_code":"def solution(grid):\n    return [row[:] for row in grid]"})
assert parse_terminal_json_output(_good_json)["ok"]
assert parse_terminal_json_output(_good_json+"<|im_end|>")["ok"], "control tokens must not break terminal parsing"
assert parse_terminal_json_output(_good_json+"\n<|endoftext|>")["ok"]
assert parse_terminal_json_output("<|im_start|>assistant\n"+_good_json+"<|im_end|>")["ok"]
assert parse_terminal_json_output('{"solution_code":"def solution(grid):')["status"].startswith("json_parse_error")
assert parse_terminal_json_output("")["status"]=="empty_output"
assert parse_terminal_json_output("<|im_end|>")["status"]=="empty_output"
assert parse_terminal_json_output(json.dumps({"solution_code":"def other(g): return g"}))["status"]=="missing_top_level_solution_function"
assert parse_terminal_json_output(json.dumps({"solution_code":"import os\ndef solution(g): return g"}))["status"].startswith("final_guard_rejected")
_literal_code='def solution(grid):\n    marker="<|im_end|>"\n    return [row[:] for row in grid]'
_literal_parsed=parse_terminal_json_output(json.dumps({"solution_code":_literal_code})+"<|im_end|>")
assert _literal_parsed["ok"] and _literal_parsed["solution_code"]==_literal_code
_unclosed_draft="<think>still reasoning\n```python\ndef solution(grid):\n    return grid\n```"
_unclosed_parsed=parse_agent_turn(_unclosed_draft)
assert _unclosed_parsed["kind"]=="text" and _unclosed_parsed["final_code"] is None and _unclosed_parsed["reasoning_mode"]=="implicit_open_think"
self_tests["v24d_terminal_literal_preservation"]=True
self_tests["v24d_unclosed_reasoning_not_final"]=True
self_tests["v24b_terminal_parser_control_tokens"]=True
# 3) Every verifier return path exposes "ok".
for _vr in (fresh_verify_solution("def solution(grid):\n    return [[9]]",_fake,timeout_sec=5),
            fresh_verify_solution("import os\ndef solution(grid): return grid",_fake,timeout_sec=5),
            fresh_verify_solution("def solution(grid):\n    return [row[:] for row in grid]",_fake,timeout_sec=5)):
    assert "ok" in _vr, _vr
self_tests["v24b_verifier_ok_key_always_present"]=True

with open(WORK_ROOT/"preload_self_tests.json","w") as f: json.dump(self_tests,f,indent=2)
print("PIV preload self-tests PASS:",self_tests)


# =====================================================================================
# v2.5 — CANDIDATE-FIRST REPAIR HELPERS (defined before model allocation)
# =====================================================================================

def _v25_exception_last_line(err, max_chars=220):
    lines=[x.strip() for x in str(err or "").splitlines() if x.strip()]
    out=(lines[-1] if lines else "execution_error")
    return out[:int(max_chars)]

def _v25_grid_dims(g):
    try:
        if not isinstance(g,list) or not g: return None
        if not all(isinstance(r,list) for r in g): return None
        widths=[len(r) for r in g]
        if not widths or min(widths)<1 or len(set(widths))!=1: return None
        return (len(g),widths[0])
    except Exception:
        return None

def _v25_pair_diag(entry, ex, mismatch_limit=4):
    idx=int(entry.get("index",0))
    if entry.get("exact"):
        return {"pair":idx,"kind":"exact"}
    err=entry.get("error")
    if err:
        return {"pair":idx,"kind":"exception","error":_v25_exception_last_line(err)}
    pred=entry.get("pred")
    expected=ex.get("output")
    identity=bool(pred==ex.get("input")) if pred is not None else False
    got_dims=_v25_grid_dims(pred); exp_dims=_v25_grid_dims(expected)
    if got_dims is None:
        return {"pair":idx,"kind":"invalid_output","got_dims":None,"expected_dims":exp_dims,"identity":identity}
    if got_dims!=exp_dims:
        return {"pair":idx,"kind":"dims_mismatch","got_dims":got_dims,"expected_dims":exp_dims,"identity":identity}
    H,W=got_dims
    mism=[]; wrong=0
    for r in range(H):
        for c in range(W):
            if pred[r][c]!=expected[r][c]:
                wrong+=1
                if len(mism)<int(mismatch_limit):
                    mism.append((r,c,expected[r][c],pred[r][c]))
    return {"pair":idx,"kind":"cells_wrong","got_dims":got_dims,"expected_dims":exp_dims,
            "wrong_cells":int(wrong),"total_cells":int(H*W),"mismatches":mism,"identity":identity}

def _v25_feedback_bounded(txt, max_feedback_tokens):
    ids=list(tokenizer.encode(txt,add_special_tokens=False))
    truncated=False
    if len(ids)>int(max_feedback_tokens):
        txt=tokenizer.decode(ids[:int(max_feedback_tokens)],skip_special_tokens=False).strip()
        truncated=True
    return txt,len(tokenizer.encode(txt,add_special_tokens=False)),truncated

def _v25_feedback_text(verifier, task, max_feedback_tokens=REPAIR_FEEDBACK_MAX_TOKENS):
    train=list((verifier or {}).get("train") or [])
    exact=sum(1 for x in train if x.get("exact"))
    n=len(task.get("train",[]))
    if not train:
        # v2.5A: train_timeout / train_code_error return no per-pair results. Report the verifier's own error
        # instead of claiming an exception on every pair.
        _vs=str((verifier or {}).get("verifier_status") or "unknown")
        _ve=_v25_exception_last_line((verifier or {}).get("train_error") or (verifier or {}).get("error") or "no verifier error text")
        _txt="\n".join([f"candidate #1 fresh-train result: 0/{n} exact.",
                        f"VERIFIER: {_vs}; {_ve}",
                        "The verifier returned no per-pair result, so the whole program failed before any pair was scored.",
                        "Return a self-contained program that terminates quickly and avoids the reported failure.",
                        "Repair the standalone solution(grid) program. Do not rely on REPL variables or helper state from exploration."]).strip()
        _txt,_tok,_tr=_v25_feedback_bounded(_txt,max_feedback_tokens)
        return {"text":_txt,"tokens":_tok,"truncated":_tr,"train_exact_count":0,"n_train":int(n),
                "identity_pairs":[],"pair_diags":[],"verifier_level_failure":True}
    diags=[]
    by_index={int(x.get("index",i)):x for i,x in enumerate(train)}
    for i,ex in enumerate(task.get("train",[])):
        ent=by_index.get(i,{"index":i,"exact":False,"error":"missing verifier pair result"})
        diags.append(_v25_pair_diag(ent,ex))
    identity_pairs=[d["pair"] for d in diags if d.get("identity")]
    lines=[f"candidate #1 fresh-train result: {exact}/{n} exact."]
    if identity_pairs:
        joined=",".join(map(str,identity_pairs))
        lines.append(f"IDENTITY FAILURE: your program returned the input unchanged on train pair(s) {joined}; no transformation was applied there.")
    for d in diags:
        if d["kind"]=="exact":
            continue
        i=d["pair"]
        if d["kind"]=="exception":
            lines.append(f"pair {i}: EXCEPTION {d['error']}")
        elif d["kind"]=="dims_mismatch":
            gd=d.get("got_dims"); ed=d.get("expected_dims"); ident="; output==input" if d.get("identity") else ""
            lines.append(f"pair {i}: DIMS got {gd[0]}x{gd[1]} / expected {ed[0]}x{ed[1]}{ident}")
        elif d["kind"]=="invalid_output":
            ed=d.get("expected_dims"); es=(f"{ed[0]}x{ed[1]}" if ed else "unknown")
            lines.append(f"pair {i}: INVALID OUTPUT / expected dims {es}")
        else:
            gd=d["got_dims"]
            mm="; ".join(f"({r},{c}): expected {e}, got {g}" for r,c,e,g in d.get("mismatches",[]))
            ident="; output==input" if d.get("identity") else ""
            lines.append(f"pair {i}: dims got {gd[0]}x{gd[1]} / expected {gd[0]}x{gd[1]}; wrong cells {d['wrong_cells']}/{d['total_cells']}{ident}; first mismatches: {mm}")
    lines.append("Repair the standalone solution(grid) program. Do not rely on REPL variables or helper state from exploration.")
    txt="\n".join(lines).strip()
    ids=list(tokenizer.encode(txt,add_special_tokens=False))
    truncated=False
    if len(ids)>int(max_feedback_tokens):
        txt=tokenizer.decode(ids[:int(max_feedback_tokens)],skip_special_tokens=False).strip()
        truncated=True
    return {"text":txt,"tokens":len(tokenizer.encode(txt,add_special_tokens=False)),"truncated":truncated,
            "train_exact_count":int(exact),"n_train":int(n),"identity_pairs":identity_pairs,"pair_diags":diags}

def _v25_repair_feedback_marker(feedback_text):
    canonical=str(feedback_text or "").strip()
    sha=hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    return canonical,sha,f"[V25_REPAIR_FEEDBACK_SHA256={sha}]"

def _v25_build_repair_messages(task, candidate_code, feedback_text):
    canonical,sha,marker=_v25_repair_feedback_marker(feedback_text)
    safe_task=task_without_test_solutions(task)
    puzzle_text=puzzle_to_source_text(safe_task)
    system=("You repair one ARC Python solution after fresh training verification. "
            "Do not use tools. Do not expose chain-of-thought. Return only the required JSON object with key solution_code.")
    user=("Repair candidate #1 into one complete standalone def solution(grid) program. Test solutions are unavailable. "
          "The puzzle below contains training input/output pairs and test inputs only.\n\n"
          f"PUZZLE:\n{puzzle_text}\n\nCANDIDATE #1:\n```python\n{strip_library(candidate_code)}\n```\n\n"
          f"{marker}\n{canonical}\n[/V25_REPAIR_FEEDBACK]\n\n"
          "Use the verifier evidence to correct the transformation. Return the full corrected program, not a patch or explanation. "
          "No comments or docstrings. The preloaded helpers (components, bbox, crop, rot90, flip_h, flip_v, transpose, "
          "color_counts, background_color, grid_shape, diff_cells) are available; do not redefine them.")
    msgs=[{"role":"system","content":system},{"role":"user","content":user}]
    rendered,method=render_terminal_chat(msgs)
    # Qwen's template may trim message-edge whitespace. Canonical stripped feedback is the transport authority.
    stripped_present=bool(canonical and canonical in rendered)
    marker_present=bool(marker in rendered)
    raw_present=bool(str(feedback_text or "") in rendered)
    return {"messages":msgs,"rendered":rendered,"render_method":method,"feedback_sha256":sha,
            "feedback_raw_sha256":hashlib.sha256(str(feedback_text or "").encode("utf-8")).hexdigest(),
            "feedback_present_raw":raw_present,"feedback_present_stripped":stripped_present,
            "feedback_marker_present":marker_present}

def _v25_repair_admission(prompt_tokens, generated_tokens):
    remaining=max(0,int(TOTAL_GENERATED_TOKEN_CAP)-int(generated_tokens))
    safe=max(0,int(MAX_MODEL_LEN)-int(prompt_tokens)-int(CONTEXT_SAFETY_TOKENS))
    cap=min(int(REPAIR_MAX_TOKENS),safe)   # v3: repair has its own budget; leftover exploration tokens no longer cap it
    return {"cap":max(0,int(cap)),"remaining_total_tokens":remaining,"context_headroom":safe}

def _v25_train_exact_count(verifier):
    return int(sum(1 for x in list((verifier or {}).get("train") or []) if x.get("exact")))

def _v25_choose_candidate(code1,v1,code2,v2):
    """Quality-preserving selection. Candidate #2 is selected only if it strictly improves train exact count or reaches all-train-exact."""
    c1=_v25_train_exact_count(v1); c2=_v25_train_exact_count(v2)
    if bool((v2 or {}).get("all_train_exact")):
        return 2,code2,v2
    if c2>c1:
        return 2,code2,v2
    return 1,code1,v1

def _v25_repair_output_cap_violation(observed_tokens,requested_cap):
    return int(observed_tokens)>int(requested_cap)

def run_v25_repair_helper_selftests():
    fake_task={"train":[
        {"input":[[1,0],[0,1]],"output":[[0,1],[1,0]]},
        {"input":[[1,2,3]],"output":[[3,2,1]]},
        {"input":[[9]],"output":[[8]]},
        {"input":[[4]],"output":[[5]]},
    ],"test":[{"input":[[2,2]]}]}
    fake_v={"train":[
        {"index":0,"exact":False,"pred":[[1,0],[0,1]],"error":None},
        {"index":1,"exact":False,"pred":[[3],[2],[1]],"error":None},
        {"index":2,"exact":False,"pred":None,"error":"Traceback...\nTypeError: NoneType not subscriptable\n"},
        {"index":3,"exact":True,"pred":None,"error":None},
    ],"all_train_exact":False,"verifier_status":"train_rejected"}
    fb=_v25_feedback_text(fake_v,fake_task)
    assert fb["tokens"]<=REPAIR_FEEDBACK_MAX_TOKENS,fb
    assert 0 in fb["identity_pairs"],fb
    assert "IDENTITY FAILURE" in fb["text"] and "DIMS got 3x1 / expected 1x3" in fb["text"] and "TypeError:" in fb["text"],fb
    # v2.5A: identity with CHANGED dims is dims_mismatch; identity with SAME dims is cells_wrong. Both must set identity.
    _dim_identity=_v25_pair_diag({"index":0,"exact":False,"pred":[[1,2,3]],"error":None},{"input":[[1,2,3]],"output":[[1],[2],[3]]})
    assert _dim_identity["kind"]=="dims_mismatch" and _dim_identity["identity"],_dim_identity
    _cells_identity=_v25_pair_diag({"index":1,"exact":False,"pred":fake_task["train"][1]["input"],"error":None},fake_task["train"][1])
    assert _cells_identity["kind"]=="cells_wrong" and _cells_identity["identity"],_cells_identity
    # Trailing whitespace must not create a false negative in the transport assertion.
    tr=_v25_build_repair_messages(fake_task,"def solution(grid):\n    return grid",fb["text"]+"\n\n")
    assert tr["feedback_present_stripped"] and tr["feedback_marker_present"],tr
    # v2.5A: verifier-level failure (no per-pair results) must surface the verifier error, not fake exceptions.
    _vl=_v25_feedback_text({"train":[],"all_train_exact":False,"verifier_status":"train_timeout",
                            "train_error":"verifier stage timeout after 20s"},fake_task)
    assert _vl["verifier_level_failure"] and "VERIFIER: train_timeout" in _vl["text"] and "timeout after 20s" in _vl["text"],_vl
    assert "missing verifier pair result" not in _vl["text"],_vl
    # Failure injection: repair prompt overflow -> no admission.
    adm=_v25_repair_admission(MAX_MODEL_LEN+100,0)
    assert adm["cap"]==0,adm
    # Failure injection: verifier timeout in repair must not erase a stronger candidate #1.
    v1={"train":[{"exact":True},{"exact":False}],"all_train_exact":False,"verifier_status":"train_rejected"}
    v2={"train":[],"all_train_exact":False,"verifier_status":"train_timeout"}
    sel=_v25_choose_candidate("c1",v1,"c2",v2)
    assert sel[0]==1,sel
    # Failure injection: over-cap detection.
    assert _v25_repair_output_cap_violation(101,100) and not _v25_repair_output_cap_violation(100,100)
    payload={"ok":True,"feedback_tokens":fb["tokens"],"identity_flag":True,"dims_diff":True,
             "exception_last_line":True,"transport_stripped_presence":True,"prompt_overflow_injection":True,
             "repair_verifier_timeout_preserves_candidate1":True,"overcap_injection":True,
             "verifier_level_failure_feedback":True}
    with open(WORK_ROOT/"v25_repair_helper_selftest.json","w") as f: json.dump(payload,f,indent=2)
    print("v2.5 repair helper self-tests PASS:",payload)
    return payload

V25_REPAIR_HELPER_SELFTEST=run_v25_repair_helper_selftests()

# ---------------------------------------------------------------- v3: library-aware verification
# Every program that is verified or executed is the preloaded helper library followed by the model's code, so model
# code may call the helpers it saw in the REPL. Self-tests above ran against the raw verifier.
_fresh_verify_solution_raw = fresh_verify_solution
def fresh_verify_solution(code, task, timeout_sec=VERIFIER_TIMEOUT_SEC):
    return _fresh_verify_solution_raw(with_library(code) if code else code, task, timeout_sec)

def run_v3_library_selftests():
    _t = {"train": [{"input": [[1, 0], [0, 0]], "output": [[1]]}, {"input": [[0, 0], [0, 7]], "output": [[7]]}],
          "test": [{"input": [[0, 3], [0, 0]]}]}
    _v = fresh_verify_solution("def solution(grid):\n    return crop(grid, *bbox(grid, ignore=0))", _t)
    assert _v.get("all_train_exact") and _v.get("test_outputs") == [[[3]]], _v
    _m = build_candidate_module(["x = puzzle['train'][0]['input']\nprint(x)",
                                 "K = 0\ndef helper(g):\n    return crop(g, *bbox(g, ignore=K))",
                                 "def solution(grid):\n    return helper(grid)\nverify_solution()"])
    assert _m and "print(x)" not in _m and "verify_solution()" not in _m and "K = 0" in _m, _m
    _v2 = fresh_verify_solution(_m, _t)
    assert _v2.get("all_train_exact"), _v2
    assert build_candidate_module(["print(1)"]) is None
    # v3.1 regression tests (review defect 6)
    _t4 = {"train": [{"input": [[0]], "output": [[4]]}, {"input": [[1]], "output": [[4]]}], "test": [{"input": [[2]]}]}
    _m = build_candidate_module(["DIRS = list(zip([0, 1, 0, -1], [1, 0, -1, 0]))",
                                 "def solution(grid):\n    return [[len(DIRS)]]"])
    assert _m and "DIRS = list(zip(" in _m and fresh_verify_solution(_m, _t4).get("all_train_exact"), _m   # computed constant
    _t7 = {"train": [{"input": [[0]], "output": [[7]]}], "test": [{"input": [[2]]}]}
    _m = build_candidate_module(["T = {}", "T[1, 1] = 3\nT.update({(2, 2): 4})",
                                 "def solution(grid):\n    return [[T[1, 1] + T[2, 2]]]"])
    assert _m and "T.update(" in _m and fresh_verify_solution(_m, _t7).get("all_train_exact"), _m        # mutation
    _m = build_candidate_module(["from typing import List\nGrid = List[List[int]]",
                                 "def solution(grid: Grid) -> Grid:\n    return [[4]]"])
    assert _m and "Grid = List[List[int]]" in _m and fresh_verify_solution(_m, _t4).get("all_train_exact"), _m  # type alias
    _m = build_candidate_module(["N = len(puzzle['train'])\nX = [p['input'] for p in puzzle['train']]\nY = N + 1",
                                 "def learn():\n    return puzzle['train'][0]['output']", "RULE = learn()",
                                 "R = print('x')\nZ = verify_solution()\nfor k in range(3):\n    pass\nM = 0\nM += 1\nU += 1",
                                 "def solution(grid):\n    return [[4]]"])
    _ms = strip_library(_m)
    assert _m and "N =" not in _ms and "X =" not in _ms and "Y =" not in _ms and "RULE" not in _ms and "R =" not in _ms \
        and "Z =" not in _ms and "for " not in _ms and "M += 1" in _ms and "U +=" not in _ms, _ms      # exclusions
    assert "def learn" in _ms and fresh_verify_solution(_m, _t4).get("all_train_exact"), _ms
    assert build_candidate_module(["def solve(g):\n    return g\nsolution = solve"]) is None            # alias is not def
    # v3.1 S12 audit: the fresh module must never silently differ from the REPL
    for _cells in (["K = 0", "K = puzzle['train'][0]['input'][0][0]", "def solution(grid):\n    return [[K]]"],
                   ["K = 0", "def solution(grid):\n    return [[K]]", "K = puzzle['train'][0]['input'][0][0]"],
                   ["def helper(g):\n    return puzzle['train'][0]['output']", "def solution(grid):\n    return helper(grid)"],
                   ["T = {}", "T.update({1: len(puzzle['train'])})", "def solution(grid):\n    return [[T[1]]]"],
                   ["T = {}", "for ex in puzzle['train']:\n    T[len(ex['input'])] = ex['output']", "def solution(grid):\n    return T[len(grid)]"],
                   ["K = 1", "for K in range(3):\n    pass", "def solution(grid):\n    return [[K]]"],
                   ["def solution(grid):\n    verify_solution()\n    return grid"]):
        _mm, _inf = build_candidate_module_ex(_cells)
        assert _mm is None and _inf["tainted_solution"], (_cells, _mm, _inf)
    _mm, _inf = build_candidate_module_ex(["K = puzzle['x']", "K = 5", "def solution(grid):\n    return [[K]]"])
    assert _mm and "K = 5" in _mm and not _inf["tainted_solution"], (_mm, _inf)                               # safe rebinding
    _mm, _inf = build_candidate_module_ex(["print(sorted(color_counts(puzzle['train'][0]['input'])))\nx = 3",
                                           "def solution(grid):\n    return [[4]]"])
    assert _mm and not _inf["tainted_solution"], _inf                                    # unrelated exploration is fine
    _mm, _inf = build_candidate_module_ex(["g = puzzle['train'][0]['input']\nfor r in range(3):\n    pass\nout = 1",
                                           "def solution(grid):\n    g = grid\n    out = [[4 for _ in row] for row in g][:1]\n"
                                           "    for r in out:\n        pass\n    return [out[0][:1]]"])
    assert _mm and not _inf["tainted_solution"] and fresh_verify_solution(_mm, _t4).get("all_train_exact"), (_mm, _inf)  # locals shadow
    # v3.1 third audit: walrus, nested-function scope, comprehension shadowing, del, except target, global write
    for _cells in (["K = 0", "(K := puzzle['train'][0]['input'][0][0])", "def solution(grid):\n    return [[K]]"],
                   ["K = 0", "K = puzzle['train'][0]['input'][0][0]",
                    "def solution(grid):\n    x = K\n    def inner():\n        K = 2\n        return K\n    return [[x]]"],
                   ["K = 0", "K = puzzle['train'][0]['input'][0][0]", "X = (K, [K for K in range(3)])",
                    "def solution(grid):\n    return [[X[0]]]"],
                   ["K = 0", "del K", "def solution(grid):\n    return [[K]]"],
                   ["E = 1", "try:\n    pass\nexcept Exception as E:\n    pass", "def solution(grid):\n    return [[E]]"],
                   ["K = 1", "def setk():\n    global K\n    K = len(puzzle['train'])", "setk()", "def solution(grid):\n    return [[K]]"],
                   ["K = 1", "Y = [w for w in range(3) if (K := w)]", "def solution(grid):\n    return [[K]]"]):
        _mm, _inf = build_candidate_module_ex(_cells)
        assert _mm is None and _inf["tainted_solution"], (_cells, _mm, _inf)
    _mm, _inf = build_candidate_module_ex(["K = 0", "def solution(grid):\n    def inner():\n        K = 2\n        return K\n    return [[inner() + 2]]"])
    assert _mm and fresh_verify_solution(_mm, _t4).get("all_train_exact"), (_mm, _inf)        # inner local is not a global read
    _mm, _inf = build_candidate_module_ex(["K = 4", "X = (K, [K for K in range(3)])", "def solution(grid):\n    return [[X[0]]]"])
    assert _mm and fresh_verify_solution(_mm, _t4).get("all_train_exact"), (_mm, _inf)        # comprehension keeps outer K
    # runtime fidelity check against a live REPL
    _wk = PersistentPythonWorker(_t4, timeout_sec=10.0)
    try:
        _wk.execute("def solution(grid):\n    return [[4]]")
        _good = fresh_verify_solution("def solution(grid):\n    return [[4]]", _t4)
        _bad = fresh_verify_solution("def solution(grid):\n    return [[0]]", _t4)
        assert _v3_fidelity_check(_wk, _good, _t4)[0] == "match"
        assert _v3_fidelity_check(_wk, _bad, _t4)[0] == "mismatch"
        _wk.execute("def solution(grid):\n    return [[grid[0][0] + 4]]")
        assert _v3_fidelity_check(_wk, _good, _t4)[0] == "mismatch"          # differs on train input [[1]]
        assert _wk.execute("print(solution([[0]]))").get("ok")                # REPL state intact after the probe
        assert len(_wk.successful_cells) == 3                                 # probe is not recorded as a cell
    finally:
        _wk.close()
    _q = lambda tr, all_ex, tok: _v3_train_quality({"train": tr, "all_train_exact": all_ex, "test_execution_ok": tok}, _tq)
    _tq = {"train": [{"input": [[0, 0]], "output": [[1, 2]]}, {"input": [[0, 0]], "output": [[3, 4]]}], "test": []}
    _E = lambda i, g: {"index": i, "exact": False, "pred": g}
    _X = lambda i: {"index": i, "exact": True}
    _order = [_q([_X(0), _X(1)], True, True), _q([_X(0), _X(1)], True, False), _q([_X(0), _E(1, [[3, 9]])], False, False),
              _q([_E(0, [[1, 9]]), _E(1, [[3, 9]])], False, False), _q([_E(0, [[9, 9]]), _E(1, [[9, 9]])], False, False),
              _q([_E(0, [[1]]), _E(1, [[3]])], False, False),
              _q([{"index": 0, "exact": False, "error": "boom"}, {"index": 1, "exact": False, "error": "boom"}], False, False)]
    assert all(_order[k] > _order[k + 1] for k in range(len(_order) - 1)), _order                           # ranking
    assert "TEST INPUT FAILURE" in _v3_report_text({"train": [_X(0), _X(1)], "all_train_exact": True,
                                                    "test_execution_ok": False, "test_error": "KeyError: 7"}, _tq)
    assert strip_library(with_library("def solution(g):\n    return g")).startswith("def solution")
    ok, why = guard_python_code(with_library("def solution(g):\n    return g"), final=True)
    assert ok, why
    payload = {"library_in_verified_program": True, "candidate_module_builder": True, "guard_accepts_library": True,
               "builder_regressions": True, "builder_taint_fidelity": True, "builder_symtable_scope": True,
               "runtime_fidelity_check": True, "near_miss_ranking": True,
               "test_failure_report": True}
    (WORK_ROOT / "v3_library_selftest.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("v3 library / candidate-builder self-tests PASS:", payload)
    return payload

V3_LIBRARY_SELFTEST = run_v3_library_selftests()


In [ ]:
# =====================================================================================
# CELL 7 — CPU-SAFE PREFLIGHT OR REAL vLLM/TP=4 MODEL LOAD (C1: CUDA graph / no custom AR)
# Dependency versions were already pinned in Cell 1 for PREFLIGHT/production parity.
# =====================================================================================
gpu_info = []
notebook_start = NOTEBOOK_WALL_START
model_load_sec = 0.0
warmup_sec = 0.0
arc_llm = None
# v2.4B: defined unconditionally so downstream cells never depend on how far bootstrap got.
TERMINAL_POSITIVE_CONTROL = {"ran": False, "ok": None}
ENGINE_BOOTSTRAP_OK = (not USE_MODEL)
ENGINE_BOOTSTRAP_STAGE = "not_requested" if not USE_MODEL else "not_started"
ENGINE_BOOTSTRAP_ERROR = None
ENGINE_BOOTSTRAP_TRACEBACK = None

RUNTIME_VLLM_VERSION = pkg_version("vllm")
RUNTIME_TORCH_VERSION = pkg_version("torch")
RUNTIME_TRANSFORMERS_VERSION = pkg_version("transformers")
RUNTIME_TOKENIZERS_VERSION = pkg_version("tokenizers")
RUNTIME_HF_HUB_VERSION = pkg_version("huggingface-hub")
RUNTIME_CUDA_VERSION = None

deadline = notebook_start + TOTAL_RUNTIME_SEC - SAFETY_RESERVE_SEC

# Fingerprint the mounted checkpoint without hashing the huge shard bytes.
with open(index_path, "r", encoding="utf-8") as f:
    _model_index = json.load(f)
_model_shards = sorted(set((_model_index.get("weight_map") or {}).values()))
_model_shard_layout = []
for _name in _model_shards:
    _p = MODEL_PATH / _name
    _model_shard_layout.append({
        "name": _name,
        "bytes": int(_p.stat().st_size) if _p.exists() else None,
    })

MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {
    "model_path": str(MODEL_PATH),
    "config_sha256": small_file_sha256(config_path),
    "safetensors_index_sha256": small_file_sha256(index_path),
    "shard_layout": _model_shard_layout,
    "chat_template_sha256": template_sha,
    "tokenizer_class": tokenizer.__class__.__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
MODEL_ARTIFACT_FINGERPRINT_SHA256 = stable_json_sha256(MODEL_ARTIFACT_FINGERPRINT_PAYLOAD)

ENGINE_CONFIG = {
    "model": str(MODEL_PATH),
    "tensor_parallel_size": 4,
    "distributed_executor_backend": "mp",
    "dtype": "auto",
    "load_format": "safetensors",
    "trust_remote_code": False,
    "language_model_only": True,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "enforce_eager": False,
    "disable_custom_all_reduce": True,
    "enable_prefix_caching": bool(ENABLE_PREFIX_CACHING),
    "seed": SEED,
}
# C1 causal-regression firewall: compare against the frozen donor engine.
DONOR_ENGINE_CONFIG = dict(ENGINE_CONFIG)
DONOR_ENGINE_CONFIG["enforce_eager"] = True
_engine_diffs = {
    k: (DONOR_ENGINE_CONFIG.get(k), ENGINE_CONFIG.get(k))
    for k in sorted(set(DONOR_ENGINE_CONFIG) | set(ENGINE_CONFIG))
    if DONOR_ENGINE_CONFIG.get(k) != ENGINE_CONFIG.get(k)
}
assert _engine_diffs == {"enforce_eager": (True, False)}, _engine_diffs
assert ENGINE_CONFIG["disable_custom_all_reduce"] is True
assert ENGINE_CONFIG["enable_prefix_caching"] is False
assert "disable_log_stats" not in ENGINE_CONFIG  # throughput instrumentation is NOT copied into solver.
assert ENGINE_CONFIG["tensor_parallel_size"] == 4
assert ENGINE_CONFIG["max_num_seqs"] == 16
assert ENGINE_CONFIG["max_num_batched_tokens"] == 32768
assert ENGINE_CONFIG["max_model_len"] == 29696
assert abs(float(ENGINE_CONFIG["gpu_memory_utilization"]) - 0.90) < 1e-12
print("C1 causal engine assertion PASS:", _engine_diffs)
ENGINE_CONFIG_SHA256 = stable_json_sha256(ENGINE_CONFIG)

if not USE_MODEL:
    # True no-allocation path: package parity is real, but do not import vLLM/CUDA runtime.
    SamplingParams = None
    StructuredOutputsParams = None
    print("=" * 116)
    print("PREFLIGHT CPU-SAFE / DEPENDENCY-PARITY MODE")
    print("=" * 116)
    print("vLLM package metadata :", RUNTIME_VLLM_VERSION)
    print("Transformers          :", RUNTIME_TRANSFORMERS_VERSION)
    print("Tokenizers            :", RUNTIME_TOKENIZERS_VERSION)
    print("HF Hub                :", RUNTIME_HF_HUB_VERSION)
    print("Skipping vLLM import, CUDA checks, SamplingParams API check and 27B allocation.")
    print("GPU/runtime-specific paths remain covered by TERMINAL_PILOT.")
else:
    ENGINE_BOOTSTRAP_OK = False
    ENGINE_BOOTSTRAP_STAGE = "real_runtime_import"
    ENGINE_BOOTSTRAP_ERROR = None
    ENGINE_BOOTSTRAP_TRACEBACK = None
    _bootstrap_t0 = time.perf_counter()
    try:
        import torch
        import vllm
        from vllm import LLM, SamplingParams
        from vllm.sampling_params import StructuredOutputsParams
        RUNTIME_VLLM_VERSION = vllm.__version__
        RUNTIME_TORCH_VERSION = torch.__version__
        RUNTIME_CUDA_VERSION = torch.version.cuda
        print("=" * 116); print("REAL vLLM / CUDA PREFLIGHT"); print("=" * 116)
        print("PyTorch :", RUNTIME_TORCH_VERSION); print("CUDA    :", RUNTIME_CUDA_VERSION); print("vLLM    :", RUNTIME_VLLM_VERSION)
        print("Transformers:", pkg_version("transformers"))
        if RUNTIME_VLLM_VERSION != EXPECTED_RUNTIME_PACKAGES["vllm"]: raise RuntimeError("vLLM runtime drift after import.")
        if pkg_version("transformers") != EXPECTED_RUNTIME_PACKAGES["transformers"]: raise RuntimeError("Transformers runtime drift after vLLM setup.")
        ENGINE_BOOTSTRAP_STAGE = "sampling_api_probe"
        _sampling_api_probe = SamplingParams(temperature=1.0,top_p=0.95,top_k=20,min_p=0.0,presence_penalty=0.0,
                                             repetition_penalty=1.0,max_tokens=16,ignore_eos=False,seed=SEED,
                                             skip_special_tokens=False,spaces_between_special_tokens=False)
        _structured_api_probe = StructuredOutputsParams(json=FINALIZER_JSON_SCHEMA)
        _structured_sampling_probe = SamplingParams(
            temperature=FINALIZER_SAMPLING["temperature"], top_p=FINALIZER_SAMPLING["top_p"],
            top_k=FINALIZER_SAMPLING["top_k"], min_p=FINALIZER_SAMPLING["min_p"],
            presence_penalty=FINALIZER_SAMPLING["presence_penalty"],
            repetition_penalty=FINALIZER_SAMPLING["repetition_penalty"],
            max_tokens=64, structured_outputs=_structured_api_probe,
        )
        print("SamplingParams + StructuredOutputsParams API self-test: PASS")
        ENGINE_BOOTSTRAP_STAGE = "cuda_topology_check"
        print("\nLOADING QWEN3.8-27B-FP8 ON 4×L4"); print("GPUs visible:", torch.cuda.device_count())
        if not torch.cuda.is_available(): raise RuntimeError("CUDA unavailable. Select Kaggle 4×L4 accelerator.")
        if torch.cuda.device_count() != 4: raise RuntimeError(f"Expected 4 GPUs; PyTorch sees {torch.cuda.device_count()}.")
        for i in range(4):
            cc=torch.cuda.get_device_capability(i); mem=torch.cuda.get_device_properties(i).total_memory/1024**3; name=torch.cuda.get_device_name(i)
            gpu_info.append({"gpu":i,"name":name,"memory_gib":mem,"cc":f"{cc[0]}.{cc[1]}"}); print(f"GPU {i}: {name} | {mem:.2f} GiB | CC {cc[0]}.{cc[1]}")
        ENGINE_BOOTSTRAP_STAGE = "model_allocation"
        load_start=time.perf_counter(); arc_llm=LLM(**ENGINE_CONFIG); model_load_sec=time.perf_counter()-load_start
        print(f"MODEL LOAD COMPLETE: {model_load_sec:.1f} sec")
        ENGINE_BOOTSTRAP_STAGE = "warmup_generate"
        _active_prompts=[r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET]
        _med=float(np.median([r["prompt_tokens"] for r in _active_prompts])); probe_record=min(_active_prompts,key=lambda r:abs(r["prompt_tokens"]-_med))
        warm_params=SamplingParams(temperature=0.0,max_tokens=32,ignore_eos=False,skip_special_tokens=False)
        t0=time.perf_counter(); _probe_ids=tokenizer.encode(probe_record["prompt"],add_special_tokens=False)
        warm=arc_llm.generate([probe_record["prompt"],{"prompt_token_ids":list(_probe_ids)}],[warm_params,warm_params],use_tqdm=False)
        warmup_sec=time.perf_counter()-t0
        assert len(warm)==2 and all(len(w.outputs)>=1 for w in warm)
        if not list(getattr(warm[0],"prompt_token_ids",None) or []): raise RuntimeError("RequestOutput.prompt_token_ids unavailable in this vLLM build; token-exact continuation cannot run.")

        # v2.4 GPU-only, score-excluded terminal positive control: tests non-thinking/no-tools render,
        # structured-output backend, JSON parsing, and standalone solution() syntax on the real stack.
        if RUN_PROFILE in {"TERMINAL_PILOT","ADAPTIVE_SMOKE","EVAL120"}:
            ENGINE_BOOTSTRAP_STAGE="terminal_positive_control"
            _pc_messages=[
                {"role":"system","content":"Return only the requested structured final artifact."},
                {"role":"user","content":"Return a standalone identity ARC program defining solution(grid) that returns a copy of grid."},
            ]
            _pc_prompt,_pc_method=render_terminal_chat(_pc_messages)
            _pc_schema=StructuredOutputsParams(json=FINALIZER_JSON_SCHEMA)
            _pc_params=SamplingParams(
                temperature=FINALIZER_SAMPLING["temperature"], top_p=FINALIZER_SAMPLING["top_p"],
                top_k=FINALIZER_SAMPLING["top_k"], min_p=FINALIZER_SAMPLING["min_p"],
                presence_penalty=FINALIZER_SAMPLING["presence_penalty"],
                repetition_penalty=FINALIZER_SAMPLING["repetition_penalty"],
                max_tokens=256, ignore_eos=False, seed=SEED+9_000_000,
                skip_special_tokens=False, spaces_between_special_tokens=False,
                structured_outputs=_pc_schema,
            )
            _pc_t0=time.perf_counter()
            _pc_req=arc_llm.generate([_pc_prompt],[_pc_params],use_tqdm=False)[0]
            _pc_sec=time.perf_counter()-_pc_t0
            _pc_text=_pc_req.outputs[0].text if getattr(_pc_req,"outputs",None) else ""
            # v2.4B CRITICAL FIX: sampling uses skip_special_tokens=False, so the decoded text ends with
            # <|im_end|>. The old json.loads(_pc_text) therefore raised JSONDecodeError on every
            # structurally CORRECT answer, failing engine bootstrap and skipping all real generation.
            # Use the same hardened parser the scheduler uses, so the control tests the production path.
            _pc_parsed=parse_terminal_json_output(_pc_text)
            if not _pc_parsed["ok"]:
                raise RuntimeError(f"terminal positive control failed at stage {_pc_parsed['status']!r}; raw tail={_pc_text[-200:]!r}")
            _pc_code=_pc_parsed["solution_code"]
            _pc_verify_task={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]}],
                             "test":[{"input":[[5,6],[7,8]]}]}
            _pc_verify=fresh_verify_solution(_pc_code,_pc_verify_task,timeout_sec=min(5.0,VERIFIER_TIMEOUT_SEC))
            if not (_pc_verify.get("all_train_exact") and _pc_verify.get("test_execution_ok")):
                raise RuntimeError(f"terminal positive control fresh verifier failed: {_pc_verify}")
            TERMINAL_POSITIVE_CONTROL={
                "ran":True,"ok":True,"render_method":_pc_method,
                "prompt_sha256":hashlib.sha256(_pc_prompt.encode()).hexdigest(),
                "output_sha256":hashlib.sha256(_pc_text.encode()).hexdigest(),
                "output_tokens":len(getattr(_pc_req.outputs[0],"token_ids",[]) or []),
                "finish_reason":str(getattr(_pc_req.outputs[0],"finish_reason",None)),
                "elapsed_sec":float(_pc_sec),"score_excluded":True,
                "parse_status":_pc_parsed["status"],"fresh_verifier_status":_pc_verify.get("verifier_status"),
                "raw_text_had_control_tokens":bool(_pc_text!=strip_model_control_tokens(_pc_text)),
            }
            (WORK_ROOT/"terminal_positive_control.json").write_text(json.dumps(TERMINAL_POSITIVE_CONTROL,indent=2),encoding="utf-8")
            print("TERMINAL POSITIVE CONTROL PASS:",TERMINAL_POSITIVE_CONTROL)

        ENGINE_BOOTSTRAP_OK=True; ENGINE_BOOTSTRAP_STAGE="complete"
        print(f"REAL MODEL WARMUP (text + token-prompt continuation API + prompt_token_ids): PASS ({warmup_sec:.2f} sec)")
    except BaseException as _bootstrap_exc:
        ENGINE_BOOTSTRAP_ERROR=f"{type(_bootstrap_exc).__name__}: {_bootstrap_exc}"; ENGINE_BOOTSTRAP_TRACEBACK=traceback.format_exc(limit=40)
        _bootstrap_payload={"run_id":RUN_ID,"run_profile":RUN_PROFILE,"ok":False,"failed_stage":ENGINE_BOOTSTRAP_STAGE,
                            "error":ENGINE_BOOTSTRAP_ERROR,"traceback":ENGINE_BOOTSTRAP_TRACEBACK,
                            "elapsed_sec":float(time.perf_counter()-_bootstrap_t0),"engine_config_sha256":ENGINE_CONFIG_SHA256,
                            "model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256}
        (WORK_ROOT/"ENGINE_BOOTSTRAP_FAILURE.json").write_text(json.dumps(_bootstrap_payload,indent=2),encoding="utf-8")
        print("ENGINE BOOTSTRAP FAILURE CAPTURED:",ENGINE_BOOTSTRAP_ERROR)
        print("Downstream cells will write terminal infrastructure-failure records and still build the evidence bundle.")
        arc_llm=None

RUNTIME_STACK = {
    "vllm": pkg_version("vllm"),
    "torch": pkg_version("torch"),
    "transformers": pkg_version("transformers"),
    "tokenizers": pkg_version("tokenizers"),
    "huggingface-hub": pkg_version("huggingface-hub"),
    "cuda": RUNTIME_CUDA_VERSION,
}
RUNTIME_STACK_SHA256 = stable_json_sha256(RUNTIME_STACK)

print("Model artifact fingerprint:", MODEL_ARTIFACT_FINGERPRINT_SHA256)
print("Engine config fingerprint :", ENGINE_CONFIG_SHA256)
print("Runtime stack fingerprint :", RUNTIME_STACK_SHA256)
ENGINE_BOOTSTRAP_STATUS = {
    "run_id":RUN_ID,"run_profile":RUN_PROFILE,"use_model":USE_MODEL,"ok":bool(ENGINE_BOOTSTRAP_OK),
    "stage":ENGINE_BOOTSTRAP_STAGE,"error":ENGINE_BOOTSTRAP_ERROR,"model_load_sec":float(model_load_sec),
    "warmup_sec":float(warmup_sec),"gpu_info":gpu_info,"engine_config_sha256":ENGINE_CONFIG_SHA256,
    "runtime_stack_sha256":RUNTIME_STACK_SHA256,
}
(WORK_ROOT/"ENGINE_BOOTSTRAP_STATUS.json").write_text(json.dumps(ENGINE_BOOTSTRAP_STATUS,indent=2,default=str),encoding="utf-8")
print("Engine bootstrap status    :",{k:ENGINE_BOOTSTRAP_STATUS[k] for k in ("ok","stage","error")})
print("Safety reserve:", SAFETY_RESERVE_SEC / 60, "minutes")


In [ ]:
# =====================================================================================
# CELL 8 — EVO: EVOLUTION OVER VERIFIED PROGRAMS
#   rule -> solution() -> fresh verification on every training pair -> error map -> mutate
# (Imbue-style code evolution, PiVOTools/Agentica-style verification; replaces the single long REPL chat)
# =====================================================================================
import concurrent.futures as _evo_futures
import zlib as _evo_zlib

EVO_CONFIG = {
    "organisms": 4,               # independent initial candidates per task
    "parents": 2,                 # distinct parents carried into each repair generation
    "repair_generations": 2,
    "reasoning_effort": "medium",
    "think_budget": 3072,         # phase 1: thinking, stops at </think> or this many tokens
    "answer_max_tokens": 2048,    # phase 2: rule + code
    "task_token_cap": 32768,      # aggregate generated tokens per task (all calls)
    "temperature": 1.0, "top_p": 0.95, "top_k": 20, "min_p": 0.0,
    "max_grid_cells_shown": 400,  # predicted grids shown in repair feedback only up to this size
    "verify_workers": 8,
}
EVO_CALL_MAX = EVO_CONFIG["think_budget"] + EVO_CONFIG["answer_max_tokens"]
EVO_SYSTEM = ("ARC-EVO v1. You are an expert at solving ARC-AGI abstraction-and-reasoning puzzles by writing short, "
              "correct Python programs.")
EVO_LENSES = [
    "",
    "Hint for this attempt: think in terms of objects (connected components): their shapes, colors, sizes, positions, "
    "and how each one moves, changes or interacts.",
    "Hint for this attempt: think in terms of geometry and layout: rows, columns, symmetry, rotation or reflection, "
    "tiling, cropping, scaling, lines and borders.",
    "Hint for this attempt: think in terms of colors and counts: which colors map to which, what is counted, and what "
    "is drawn where because of it.",
]
EVO_FORMAT = (
    "Answer in exactly this format:\n"
    "## Rule\n<the transformation in a few plain sentences>\n"
    "## Code\n```python\ndef solution(grid):\n    ...\n```\n\n"
    "Code rules: `grid` is a list of lists of ints 0-9; return a NEW list of lists of ints 0-9. The standard library "
    "and numpy may be used. These helpers are preloaded and may be called inside solution (do not redefine them): "
    "`components(grid, color=None, diag=False, background=None)` -> list of dicts {color, cells, bbox=(r0,c0,r1,c1), size}; "
    "`bbox(grid, ignore=None)` -> (r0,c0,r1,c1) or None; `crop(grid, r0, c0, r1, c1)` (inclusive); `rot90(grid)` "
    "(clockwise); `flip_h(grid)`; `flip_v(grid)`; `transpose(grid)`; `color_counts(grid)` -> dict; "
    "`background_color(grid)` (most common colour: a heuristic, pass background= explicitly when unsure); "
    "`grid_shape(grid)` -> (rows, cols); `diff_cells(a, b)`. No printing, no files, no network. Everything else the "
    "program uses must be defined in the code block. It must work for every training input and the test input(s)."
)

def _evo_grid_text(g):
    return "\n".join(" ".join(str(int(v)) for v in row) for row in g)

def evo_initial_prompt(task, organism):
    lens = EVO_LENSES[organism % len(EVO_LENSES)]
    return (puzzle_to_source_text(task_without_test_solutions(task)) + "\n\n"
            "Find the transformation that maps every training input to its output; the same transformation must "
            "produce the test output(s).\n" + (lens + "\n" if lens else "") + "\n" + EVO_FORMAT)

def _evo_failure_grids(verifier, task, limit_pairs=2):
    out = []
    by_index = {int(x.get("index", i)): x for i, x in enumerate(list((verifier or {}).get("train") or []))}
    for i, ex in enumerate(task["train"]):
        e = by_index.get(i) or {}
        pred = e.get("pred")
        if e.get("exact") or pred is None:
            continue
        exp = ex["output"]
        if len(pred) * len(pred[0]) > EVO_CONFIG["max_grid_cells_shown"]:
            continue
        same = len(pred) == len(exp) and all(len(r) == len(exp[0]) for r in pred)
        if same:
            rows = [" ".join((str(p) if p == q else f"{p}*") for p, q in zip(pr, er)) for pr, er in zip(pred, exp)]
            out.append(f"Your output for training example {i} (cells marked * are wrong):\n" + "\n".join(rows))
        else:
            out.append(f"Your output for training example {i} (wrong size {len(pred)}x{len(pred[0])}, expected "
                       f"{len(exp)}x{len(exp[0])}):\n" + _evo_grid_text(pred))
        if len(out) >= limit_pairs:
            break
    return "\n\n".join(out)

def evo_repair_prompt(task, parent, with_grids=True):
    report = _v3_report_text(parent["verifier"], task) if parent.get("verifier") else "The previous code could not be run."
    grids = _evo_failure_grids(parent.get("verifier"), task) if with_grids else ""
    return (puzzle_to_source_text(task_without_test_solutions(task)) + "\n\n"
            "A previous attempt at this puzzle:\n## Rule\n" + (parent.get("rule") or "(no rule given)") + "\n## Code\n"
            "```python\n" + strip_library(parent.get("code") or "") + "\n```\n\n"
            "Result of running this code on every training example in a fresh process:\n" + report +
            ("\n\n" + grids if grids else "") + "\n\n"
            "Improve it. Fix the code if it does not implement the rule correctly; if the rule itself is wrong, replace "
            "it with a better rule that explains ALL training examples.\n\n" + EVO_FORMAT)

def render_evo_chat(user_text):
    msgs = [{"role": "system", "content": EVO_SYSTEM}, {"role": "user", "content": user_text}]
    kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=True,
              reasoning_effort=EVO_CONFIG["reasoning_effort"])
    return tokenizer.apply_chat_template(msgs, **kw)

_EVO_THINK_END_ID = tokenizer.encode("</think>", add_special_tokens=False)
assert len(_EVO_THINK_END_ID) == 1, _EVO_THINK_END_ID
_EVO_THINK_END_ID = _EVO_THINK_END_ID[0]
_EVO_FORCE_IDS = list(tokenizer.encode(STOP_THINKING_TEXT, add_special_tokens=False))
_EVO_AFTER_THINK_IDS = list(tokenizer.encode("\n\n", add_special_tokens=False))
_EVO_RULE_RE = re.compile(r"##\s*Rule\s*\n(.*?)(?:\n##\s*Code|```)", re.S | re.I)
_EVO_EOS_IDS = {e[0] for e in (tokenizer.encode(x, add_special_tokens=False) for x in ("<|im_end|>", "<|endoftext|>")) if len(e) == 1}
assert _EVO_EOS_IDS, "EOS tokens must encode to single ids"

def evo_alternative_prompt(task, exact):
    """Second pass@2 slot: a candidate already fits every training pair; ask for a genuinely different reading."""
    return (puzzle_to_source_text(task_without_test_solutions(task)) + "\n\n"
            "This program reproduces EVERY training example exactly:\n## Rule\n" + (exact.get("rule") or "(no rule given)") +
            "\n## Code\n```python\n" + strip_library(exact.get("code") or "") + "\n```\n\n"
            "Its output for the test input(s) is not known to be correct. Find a genuinely different transformation that "
            "ALSO fits every training example but could produce a different test output (a different reading of the "
            "examples). Do not merely rewrite the same idea.\n\n" + EVO_FORMAT)

def evo_fresh_prompt(task, k):
    """Backfill when a task has fewer usable parents than slots: a fresh attempt with a rotated lens."""
    return (evo_initial_prompt(task, k) + "\n\nEarlier attempts at this puzzle did not produce a usable program. Try a "
            "substantially different hypothesis.")

def _evo_seed(tid, round_no, slot):
    return int(_evo_zlib.crc32(f"{SEED}|{tid}|{round_no}|{slot}".encode()) & 0x7FFFFFFF)

def _evo_sp(max_tokens, seed, stop_ids=None):
    kw = dict(temperature=EVO_CONFIG["temperature"], top_p=EVO_CONFIG["top_p"], top_k=EVO_CONFIG["top_k"],
              min_p=EVO_CONFIG["min_p"], max_tokens=int(max_tokens), seed=int(seed), skip_special_tokens=False)
    if stop_ids:
        kw["stop_token_ids"] = list(stop_ids)
    return SamplingParams(**kw)

def _evo_close_phase1(ids, text, stop_reason):
    """Continuation ids after phase 1 and whether thinking closed naturally. Only the </think> token (in the ids,
    as stop_reason, or in the text) counts as a natural close; EOS or the length limit inside thinking is a forced
    close with Qwen's documented stop-thinking text."""
    ids = list(ids)
    if _EVO_THINK_END_ID in ids:
        return ids[:ids.index(_EVO_THINK_END_ID) + 1] + _EVO_AFTER_THINK_IDS, True
    if stop_reason == _EVO_THINK_END_ID or "</think>" in (text or ""):
        return ids + [_EVO_THINK_END_ID] + _EVO_AFTER_THINK_IDS, True      # stopped on </think>, token not echoed
    while ids and ids[-1] in _EVO_EOS_IDS:
        ids = ids[:-1]
    return ids + _EVO_FORCE_IDS, False

EVO_ENGINE_FAILURES = []

def _evo_safe_generate(prompts, params, phase):
    """vLLM generate that never raises: an exception or a missing/short output list is recorded and returns None,
    so the run stops as INVALID / ENGINE FAILURE with its report and bundle instead of crashing the notebook."""
    try:
        out = arc_llm.generate(prompts, params, use_tqdm=False)
    except Exception as exc:
        EVO_ENGINE_FAILURES.append({"phase": phase, "type": type(exc).__name__, "error": str(exc)[-2000:],
                                    "requests": len(prompts)})
        return None
    out = list(out or [])
    if len(out) != len(prompts) or any(not getattr(x, "outputs", None) for x in out):
        EVO_ENGINE_FAILURES.append({"phase": phase, "type": "output_cardinality", "expected": len(prompts),
                                    "received": len(out)})
        return None
    return out

def evo_generate(jobs):
    """Two-phase generation for a batch of jobs. Phase 1: think until </think> or think_budget. Phase 2: token-exact
    continuation (forced close when needed), then the answer. Fills job['answer'], ['tokens'], ['think_closed'].
    Returns False on an engine failure (recorded in EVO_ENGINE_FAILURES)."""
    if not jobs:
        return True
    p1 = _evo_safe_generate([{"prompt_token_ids": j["prompt_ids"]} for j in jobs],
                            [_evo_sp(EVO_CONFIG["think_budget"], j["seed"], [_EVO_THINK_END_ID]) for j in jobs], "phase1")
    if p1 is None:
        return False
    cont = []
    for j, r in zip(jobs, p1):
        o = r.outputs[0]
        ids, closed = _evo_close_phase1(o.token_ids, o.text, getattr(o, "stop_reason", None))
        j["think_tokens"] = len(o.token_ids); j["think_closed"] = bool(closed); j["think_finish"] = o.finish_reason
        cont.append(j["prompt_ids"] + ids)
    p2 = _evo_safe_generate([{"prompt_token_ids": c} for c in cont],
                            [_evo_sp(EVO_CONFIG["answer_max_tokens"], j["seed"] + 1) for j in jobs], "phase2")
    if p2 is None:
        return False
    for j, r in zip(jobs, p2):
        o = r.outputs[0]
        j["answer"] = o.text or ""
        j["answer_tokens"] = len(o.token_ids); j["answer_finish"] = o.finish_reason
        j["tokens"] = int(j["think_tokens"] + j["answer_tokens"])
    return True

def _evo_parse(answer):
    code = extract_last_solution_code(answer)
    m = _EVO_RULE_RE.search(answer or "")
    rule = (m.group(1).strip() if m else "")[:1500]
    return rule, code

def _evo_train_signature(verifier):
    return json.dumps([(x.get("exact"), x.get("pred")) for x in list((verifier or {}).get("train") or [])])[:20000]

def evo_identity_wrong_frac(task):
    """Wrong-cell fraction of the identity program (return the input), counted like _v3_train_quality: a wrong-shaped
    pair counts every expected cell wrong. The reference every near-miss claim is measured against (S12 lesson)."""
    wrong = total = 0
    for ex in task["train"]:
        a, b = ex["input"], ex["output"]
        cells = len(b) * len(b[0]); total += cells
        if len(a) == len(b) and all(len(r) == len(b[0]) for r in a):
            wrong += sum(x != y for ra, rb in zip(a, b) for x, y in zip(ra, rb))
        else:
            wrong += cells
    return round(wrong / max(total, 1), 6)

EVO_INFRA_FAILURES = []
EVO_ROUND0_MIN_TOK_PER_SEC = 60.0   # assumed aggregate throughput before anything is measured (baseline 65 tok/s)
EVO_CHUNK_CALLS = 128               # a round is generated in chunks of this many calls; deadline checked before each
EVO_SUBMISSION_PATH = Path("/kaggle/working/submission.json")

def _evo_valid_grid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30 and all(isinstance(r, list) and len(r) == len(g[0]) for r in g)
            and 1 <= len(g[0]) <= 30 and all(isinstance(v, int) and not isinstance(v, bool) and 0 <= v <= 9 for r in g for v in r))

def evo_build_submission(task_ids):
    """Competition format: {task_id: [{"attempt_1": grid, "attempt_2": grid} per test query]}. Attempts come from
    evo_attempts_by_query; a query without an attempt gets the test input as fallback."""
    sub = {}
    for tid in task_ids:
        task = eval_challenges[tid]
        n = len(task["test"])
        att = evo_attempts_by_query(EVO_STATE[tid]["cands"], n) if tid in EVO_STATE else [[] for _ in range(n)]
        rows = []
        for i in range(n):
            fb = [list(r) for r in task["test"][i]["input"]]
            a = [g for g in (att[i] if i < len(att) else []) if _evo_valid_grid(g)]
            rows.append({"attempt_1": a[0] if a else fb, "attempt_2": a[1] if len(a) > 1 else fb})
        sub[tid] = rows
    return sub

def evo_write_submission(task_ids):
    sub = evo_build_submission(task_ids)
    assert set(sub) == set(task_ids)
    assert all(_evo_valid_grid(x["attempt_1"]) and _evo_valid_grid(x["attempt_2"]) for rows in sub.values() for x in rows)
    EVO_SUBMISSION_PATH.write_text(json.dumps(sub), encoding="utf-8")
    return sub

def evo_submission_task_ids():
    # rerun: EVERY hidden test task (also tasks dropped by the context gate); evaluation: the frozen eval subset
    return sorted(test_challenges) if EVO_RERUN else list(EVO_TASKS)

def _evo_verify(cand, task):
    try:
        return _evo_verify_inner(cand, task)
    except Exception as exc:
        try:
            ident = evo_identity_wrong_frac(task); qual = list(_v3_train_quality(None, task))
        except Exception:
            ident = 1.0; qual = list(_v3_train_quality(None, {"train": []}))
        cand["verifier"] = None; cand["quality"] = qual; cand["infra_failure"] = True
        cand["wrong_frac"] = 1.0; cand["identity_wrong_frac"] = ident
        cand["gain_vs_identity"] = round(ident - 1.0, 6)
        EVO_INFRA_FAILURES.append({"cand_id": cand.get("cand_id"), "errors": [f"verify exception {type(exc).__name__}: {exc}"[:500]]})
        return cand

def _evo_verify_inner(cand, task):
    ident = evo_identity_wrong_frac(task)
    if not cand.get("code"):
        cand["verifier"] = None
        cand["quality"] = list(_v3_train_quality(None, task))
    else:
        v = fresh_verify_solution(cand["code"], task, VERIFIER_TIMEOUT_SEC)
        cand["verifier"] = v
        cand["quality"] = list(_v3_train_quality(v, task))
        cand["solved_train"] = bool(v.get("all_train_exact") and v.get("test_execution_ok"))
        if v.get("infrastructure_failure"):
            cand["infra_failure"] = True
            EVO_INFRA_FAILURES.append({"cand_id": cand.get("cand_id"), "errors": v.get("infrastructure_errors")})
    cand["wrong_frac"] = round(-float(cand["quality"][6]), 6)
    cand["identity_wrong_frac"] = ident
    cand["gain_vs_identity"] = round(ident - cand["wrong_frac"], 6)
    return cand

def _evo_rank_key(c):
    """(train-exact and test runs, train-exact, exact pairs, right-shaped pairs, gain over the identity program,
    test runs, pairs without exception, -wrong-cell fraction, shorter code)."""
    q = c["quality"]
    return (q[0], q[1], q[2], q[3], c.get("gain_vs_identity", 0.0), q[4], q[5], q[6], -len(c.get("code") or ""))

def _evo_select_parents(cands, k, exclude=()):
    out, seen = [], set()
    for c in sorted(cands, key=_evo_rank_key, reverse=True):
        if not c.get("code") or c.get("cand_id") in exclude:
            continue
        sig = _evo_train_signature(c.get("verifier"))
        if sig in seen:
            continue
        seen.add(sig); out.append(c)
        if len(out) >= k:
            break
    return out

def _evo_exact_cands(cands):
    """Train-exact candidates, best first (ranking uses training evidence only, never test labels)."""
    return sorted([c for c in cands if (c.get("verifier") or {}).get("all_train_exact")], key=_evo_rank_key, reverse=True)

def evo_attempts_by_query(cands, n_queries):
    """Canonical pass@2, per test query: for each query, up to two distinct non-None outputs taken from train-exact
    candidates in rank order. A program that fails on one query still contributes its valid outputs on the others."""
    out = [[] for _ in range(int(n_queries))]
    for c in _evo_exact_cands(cands):
        outs = list((c.get("verifier") or {}).get("test_outputs") or [])
        for i in range(int(n_queries)):
            g = outs[i] if i < len(outs) else None
            if g is not None and len(out[i]) < 2 and g not in out[i]:
                out[i].append(g)
    return out

def evo_task_complete(cands, n_queries):
    """Search for a task stops only when every test query has two distinct attempts."""
    return all(len(a) >= 2 for a in evo_attempts_by_query(cands, n_queries))

def evo_verdict(solved, evo_only, stop_reason, infra_failures):
    """Pre-registered verdict. A run that did not execute as planned never yields a model verdict."""
    if infra_failures:
        return "INVALID / VERIFIER INFRASTRUCTURE FAILURE"
    if stop_reason == "no_model":
        return "INVALID / NO MODEL"
    if stop_reason == "engine_error":
        return "INVALID / ENGINE FAILURE"
    if stop_reason == "evo_exception":
        return "INVALID / EVO EXCEPTION"
    if stop_reason == "deadline":
        return "INCOMPLETE / NO DECISION"
    if stop_reason == "interface_code_extraction":
        return "WEAK / REDESIGN - INTERFACE FAILURE"
    if solved >= 3 and evo_only >= 2:          # 12-task run (9 NVARC misses): pre-registered before the run
        return "STRONG GO"
    if evo_only >= 1:
        return "PROMISING"
    return "WEAK / REDESIGN"

EVO_RECORDS_PATH = WORK_ROOT / "evo_records.jsonl"
EVO_TASKS = list(ACTIVE_TASK_IDS)
EVO_STATE = {t: {"cands": [], "tokens": 0, "calls": 0, "skipped_calls": 0, "backfills": 0, "alternatives": 0,
                 "repair_parents_nonpositive_gain": 0, "exact_partial_repairs": 0}
             for t in EVO_TASKS}
EVO_ROUNDS = []

def _evo_log(rec):
    with open(EVO_RECORDS_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")

def evo_run_round(round_no, plan):
    """plan: list of (tid, slot, kind, prompt_text, parent_or_None). Generates, verifies and records one round.
    The per-task token cap is a hard ceiling: every admitted call reserves its maximum size."""
    t0 = time.perf_counter()
    jobs, reserved = [], {}
    for tid, slot, kind, text, parent in plan:
        st = EVO_STATE[tid]
        if st["tokens"] + reserved.get(tid, 0) + EVO_CALL_MAX > EVO_CONFIG["task_token_cap"]:
            st["skipped_calls"] += 1
            continue
        rendered = render_evo_chat(text)
        ids = list(tokenizer.encode(rendered, add_special_tokens=False))
        if len(ids) + EVO_CALL_MAX + CONTEXT_SAFETY_TOKENS > MAX_MODEL_LEN and kind == "repair":
            rendered = render_evo_chat(evo_repair_prompt(eval_challenges[tid], parent, with_grids=False))
            ids = list(tokenizer.encode(rendered, add_special_tokens=False))
        if len(ids) + EVO_CALL_MAX + CONTEXT_SAFETY_TOKENS > MAX_MODEL_LEN:
            st["skipped_calls"] += 1
            _evo_log({"run_id": RUN_ID, "task_id": tid, "round": round_no, "slot": slot, "skipped": "context"})
            continue
        reserved[tid] = reserved.get(tid, 0) + EVO_CALL_MAX
        jobs.append({"task_id": tid, "slot": slot, "kind": kind, "round": round_no, "prompt_ids": ids,
                     "prompt_tokens": len(ids), "seed": _evo_seed(tid, round_no, slot),
                     "parent_id": (parent or {}).get("cand_id")})
    gen_t0 = time.perf_counter()
    if not evo_generate(jobs):
        for j in jobs:
            _evo_log({"run_id": RUN_ID, "task_id": j["task_id"], "round": round_no, "slot": j["slot"],
                      "kind": j["kind"], "engine_failure": True})
        info = {"round": round_no, "calls": len(jobs), "engine_failure": EVO_ENGINE_FAILURES[-1],
                "gen_sec": round(time.perf_counter() - gen_t0, 1), "round_sec": round(time.perf_counter() - t0, 1)}
        EVO_ROUNDS.append(info)
        print(f"[EVO] round {round_no}: ENGINE FAILURE", json.dumps(info, default=str))
        return info
    gen_sec = time.perf_counter() - gen_t0
    cands = []
    for j in jobs:
        rule, code = _evo_parse(j["answer"])
        cands.append({"cand_id": f"{j['task_id']}:r{round_no}:s{j['slot']}", "task_id": j["task_id"], "round": round_no,
                      "slot": j["slot"], "kind": j["kind"], "parent_id": j["parent_id"], "rule": rule, "code": code,
                      "tokens": j["tokens"], "think_tokens": j["think_tokens"], "think_closed": j["think_closed"],
                      "think_finish": j["think_finish"], "answer_tokens": j["answer_tokens"],
                      "answer_finish": j["answer_finish"], "prompt_tokens": j["prompt_tokens"],
                      "answer_tail": j["answer"][-600:]})
    ver_t0 = time.perf_counter()
    with _evo_futures.ThreadPoolExecutor(max_workers=EVO_CONFIG["verify_workers"]) as ex:
        cands = list(ex.map(lambda c: _evo_verify(c, eval_challenges[c["task_id"]]), cands))
    ver_sec = time.perf_counter() - ver_t0
    for c in cands:
        st = EVO_STATE[c["task_id"]]
        st["cands"].append(c); st["tokens"] += int(c["tokens"]); st["calls"] += 1
        v = c.get("verifier") or {}
        _evo_log({"run_id": RUN_ID, **{k: c[k] for k in c if k not in ("verifier",)},
                  "verifier_status": v.get("verifier_status"), "all_train_exact": bool(v.get("all_train_exact")),
                  "test_execution_ok": bool(v.get("test_execution_ok")), "train_exact_pairs": _v25_train_exact_count(v),
                  "infrastructure_failure": bool(v.get("infrastructure_failure")),
                  "test_outputs": v.get("test_outputs") if v.get("all_train_exact") else None})
    gen_tokens = sum(int(j["tokens"]) for j in jobs)
    info = {"round": round_no, "calls": len(jobs), "kinds": dict(collections.Counter(j["kind"] for j in jobs)),
            "gen_sec": round(gen_sec, 1), "verify_sec": round(ver_sec, 1),
            "round_sec": round(time.perf_counter() - t0, 1), "generated_tokens": gen_tokens,
            "tok_per_sec": round(gen_tokens / max(gen_sec, 1e-6), 1),
            "code_extracted": sum(1 for c in cands if c.get("code")),
            "think_closed": sum(1 for c in cands if c.get("think_closed")),
            "train_exact": sum(1 for c in cands if (c.get("verifier") or {}).get("all_train_exact")),
            "solved_train": sum(1 for c in cands if c.get("solved_train")),
            "infra_failures_total": len(EVO_INFRA_FAILURES),
            "tasks_all_queries_attempted": sum(1 for t in EVO_TASKS if all(evo_attempts_by_query(EVO_STATE[t]["cands"], len(eval_challenges[t]["test"])))),
            "tasks_complete_2_per_query": sum(1 for t in EVO_TASKS if evo_task_complete(EVO_STATE[t]["cands"], len(eval_challenges[t]["test"])))}
    EVO_ROUNDS.append(info)
    print(f"[EVO] round {round_no}:", json.dumps(info))
    return info

def evo_run_round_chunked(round_no, plan):
    """Run one round in chunks of EVO_CHUNK_CALLS. Before each chunk, project its duration from the throughput
    measured so far (1.25x margin; before any measurement: maximum call size at EVO_ROUND0_MIN_TOK_PER_SEC) and stop
    cleanly if it would cross the deadline. Unfinished tasks keep fallback answers in submission.json."""
    agg = {"round": round_no, "calls": 0, "code_extracted": 0, "deadline_stop": False, "engine_failure": False, "chunks": 0}
    for k in range(0, len(plan), EVO_CHUNK_CALLS):
        chunk = plan[k:k + EVO_CHUNK_CALLS]
        done = [i for i in EVO_ROUNDS if "generated_tokens" in i and i.get("calls")]
        if done:
            rate = sum(i["generated_tokens"] for i in done) / max(sum(i["gen_sec"] for i in done), 1e-6)
            per_call = sum(i["generated_tokens"] for i in done) / max(sum(i["calls"] for i in done), 1)
            verify = sum(i["verify_sec"] for i in done) / max(sum(i["calls"] for i in done), 1)
            est = 1.25 * len(chunk) * (per_call / max(rate, 1e-6) + verify)
        else:
            est = len(chunk) * EVO_CALL_MAX / EVO_ROUND0_MIN_TOK_PER_SEC
        if time.perf_counter() + est > deadline:   # deadline is on the perf_counter clock (cell 7)
            print(f"[EVO] STOP before round {round_no} chunk {k // EVO_CHUNK_CALLS + 1}: projected {est:.0f}s would cross the deadline.")
            agg["deadline_stop"] = True
            return agg
        info = evo_run_round(round_no, chunk)
        agg["chunks"] += 1; agg["calls"] += int(info.get("calls") or 0); agg["code_extracted"] += int(info.get("code_extracted") or 0)
        if info.get("engine_failure"):
            agg["engine_failure"] = True
            return agg
        try:
            evo_write_submission(evo_submission_task_ids())   # keep submission.json current after every chunk
        except Exception as exc:
            print("[EVO] submission refresh failed (kept previous file):", type(exc).__name__, exc)
    return agg

def evo_plan_generation(g):
    """Two slots per unfinished task. A task is finished only with two distinct pass@2 outputs. With one: slot 0
    asks for a genuinely different program, slot 1 repairs the best other candidate. Slots without a usable parent
    are backfilled with fresh attempts, so a task whose first answers had no code is not abandoned."""
    plan = []
    k = EVO_CONFIG["parents"]
    for t in EVO_TASKS:
        st = EVO_STATE[t]; task = eval_challenges[t]
        good = _evo_exact_cands(st["cands"])
        if evo_task_complete(st["cands"], len(task["test"])):
            continue
        slots = []
        if good:
            slots.append(("alternative", evo_alternative_prompt(task, good[0]), good[0]))
            st["alternatives"] += 1
            partial = [c for c in good if not (c.get("verifier") or {}).get("test_execution_ok")]
            if partial:   # train-exact but fails on a test input: repair it with the TEST INPUT FAILURE report
                slots.append(("repair", evo_repair_prompt(task, partial[0]), partial[0]))
                st["exact_partial_repairs"] += 1
                parents = []
            else:
                parents = _evo_select_parents(st["cands"], k - 1, exclude={c["cand_id"] for c in good})
        else:
            parents = _evo_select_parents(st["cands"], k)
        slots += [("repair", evo_repair_prompt(task, p), p) for p in parents]
        st["repair_parents_nonpositive_gain"] += sum(1 for p in parents if p.get("gain_vs_identity", 0.0) <= 0 and p["quality"][2] == 0)
        while len(slots) < k:
            slots.append(("fresh", evo_fresh_prompt(task, g * k + len(slots)), None))
            st["backfills"] += 1
        plan += [(t, s, kind, text, par) for s, (kind, text, par) in enumerate(slots)]
    return plan

def run_evo():
    EVO_RECORDS_PATH.write_text("", encoding="utf-8")
    loop_t0 = time.perf_counter()
    plan = [(t, s, "initial", evo_initial_prompt(eval_challenges[t], s), None)
            for t in EVO_TASKS for s in range(EVO_CONFIG["organisms"])]
    r0 = evo_run_round_chunked(0, plan)
    if r0["deadline_stop"]:
        return {"stopped": "deadline", "loop_sec": time.perf_counter() - loop_t0}
    if EVO_ENGINE_FAILURES:
        print("[EVO] STOP: engine failure; the run is invalid as a measurement.", EVO_ENGINE_FAILURES[-1])
        return {"stopped": "engine_error", "loop_sec": time.perf_counter() - loop_t0}
    if EVO_INFRA_FAILURES:
        print("[EVO] STOP: verifier infrastructure failure; the run is invalid as a measurement.", EVO_INFRA_FAILURES[:3])
        return {"stopped": "verifier_infrastructure", "loop_sec": time.perf_counter() - loop_t0}
    if r0["calls"] and r0["code_extracted"] < 0.25 * r0["calls"]:
        print("[EVO] STOP: fewer than 25 % of round-0 answers contained a solution(); interface problem, no repairs.")
        return {"stopped": "interface_code_extraction", "loop_sec": time.perf_counter() - loop_t0}
    for g in range(1, EVO_CONFIG["repair_generations"] + 1):
        plan = evo_plan_generation(g)
        if not plan:
            break
        rg = evo_run_round_chunked(g, plan)
        if rg["deadline_stop"]:
            return {"stopped": "deadline", "loop_sec": time.perf_counter() - loop_t0}
        if EVO_ENGINE_FAILURES:
            print("[EVO] STOP: engine failure; the run is invalid as a measurement.", EVO_ENGINE_FAILURES[-1])
            return {"stopped": "engine_error", "loop_sec": time.perf_counter() - loop_t0}
        if EVO_INFRA_FAILURES:
            print("[EVO] STOP: verifier infrastructure failure; the run is invalid as a measurement.", EVO_INFRA_FAILURES[:3])
            return {"stopped": "verifier_infrastructure", "loop_sec": time.perf_counter() - loop_t0}
    return {"stopped": None, "loop_sec": time.perf_counter() - loop_t0}

def run_evo_selftests():
    _t = {"train": [{"input": [[1, 0], [0, 0]], "output": [[1]]}, {"input": [[0, 0], [0, 7]], "output": [[7]]}],
          "test": [{"input": [[0, 3], [0, 0]]}]}
    ans = "## Rule\nCrop to the single coloured cell.\n## Code\n```python\ndef solution(grid):\n    return crop(grid, *bbox(grid, ignore=0))\n```"
    rule, code = _evo_parse(ans)
    assert rule.startswith("Crop") and code and "def solution" in code, (rule, code)
    c = _evo_verify({"code": code, "cand_id": "c"}, _t)
    assert c["solved_train"] and c["verifier"]["test_outputs"] == [[[3]]] and not c.get("infra_failure"), c
    bad = _evo_verify({"code": "def solution(grid):\n    return [row[:] for row in grid]", "cand_id": "b"}, _t)
    assert not bad.get("solved_train") and _evo_rank_key(c) > _evo_rank_key(bad)
    none = _evo_verify({"code": None, "cand_id": "n"}, _t)
    assert _evo_rank_key(bad) >= _evo_rank_key(none)
    rp = evo_repair_prompt(_t, {**bad, "rule": "copy"})
    assert "wrong size 2x2, expected 1x1" in rp and "## Rule\ncopy" in rp and "ARC-EVO" not in rp, rp[-800:]
    ap = evo_alternative_prompt(_t, {**c, "rule": "crop"})
    assert "genuinely different transformation" in ap and "crop(grid" in ap
    assert "substantially different hypothesis" in evo_fresh_prompt(_t, 5)
    assert len(_evo_select_parents([c, bad, none, dict(bad)], 2)) == 2
    assert _evo_select_parents([c, bad], 1, exclude={"c"})[0]["cand_id"] == "b"
    assert evo_attempts_by_query([c, dict(c)], 1) == [[[[3]]]]     # identical outputs are not used twice
    c2 = _evo_verify({"code": "def solution(grid):\n    return [[max(max(r) for r in grid)]]", "cand_id": "c2"}, _t)
    assert evo_attempts_by_query([c, c2], 1) == [[[[3]]]] and not evo_task_complete([c, c2], 1)
    # review R2: per-query pass@2. A:(A0,X) > B:(B0,X) > C:(B0,Y); truth (B0,Y) is reachable only per query.
    _q = [True, True, 3, 3, True, 3, -0.0]
    _mk = lambda cid, outs, code: {"cand_id": cid, "code": code, "quality": list(_q), "gain_vs_identity": 0.5,
                                   "verifier": {"all_train_exact": True, "test_outputs": outs}}
    A0, B0, X, Y = [[1]], [[2]], [[3]], [[4]]
    _abc = [_mk("A", [A0, X], "a"), _mk("B", [B0, X], "bb"), _mk("C", [B0, Y], "ccc")]
    _att = evo_attempts_by_query(_abc, 2)
    assert _att == [[A0, B0], [X, Y]], _att
    assert all(any(g == gt for g in _att[i]) for i, gt in enumerate([B0, Y]))     # strict under canonical pass@2
    assert not evo_task_complete(_abc[:2], 2) and evo_task_complete(_abc, 2)
    _part = [_mk("P", [A0, None], "p"), _mk("Q", [None, Y], "qq")]
    assert evo_attempts_by_query(_part, 2) == [[A0], [Y]]                          # partial outputs are kept
    # identity-relative ranking (S12 lesson): a candidate worse than returning the input has negative gain
    _ti = {"train": [{"input": [[1, 1, 1, 1], [1, 1, 1, 1]], "output": [[1, 1, 1, 1], [1, 1, 1, 2]]}], "test": [{"input": [[1]]}]}
    assert abs(evo_identity_wrong_frac(_ti) - 0.125) < 1e-9
    worse = _evo_verify({"code": "def solution(grid):\n    return [[1, 1, 1, 1], [1, 1, 2, 3]]", "cand_id": "w"}, _ti)
    near = _evo_verify({"code": "def solution(grid):\n    return [[1, 1, 1, 1], [1, 1, 1, 3]]", "cand_id": "m"}, _ti)
    assert worse["gain_vs_identity"] < 0 and near["gain_vs_identity"] == 0.0 and _evo_rank_key(near) > _evo_rank_key(worse)
    # phase-1 close: </think> in ids / as stop_reason / EOS inside thinking / length
    i1, k1 = _evo_close_phase1([5, _EVO_THINK_END_ID, 9], "", None)
    assert k1 and i1 == [5, _EVO_THINK_END_ID] + _EVO_AFTER_THINK_IDS
    i2, k2 = _evo_close_phase1([5], "x", _EVO_THINK_END_ID)
    assert k2 and i2 == [5, _EVO_THINK_END_ID] + _EVO_AFTER_THINK_IDS
    _eos = sorted(_EVO_EOS_IDS)[0]
    i3, k3 = _evo_close_phase1([5, _eos], "x", None)
    assert not k3 and i3 == [5] + _EVO_FORCE_IDS
    i4, k4 = _evo_close_phase1([5, 6], "x", None)
    assert not k4 and i4 == [5, 6] + _EVO_FORCE_IDS
    # verdicts: an incomplete or invalid run never yields a model verdict
    assert evo_verdict(9, 9, "deadline", []) == "INCOMPLETE / NO DECISION"
    assert evo_verdict(9, 9, None, [{"x": 1}]).startswith("INVALID")
    assert evo_verdict(3, 2, None, []) == "STRONG GO" and evo_verdict(3, 0, None, []) == "WEAK / REDESIGN"
    assert evo_verdict(1, 1, None, []) == "PROMISING" and evo_verdict(5, 1, None, []) == "PROMISING"
    assert _evo_valid_grid([[0, 9]]) and not _evo_valid_grid([[10]]) and not _evo_valid_grid([[1], [1, 2]]) and not _evo_valid_grid([[True]])
    assert evo_verdict(9, 9, "engine_error", []) == "INVALID / ENGINE FAILURE"
    assert evo_verdict(9, 9, "evo_exception", []) == "INVALID / EVO EXCEPTION"
    _bv = _evo_verify({"code": "def solution(grid):\n    return grid", "cand_id": "boom"}, None)   # task=None raises inside
    assert _bv.get("infra_failure") and EVO_INFRA_FAILURES, _bv
    _pr = evo_repair_prompt(_t, {"rule": "r", "code": "def solution(grid):\n    return [[1]]",
                                 "verifier": {"all_train_exact": True, "test_execution_ok": False, "train": [],
                                              "test_error": "KeyError: 9"}})
    assert "TEST INPUT FAILURE" in _pr, _pr[-600:]
    r = render_evo_chat("x")
    assert "<tools>" not in r and r.rstrip().endswith("<think>"), r[-200:]
    EVO_INFRA_FAILURES.clear(); EVO_ENGINE_FAILURES.clear()
    print("EVO self-tests PASS")
    return True

import collections
EVO_SELFTEST_OK = run_evo_selftests()
# a valid submission.json exists before any generation (fallback = test inputs), then is refreshed after every chunk
EVO_FALLBACK_SUBMISSION = evo_write_submission(evo_submission_task_ids())
print("fallback submission.json written:", len(EVO_FALLBACK_SUBMISSION), "tasks")
if USE_MODEL:
    try:
        EVO_RESULT = run_evo()
    except Exception as _evo_exc:
        import traceback as _evo_tb
        EVO_RESULT = {"stopped": "evo_exception", "error": f"{type(_evo_exc).__name__}: {_evo_exc}"[:2000],
                      "traceback": _evo_tb.format_exc()[-4000:]}
        print("[EVO] UNEXPECTED EXCEPTION; run marked INVALID, report and bundle still written:", EVO_RESULT["error"])
else:
    EVO_RESULT = {"stopped": "no_model"}
print("Cell 8 (EVO) complete:", json.dumps(EVO_RESULT, default=str))


In [ ]:
# =====================================================================================
# CELL 9 — EVO SCORING: pass@2 strict score, complementarity with NVARC and old PIV, go/no-go, bundle
# =====================================================================================
import csv as _evo_csv, zipfile

# Frozen post-hoc references (never used by the solver).
NVARC_EVAL120_FROZEN = {"source_notebook": "31_experiment Notebook_arc-2026d-with-codex (22).ipynb", "source_sha256": "69a64354b6b1c696af1f1c7a68984275d4db9b7eef5f2a382f879dc862d7c978", "rule": "query solved iff Probmul rank in {1,2}", "per_task_score": {"0934a4d8": 0.0, "135a2760": 1.0, "136b0064": 0.0, "13e47133": 0.0, "142ca369": 0.0, "16b78196": 0.0, "16de56c4": 0.0, "1818057f": 1.0, "195c6913": 0.0, "1ae2feb7": 0.3333333333333333, "20270e3b": 0.0, "20a9e565": 0.0, "21897d95": 0.0, "221dfab4": 0.0, "247ef758": 0.0, "269e22fb": 0.0, "271d71e2": 0.0, "28a6681f": 0.0, "291dc1e1": 0.0, "2b83f449": 0.0, "2ba387bc": 1.0, "2c181942": 0.0, "2d0172a1": 1.0, "31f7f899": 1.0, "332f06d7": 0.0, "35ab12c3": 0.0, "36a08778": 1.0, "38007db0": 1.0, "3a25b0d8": 1.0, "3dc255db": 0.0, "3e6067c3": 0.0, "409aa875": 0.0, "446ef5d2": 0.0, "45a5af55": 1.0, "4a21e3da": 0.5, "4c3d4a41": 1.0, "4c416de3": 0.0, "4c7dc4dd": 0.0, "4e34c42c": 0.0, "53fb4810": 1.0, "5545f144": 0.0, "581f7754": 0.0, "58490d8a": 0.0, "58f5dbd5": 1.0, "5961cc34": 0.0, "5dbc8537": 0.0, "62593bfd": 0.0, "64efde09": 0.0, "65b59efc": 0.5, "67e490f4": 0.0, "6e453dd6": 0.0, "6e4f6532": 0.0, "6ffbe589": 0.0, "71e489b6": 1.0, "7491f3cf": 0.0, "7666fa5d": 1.0, "78332cb0": 0.5, "7b0280bc": 1.0, "7b3084d4": 0.0, "7b5033c1": 0.0, "7b80bb43": 1.0, "7c66cb00": 1.0, "7ed72f31": 0.5, "800d221b": 0.0, "80a900e0": 0.0, "8698868d": 0.0, "88bcf3b4": 0.0, "88e364bc": 0.0, "89565ca0": 0.0, "898e7135": 0.0, "8b7bacbf": 0.0, "8b9c3697": 1.0, "8e5c0c38": 0.5, "8f215267": 0.0, "8f3a5a89": 1.0, "9385bd28": 0.5, "97d7923e": 0.0, "981571dc": 1.0, "9aaea919": 1.0, "9bbf930d": 0.0, "a251c730": 1.0, "a25697e4": 0.0, "a32d8b75": 0.0, "a395ee82": 0.0, "a47bf94d": 0.0, "a6f40cea": 1.0, "aa4ec2a5": 1.0, "abc82100": 0.0, "b0039139": 0.0, "b10624e5": 0.0, "b5ca7ac4": 0.0, "b6f77b65": 0.3333333333333333, "b99e7126": 0.0, "b9e38dc0": 0.0, "bf45cf4b": 1.0, "c4d067a0": 0.0, "c7f57c3e": 0.0, "cb2d8a2c": 0.0, "cbebaa4b": 0.0, "d35bdbdc": 0.3333333333333333, "d59b0160": 0.0, "d8e07eb2": 0.5, "da515329": 0.0, "db0c5428": 1.0, "db695cfb": 0.0, "dbff022c": 0.0, "dd6b8c4b": 0.0, "de809cff": 1.0, "dfadab01": 0.0, "e12f9a14": 0.0, "e3721c99": 0.0, "e376de54": 0.0, "e8686506": 1.0, "e87109e9": 0.0, "edb79dae": 0.0, "eee78d87": 1.0, "f560132c": 0.0, "f931b4a8": 0.0, "faa9f03d": 0.0, "fc7cae8d": 0.0}}
PIV_OLD_STRICT = {"1818057f", "2ba387bc"}          # canonical baseline, Strict Solved
PIV_OLD_TRAIN_VERIFIED = {"1818057f", "2ba387bc", "7b5033c1"}
EVO_GO_RULES = {"strong_go": "solved>=3 and evo_only>=2 (12 tasks, 9 NVARC misses)", "promising": "evo_only>=1",
                "weak": "otherwise", "no_decision": "deadline stop -> INCOMPLETE; verifier infrastructure failure or no model -> INVALID"}

def evo_score():
    rows = []
    nv_score = NVARC_EVAL120_FROZEN["per_task_score"]
    for t in EVO_TASKS:
        st = EVO_STATE[t]
        gt = eval_solutions.get(t) or []
        attempts = evo_attempts_by_query(st["cands"], len(eval_challenges[t]["test"]))   # canonical per-query pass@2
        q = [any(a == g for a in attempts[i]) if i < len(attempts) else False for i, g in enumerate(gt)]
        strict = bool(q) and all(q)
        cands = st["cands"]
        best = max(cands, key=_evo_rank_key) if cands else None
        nvs = nv_score.get(t)
        rows.append({
            "task_id": t, "nvarc_task_score": nvs, "nvarc_miss": nvs == 0.0, "nvarc_partial": nvs is not None and 0.0 < nvs < 1.0,
            "piv_old_strict": t in PIV_OLD_STRICT, "evo_strict": strict, "evo_query_frac": (sum(q) / len(q)) if q else 0.0,
            "evo_train_exact_candidates": sum(1 for c in cands if (c.get("verifier") or {}).get("all_train_exact")),
            "evo_attempts": min((len(a) for a in attempts), default=0), "attempts_per_query": [len(a) for a in attempts],
            "repair_parents_nonpositive_gain": st["repair_parents_nonpositive_gain"], "calls": st["calls"], "tokens": st["tokens"], "skipped_calls": st["skipped_calls"],
            "code_extracted": sum(1 for c in cands if c.get("code")),
            "think_closed_naturally": sum(1 for c in cands if c.get("think_closed")),
            "best_train_exact_pairs": (best["quality"][2] if best else 0), "n_train": len(eval_challenges[t]["train"]),
            "identity_wrong_frac": evo_identity_wrong_frac(eval_challenges[t]),
            "best_wrong_frac": (best["wrong_frac"] if best else None),
            "best_gain_vs_identity": (best["gain_vs_identity"] if best else None),
            "alternatives": st["alternatives"], "backfills": st["backfills"],
            "first_train_exact_round": min([c["round"] for c in cands if (c.get("verifier") or {}).get("all_train_exact")], default=None),
        })
    n = len(rows)
    solved = sum(r["evo_strict"] for r in rows)
    evo_only = sum(1 for r in rows if r["evo_strict"] and r["nvarc_miss"])
    both = sum(1 for r in rows if r["evo_strict"] and (r["nvarc_task_score"] or 0) == 1.0)
    nvarc_only = sum(1 for r in rows if not r["evo_strict"] and (r["nvarc_task_score"] or 0) == 1.0)
    evo_on_partial = sum(1 for r in rows if r["evo_strict"] and r["nvarc_partial"])
    stop_reason = (EVO_RESULT or {}).get("stopped")
    verdict = evo_verdict(solved, evo_only, stop_reason, EVO_INFRA_FAILURES)
    calls = sum(r["calls"] for r in rows)
    summary = {
        "tasks": n, "planned_tasks": len(PLANNED_TASK_IDS), "queries": sum(len(eval_solutions.get(r["task_id"]) or []) for r in rows),
        "evo_strict_solved": solved, "evo_arc_score_pct": round(100 * sum(r["evo_query_frac"] for r in rows) / max(n, 1), 2),
        "EVOLUTION_ONLY_(nvarc_miss)": evo_only, "BOTH": both, "NVARC_ONLY": nvarc_only,
        "NVARC_PARTIAL_tasks": sum(r["nvarc_partial"] for r in rows), "EVO_SOLVES_NVARC_PARTIAL": evo_on_partial,
        "stop_reason": stop_reason, "verifier_infrastructure_failures": len(EVO_INFRA_FAILURES),
        "tasks_complete_2_per_query": sum(1 for r in rows if r["evo_attempts"] >= 2),
        "tasks_with_positive_gain_vs_identity": sum(1 for r in rows if (r["best_gain_vs_identity"] or 0) > 0),
        "nvarc_misses_in_subset": sum(r["nvarc_miss"] for r in rows),
        "piv_old_strict_in_subset": sum(r["piv_old_strict"] for r in rows),
        "tasks_with_train_exact_candidate": sum(1 for r in rows if r["evo_train_exact_candidates"] > 0),
        "train_exact_but_wrong_test": sum(1 for r in rows if r["evo_train_exact_candidates"] > 0 and not r["evo_strict"]),
        "repair_parents_nonpositive_gain_total": sum(r["repair_parents_nonpositive_gain"] for r in rows),
        "tasks_with_8_calls": sum(1 for r in rows if r["calls"] >= 8), "tasks_with_7_calls": sum(1 for r in rows if r["calls"] == 7),
        "tasks_with_6_or_fewer_calls": sum(1 for r in rows if r["calls"] <= 6),
        "budget_skipped_calls": sum(r["skipped_calls"] for r in rows),
        "exact_partial_repairs": sum(EVO_STATE[r["task_id"]]["exact_partial_repairs"] for r in rows),
        "engine_failures": EVO_ENGINE_FAILURES,
        "calls": calls, "code_extraction_rate": round(sum(r["code_extracted"] for r in rows) / max(calls, 1), 3),
        "natural_think_close_rate": round(sum(r["think_closed_naturally"] for r in rows) / max(calls, 1), 3),
        "tokens_total": sum(r["tokens"] for r in rows), "tokens_per_task_mean": round(sum(r["tokens"] for r in rows) / max(n, 1)),
        "rounds": EVO_ROUNDS, "evo_result": EVO_RESULT, "config": EVO_CONFIG,
        "verdict_rule": EVO_GO_RULES, "verdict": verdict,
        "solved_tasks": [r["task_id"] for r in rows if r["evo_strict"]],
        "evolution_only_tasks": [r["task_id"] for r in rows if r["evo_strict"] and r["nvarc_miss"]],
    }
    (WORK_ROOT / "evo_report.json").write_text(json.dumps({"summary": summary, "rows": rows}, indent=2, default=str), encoding="utf-8")
    with open(WORK_ROOT / "evo_task_table.csv", "w", newline="", encoding="utf-8") as f:
        w = _evo_csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print("=" * 116)
    print("EVO REPORT:", json.dumps({k: v for k, v in summary.items() if k not in ("rounds", "config", "evo_result")}, indent=2))
    print("ROUNDS:", json.dumps(EVO_ROUNDS, indent=1))
    for r in rows:
        print(f"  {r['task_id']} nvarc={r['nvarc_task_score']} evo_strict={r['evo_strict']} exact_cands={r['evo_train_exact_candidates']} "
              f"best_pairs={r['best_train_exact_pairs']}/{r['n_train']} wrong={r['best_wrong_frac']} identity={r['identity_wrong_frac']} "
              f"gain={r['best_gain_vs_identity']} attempts={r['evo_attempts']} calls={r['calls']} tok={r['tokens']}")
    print("VERDICT:", verdict)
    return summary

def evo_rerun_summary():
    n_att = {t: evo_attempts_by_query(EVO_STATE[t]["cands"], len(eval_challenges[t]["test"])) for t in EVO_TASKS}
    summary = {"mode": "competition_rerun", "hidden_tasks": len(test_challenges), "tasks_run": len(EVO_TASKS),
               "tasks_with_any_attempt": sum(1 for a in n_att.values() if any(a)),
               "tasks_all_queries_attempted": sum(1 for a in n_att.values() if a and all(a)),
               "calls": sum(EVO_STATE[t]["calls"] for t in EVO_TASKS),
               "tokens_total": sum(EVO_STATE[t]["tokens"] for t in EVO_TASKS),
               "stop_reason": (EVO_RESULT or {}).get("stopped"), "rounds": EVO_ROUNDS,
               "engine_failures": EVO_ENGINE_FAILURES, "verifier_infrastructure_failures": len(EVO_INFRA_FAILURES)}
    (WORK_ROOT / "evo_report.json").write_text(json.dumps({"summary": summary}, indent=2, default=str), encoding="utf-8")
    print("EVO RERUN SUMMARY:", json.dumps({k: v for k, v in summary.items() if k != "rounds"}, indent=2, default=str))
    return summary

try:
    EVO_SUMMARY = (evo_rerun_summary() if EVO_RERUN else evo_score()) if EVO_TASKS else {}
except Exception as _rep_exc:   # the bundle below must still be written
    import traceback as _rep_tb
    EVO_SUMMARY = {"scoring_exception": f"{type(_rep_exc).__name__}: {_rep_exc}", "traceback": _rep_tb.format_exc()[-4000:],
                   "evo_result": EVO_RESULT, "engine_failures": EVO_ENGINE_FAILURES}
    (WORK_ROOT / "evo_report.json").write_text(json.dumps({"summary": EVO_SUMMARY}, indent=2, default=str), encoding="utf-8")
    print("VERDICT: INVALID / SCORING EXCEPTION", EVO_SUMMARY["scoring_exception"])

# final submission.json (both modes); on any failure the last valid file (fallback or per-chunk refresh) stays
try:
    EVO_FINAL_SUBMISSION = evo_write_submission(evo_submission_task_ids())
    print("submission.json written:", len(EVO_FINAL_SUBMISSION), "tasks |", EVO_SUBMISSION_PATH)
except Exception as _sub_exc:
    print("FINAL submission write failed, previous submission.json kept:", type(_sub_exc).__name__, _sub_exc)
RUN_META = {"experiment_id": EXPERIMENT_ID, "run_id": RUN_ID, "run_profile": RUN_PROFILE, "tasks": EVO_TASKS,
            "engine_config": ENGINE_CONFIG if USE_MODEL else None, "model_fingerprint": MODEL_ARTIFACT_FINGERPRINT_SHA256,
            "wall_sec_total": round(time.perf_counter() - notebook_start, 1)}
(WORK_ROOT / "evo_run_metadata.json").write_text(json.dumps(RUN_META, indent=2, default=str), encoding="utf-8")
ZIP_PATH = Path("/kaggle/working")/BUNDLE_NAME
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORK_ROOT.rglob("*")):
        if p.is_file() and p.stat().st_size < 200 * 1024**2:
            zf.write(p, p.relative_to(WORK_ROOT.parent).as_posix())
print("=" * 116); print(EXPERIMENT_ID, "PIPELINE COMPLETE | bundle:", ZIP_PATH, "| MB:", round(ZIP_PATH.stat().st_size / 1024**2, 2))


## Run sequence — PIV-Q27-EVO12

Cells 1-7: config, data, tokenizer/template, guard, fresh verifier, helper library, model load (unchanged from V3.1). Cell 8: evolution (4 candidates/task -> verify -> top-2 distinct parents -> 2 repair generations). Cell 9: pass@2 scoring, complementarity with the frozen NVARC map, verdict, bundle.
